# De Novo Drug Design
### PHS 312 - Drug Discovery, Design, and Development
**Lecture 27 · Wednesday, November 4, 2026 · 50 minutes**

Pre-class reading: Patrick 7e, Section 17.15  (optional: 17.16–17.17)

**This activity takes 30-35 minutes.** Work in pairs. Run each cell with the
play button on its left, answer the question that follows, and keep going.
You will not write any code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spinner101/phs312-cadd-notebooks/blob/main/notebooks/PHS312_L27_De_Novo_Drug_Design.ipynb)

> **New:** Parts 2–4 are **lecture notes**: key ideas, comparison tables (growing vs linking vs building; LUDI, SPROUT, LEGEND, GROW, SYNOPSIS, CAESA), annotated structures, worked examples, step-by-step calculation cards with **Solve it yourself** cells (ligand efficiency, LLE, growth bookkeeping, fragment linking, rule of three, fold → ΔΔG) and a fragment-linking calculator; every calculation you solve goes into an automatic lab report, and the thymidylate synthase de novo case study. Read them before class or use them afterwards for review. The timed pair activity begins at Part 5.

---
### Before you start - what this notebook does with your work

* **Nothing you type is sent anywhere.** This notebook makes no network
  calls, writes no files, and does not connect to Google Drive. Your answers
  live in the memory of this browser tab and disappear when you close it.
* **Your instructor cannot see this page.** At the end you will get a short
  report code. You choose to paste it into Canvas - that is the only thing
  that reaches your instructor.
* **Do not type anything personal.** No patient information, no identifiers,
  nothing about a real person. Everything here is simulated.
* Opening this notebook signs you in with your Hampton Google account, which
  is covered by the university's agreement with Google. If you would rather
  not use a Google account, tell your instructor and you will be given an
  offline copy.
---

In [ ]:
#@title Set-up  -  run this cell first { display-mode: "form" }
#@markdown Press the play button once. Nothing is downloaded and nothing
#@markdown leaves this browser tab. Takes a few seconds.
import sys, types, textwrap

_tk = types.ModuleType("toolkit"); _tk.__dict__["__name__"] = "toolkit"
exec(compile("\"\"\"PHS 312 activity toolkit - assessment engine and simulation helpers.\n\nEverything runs in the browser session. Nothing is transmitted anywhere:\nno network calls, no file writes, no analytics. Scores exist only in the\nstudent's own Colab session until they choose to report them in Canvas.\n\"\"\"\nimport textwrap, hashlib, json\nfrom datetime import datetime\n\nW = 86\ndef _wrap(s, indent=\"\"):\n    out = []\n    for para in str(s).split(\"\\n\"):\n        out.extend(textwrap.wrap(para, W, initial_indent=indent,\n                                 subsequent_indent=indent) or [\"\"])\n    return \"\\n\".join(out)\n\ndef _rule(ch=\"-\"):\n    return ch * W\n\n\nclass Item:\n    \"\"\"One multiple-choice item with per-distractor feedback.\"\"\"\n    def __init__(self, qid, stem, options, key, why, bloom=\"\", silo=\"\",\n                 section=\"concept\", topic=\"\"):\n        if key not in options:\n            raise ValueError(f\"{qid}: key {key!r} not among options {list(options)}\")\n        missing = [k for k in options if k != key and k not in why]\n        if missing:\n            raise ValueError(f\"{qid}: no feedback written for distractor(s) {missing}\")\n        self.qid, self.stem, self.options = qid, stem, options\n        self.key, self.why = key, why\n        self.bloom, self.silo, self.section = bloom, silo, section\n        self.topic = topic\n\n\nclass Gradebook:\n    \"\"\"Tracks responses and reports performance. Local to this session.\"\"\"\n\n    SECTIONS = (\"precheck\", \"concept\", \"post\")\n    LABELS = {\"precheck\": \"Prior-knowledge check\",\n              \"concept\": \"Concept checks during the activity\",\n              \"post\": \"End-of-activity assessment\"}\n\n    def __init__(self, lecture, title):\n        self.lecture, self.title = lecture, title\n        self.items, self.responses = {}, {}\n        self.started = datetime.now()\n\n    # ---------- authoring ----------\n    def add(self, item):\n        if item.qid in self.items:\n            raise ValueError(f\"duplicate qid {item.qid}\")\n        self.items[item.qid] = item\n\n    def add_many(self, items):\n        for it in items:\n            self.add(it)\n\n    # ---------- student interaction ----------\n    def ask(self, qid):\n        \"\"\"Print the stem and options for an item.\"\"\"\n        it = self.items[qid]\n        print(_rule(\"=\"))\n        print(_wrap(f\"{qid}.  {it.stem}\"))\n        print()\n        for letter, text in it.options.items():\n            print(_wrap(f\"{letter}) {text}\", indent=\"    \"))\n        print(_rule(\"=\"))\n\n    def answer(self, qid, choice):\n        \"\"\"Record a response and give feedback appropriate to the section.\"\"\"\n        if qid not in self.items:\n            print(f\"[!] Unknown question id {qid!r}. Check the notebook.\")\n            return\n        it = self.items[qid]\n        choice = (choice or \"\").strip().upper()\n        if choice in (\"\", \"-\", \"SELECT\", \"NONE\"):\n            print(\"Choose an option from the dropdown, then press the play button again.\")\n            return\n        if choice not in it.options:\n            print(f\"[!] {choice!r} is not one of {list(it.options)}.\")\n            return\n\n        first = qid not in self.responses\n        if first:\n            self.responses[qid] = choice      # only the first response scores\n        correct = choice == it.key\n\n        if it.section == \"precheck\":\n            # No feedback yet: revealing it now would teach the answer\n            # before the activity and destroy the pre/post comparison.\n            n = len([q for q in self.responses\n                     if self.items[q].section == \"precheck\"])\n            total = len([q for q in self.items\n                         if self.items[q].section == \"precheck\"])\n            print(f\"Recorded. ({n} of {total} answered)\")\n            if n == total:\n                print(\"\\nAll set - run the next cell to begin the activity.\")\n            return\n\n        print(_rule())\n        if correct:\n            print(f\"Correct - {choice}.\")\n            print(_wrap(it.why.get(\"_correct\", \"\"), \"\"))\n        else:\n            print(f\"Not quite. You chose {choice}.\")\n            print(_wrap(it.why[choice]))\n            print()\n            print(_wrap(f\"The defensible answer is {it.key}. \"\n                        + it.why.get(\"_correct\", \"\")))\n        if not first:\n            kept = self.responses[qid]\n            print(f\"\\n(Your first response, {kept}, is the one on record - \"\n                  f\"this retry is for your own understanding.)\")\n        print(_rule())\n\n    # ---------- reporting ----------\n    def _score(self, section):\n        qs = [q for q, it in self.items.items() if it.section == section]\n        got = sum(1 for q in qs if self.responses.get(q) == self.items[q].key)\n        return got, len(qs)\n\n    def summary(self, show_pre_post=True):\n        print(_rule(\"=\"))\n        print(f\"  {self.lecture} - {self.title}\")\n        print(f\"  Performance summary\")\n        print(_rule(\"=\"))\n        any_missing = False\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            if tot == 0:\n                continue\n            answered = len([q for q, it in self.items.items()\n                            if it.section == sec and q in self.responses])\n            if answered < tot:\n                any_missing = True\n            pct = 100 * got / tot if tot else 0\n            bar = \"#\" * int(round(pct / 5)) + \".\" * (20 - int(round(pct / 5)))\n            print(f\"  {self.LABELS[sec]:<38} {got:>2}/{tot:<3} {pct:5.1f}%  [{bar}]\")\n            if answered < tot:\n                print(f\"      ({tot - answered} unanswered)\")\n        print(_rule())\n\n        if show_pre_post:\n            pre_g, pre_t = self._score(\"precheck\")\n            post_g, post_t = self._score(\"post\")\n            if pre_t and post_t:\n                pre = 100 * pre_g / pre_t\n                post = 100 * post_g / post_t\n                print(f\"  Before the activity: {pre:5.1f}%     \"\n                      f\"After the activity: {post:5.1f}%\")\n                if post >= pre:\n                    if pre < 100:\n                        g = (post - pre) / (100 - pre)\n                        print(f\"  Normalised learning gain  g = {g:.2f}\"\n                              f\"   {self._gain_note(g)}\")\n                    else:\n                        print(\"  You came in at ceiling on the prior check.\")\n                else:\n                    print(\"  Your post score is below your pre score - look back at the\")\n                    print(\"  concept checks you missed before you leave.\")\n                print(_rule())\n\n        self._missed()\n        if any_missing:\n            print(\"\\n  Some questions are unanswered. Scroll back and complete them\")\n            print(\"  before copying your report code.\")\n        return self\n\n    @staticmethod\n    def _gain_note(g):\n        if g >= 0.7: return \"(high)\"\n        if g >= 0.3: return \"(medium)\"\n        return \"(low - review before the exam)\"\n\n    def _missed(self):\n        missed = [(q, it) for q, it in self.items.items()\n                  if it.section != \"precheck\"\n                  and q in self.responses and self.responses[q] != it.key]\n        if not missed:\n            print(\"  Nothing missed in the graded sections.\")\n            return\n        print(\"  Worth reviewing before the exam:\")\n        for q, it in missed:\n            tag = f\"[{it.bloom}\" + (f\" | SILO {it.silo}\" if it.silo else \"\") + \"]\"\n            print(_wrap(f\"- {q} {tag}: {it.stem}\", indent=\"    \"))\n\n    def report_code(self, student_name=\"\"):\n        \"\"\"A short code the student pastes into Canvas.\n\n        This is a participation signal, not a secure exam score: it is\n        generated in the browser and a determined student could edit it.\n        Weight it accordingly.\n        \"\"\"\n        tags = {\"precheck\": \"PRE\", \"concept\": \"CON\", \"post\": \"POST\"}\n        parts = []\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            parts.append(f\"{tags[sec]}{got}/{tot}\" if tot else f\"{tags[sec]}-\")\n        body = f\"{self.lecture}|\" + \",\".join(parts)\n        digest = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper()\n        code = f\"{body}|{digest}\"\n        print(_rule(\"=\"))\n        print(\"  COPY THIS LINE INTO THE CANVAS ASSIGNMENT\")\n        print(_rule(\"=\"))\n        print(f\"  {code}\")\n        print(_rule(\"=\"))\n        print(_wrap(\"Nothing was sent anywhere. This code exists only in your \"\n                    \"browser until you paste it into Canvas yourself.\"))\n        return code\n\n\ndef verify_code(code):\n    \"\"\"Instructor-side check that a pasted code has not been casually edited.\"\"\"\n    try:\n        body, digest = code.strip().rsplit(\"|\", 1)\n        ok = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper() == digest.upper()\n        return ok, body\n    except Exception:\n        return False, \"\"\n\n\n    # ---------- diagnostic review (used by the L28 review session) ----------\ndef _topic_table(gb, guide=None, section=None):\n    \"\"\"Score by topic and say what to revisit. Attached to Gradebook below.\n\n    `section` limits the table to one section, so the mid-session diagnostic\n    does not count questions the student has not reached yet.\n    \"\"\"\n    rows = {}\n    for qid, it in gb.items.items():\n        if not it.topic:\n            continue\n        if section and it.section != section:\n            continue\n        got = gb.responses.get(qid)\n        r = rows.setdefault(it.topic, [0, 0, 0])\n        r[1] += 1\n        if got is not None:\n            r[2] += 1\n            if got == it.key:\n                r[0] += 1\n    if not rows:\n        print(\"No topic-tagged questions have been answered yet.\")\n        return {}\n    print(_rule(\"=\"))\n    print(\"  Where you stand, by lecture\")\n    print(_rule(\"=\"))\n    print(f\"  {'lecture':<9}{'answered':>10}{'correct':>9}{'':>4}\")\n    weak = []\n    for t in sorted(rows):\n        ok, total, ans = rows[t]\n        pct = (100.0 * ok / ans) if ans else 0.0\n        bar = \"#\" * int(round(pct / 10)) + \".\" * (10 - int(round(pct / 10)))\n        flag = \"\"\n        if ans == 0:\n            bar, flag = \"-\" * 10, \"  not attempted\"\n        elif pct < 67:\n            flag = \"  <- revisit\"\n            weak.append(t)\n        print(f\"  {t:<9}{ans:>5} of {total:<4}{ok:>5}   {bar}{flag}\")\n    print()\n    if weak:\n        print(\"  Before the exam, go back to:\")\n        for t in weak:\n            where = (guide or {}).get(t, \"\")\n            print(f\"    {t}  {where}\" if where else f\"    {t}\")\n    else:\n        done = sum(r[2] for r in rows.values())\n        total = sum(r[1] for r in rows.values())\n        if done < total:\n            print(f\"  Nothing below two thirds so far, with {total - done} \"\n                  f\"question(s) still unanswered.\")\n        else:\n            print(\"  No lecture is below two thirds. Review your missed \"\n                  \"questions above,\\n  then re-run the activities you found \"\n                  \"slowest rather than re-reading.\")\n    print(_rule(\"=\"))\n    return {t: dict(correct=v[0], answered=v[2], total=v[1]) for t, v in rows.items()}\n\n\nGradebook.topic_report = _topic_table\n", "toolkit.py", "exec"), _tk.__dict__)
sys.modules["toolkit"] = _tk
Item, Gradebook, verify_code = _tk.Item, _tk.Gradebook, _tk.verify_code

_sb = types.ModuleType("swissbio"); _sb.__dict__["__name__"] = "swissbio"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"Curated bioisosteric replacement data for PHS 312 Lecture 31.\n\nSOURCE\n    SwissBioisostere, https://www.swissbioisostere.ch\n    Molecular Modelling Group, SIB Swiss Institute of Bioinformatics and\n    the University of Lausanne. Underlying data derived from ChEMBL 36.\n\nLICENCE\n    CC-BY 4.0  https://creativecommons.org/licenses/by/4.0/\n    Redistribution and adaptation permitted with attribution.\n\nCHANGES MADE (required by CC-BY)\n    Retrieved 20 October 2026 through the SwissBioisostere web interface.\n    Two single-attachment-point (\"side chain\") fragment queries were run:\n    carboxylic acid [*:1]C(=O)O  and primary amide [*:1]C(=O)N.\n    The full result tables (7,714 and 6,955 replacements) were reduced to the\n    highest-frequency entries shown below. Column names were shortened and\n    values were not otherwise altered. No structures were recomputed.\n\nCITE\n    Cuozzo A., Daina A., Perez M.A.S., Michielin O., Zoete V.\n    SwissBioisostere: molecular replacements for drug discovery guided by\n    updated structural, bioactivity and physico-chemical data accessible\n    through a reshaped Web interface. Nucleic Acids Research, 2021.\n    Wirth M., Zoete V., Michielin O., Sauer W.H.B. SwissBioisostere: a\n    database of molecular replacements for ligand design. Nucleic Acids\n    Research 2013, 41(D1), D1137-D1143. doi:10.1093/nar/gks1059\n\nCOLUMNS\n    smiles  replacement fragment, [*:1] marks the attachment point\n    freq    number of matched molecular pairs in which this replacement occurs\n    rank    rank by frequency within that query\n    better / similar / worse\n            how the replacement changed measured activity across those pairs\n    dlogp, dtpsa, dmw\n            mean change in calculated logP, topological PSA and molecular\n            weight on making the replacement\n\"\"\"\n\nFIELDS = (\"smiles\", \"freq\", \"rank\", \"better\", \"similar\", \"worse\",\n          \"dlogp\", \"dtpsa\", \"dmw\")\n\n# common names for the fragments students meet in Patrick 7e 13.3.7\nNAMES = {\n    \"COC(=O)[*:1]\": \"methyl ester\",\n    \"NC(=O)[*:1]\": \"primary amide\",\n    \"CCOC(=O)[*:1]\": \"ethyl ester\",\n    \"O[*:1]\": \"hydroxyl\",\n    \"C[*:1]\": \"methyl\",\n    \"OC[*:1]\": \"hydroxymethyl\",\n    \"O=C(O)C[*:1]\": \"carboxymethyl (homologue)\",\n    \"N#C[*:1]\": \"nitrile\",\n    \"CNC(=O)[*:1]\": \"N-methyl amide\",\n    \"CO[*:1]\": \"methoxy\",\n    \"N[*:1]\": \"amine\",\n    \"CN(C)C(=O)[*:1]\": \"N,N-dimethyl amide\",\n    \"O=C(NO)[*:1]\": \"hydroxamic acid\",\n    \"c1ccc([*:1])cc1\": \"phenyl\",\n    \"F[*:1]\": \"fluorine\",\n    \"Cl[*:1]\": \"chlorine\",\n    \"n1n[nH]c([*:1])n1\": \"tetrazol-5-yl\",\n    \"FC(F)(F)[*:1]\": \"trifluoromethyl\",\n    \"n1nc([*:1])n[nH]1\": \"tetrazol-5-yl (tautomer)\",\n    \"O=[N+]([O-])[*:1]\": \"nitro\",\n    \"NS(=O)(=O)[*:1]\": \"sulfonamide\",\n    \"CS(=O)(=O)NC(=O)[*:1]\": \"acylsulfonamide\",\n    \"O=C(O)CC[*:1]\": \"2-carboxyethyl (homologue)\",\n    \"O=C(NCCO)[*:1]\": \"N-(2-hydroxyethyl) amide\",\n    \"CC(=O)N[*:1]\": \"acetamido\",\n    \"Br[*:1]\": \"bromine\",\n    \"CS(=O)(=O)[*:1]\": \"methylsulfonyl\",\n    \"O=C(N1CCOCC1)[*:1]\": \"morpholine amide\",\n    \"CC(=O)[*:1]\": \"acetyl\",\n    \"CN(C)[*:1]\": \"dimethylamino\",\n    \"O=C(O)[*:1]\": \"carboxylic acid\",\n    \"CCNC(=O)[*:1]\": \"N-ethyl amide\",\n    \"O=C(NC1CC1)[*:1]\": \"N-cyclopropyl amide\",\n    \"CS(=O)(=O)N[*:1]\": \"methanesulfonamido\",\n}\n\n# ---- carboxylic acid: what replaces [*:1]C(=O)O  (7,714 replacements total)\nACID_TOTAL = 7714\nACID = [\n (\"COC(=O)[*:1]\", 4530, 1, 1238, 2353, 939, 0.09, -11.00, 14.03),\n (\"NC(=O)[*:1]\", 2781, 2, 1000, 1334, 447, -0.60, 5.79, -0.99),\n (\"CCOC(=O)[*:1]\", 2449, 3, 703, 1024, 722, 0.48, -11.00, 28.06),\n (\"O[*:1]\", 1626, 4, 604, 793, 229, 0.01, -17.07, -28.01),\n (\"C[*:1]\", 1478, 5, 620, 616, 242, 0.61, -37.30, -29.98),\n (\"OC[*:1]\", 1144, 6, 395, 546, 203, -0.20, -17.07, -13.98),\n (\"O=C(O)C[*:1]\", 956, 7, 253, 530, 173, -0.08, 0.00, 14.02),\n (\"N#C[*:1]\", 877, 8, 346, 353, 178, 0.17, -13.51, -19.00),\n (\"CNC(=O)[*:1]\", 818, 9, 298, 412, 108, -0.34, -8.20, 13.04),\n (\"CO[*:1]\", 788, 10, 331, 334, 123, 0.31, -28.07, -13.98),\n (\"N[*:1]\", 721, 11, 201, 368, 152, -0.51, -11.28, -29.00),\n (\"CN(C)C(=O)[*:1]\", 692, 12, 230, 331, 131, 0.00, -16.99, 27.07),\n (\"O=C(NO)[*:1]\", 654, 13, 440, 157, 57, -0.58, 12.03, 15.02),\n (\"c1ccc([*:1])cc1\", 635, 14, 271, 262, 102, 1.96, -37.30, 32.09),\n (\"F[*:1]\", 625, 15, 265, 259, 101, 0.44, -37.30, -26.02),\n (\"Cl[*:1]\", 604, 16, 276, 240, 88, 0.95, -37.30, -9.57),\n (\"n1n[nH]c([*:1])n1\", 545, 17, 105, 325, 115, -0.52, 17.16, 24.03),\n (\"FC(F)(F)[*:1]\", 503, 18, 214, 206, 83, 1.32, -37.30, 23.98),\n (\"n1nc([*:1])n[nH]1\", 484, 19, 135, 256, 93, -0.51, 17.16, 24.03),\n (\"O=[N+]([O-])[*:1]\", 384, 20, 166, 167, 51, 0.21, 5.84, 0.99),\n (\"NS(=O)(=O)[*:1]\", 382, 21, 157, 168, 57, -1.05, 22.86, 35.07),\n (\"CS(=O)(=O)NC(=O)[*:1]\", 369, 22, 36, 226, 107, -1.01, 25.94, 77.11),\n (\"O=C(O)CC[*:1]\", 348, 23, 112, 191, 45, 0.32, 0.00, 28.06),\n (\"O=C(NCCO)[*:1]\", 341, 24, 148, 142, 51, -0.98, 12.03, 43.07),\n (\"CC(=O)N[*:1]\", 331, 25, 111, 160, 60, 0.26, -8.20, 13.04),\n (\"Br[*:1]\", 330, 26, 153, 125, 52, 1.06, -37.30, 34.89),\n (\"CS(=O)(=O)[*:1]\", 327, 27, 132, 141, 54, -0.43, -3.16, 34.08),\n (\"O=C(N1CCOCC1)[*:1]\", 315, 28, 93, 151, 71, -0.23, -7.76, 69.10),\n (\"CC(=O)[*:1]\", 296, 29, 120, 118, 58, 0.50, -20.23, -1.97),\n (\"CN(C)[*:1]\", 283, 30, 93, 116, 74, 0.08, -34.06, -0.94),\n]\n\n# ---- primary amide: what replaces [*:1]C(=O)N  (6,955 replacements total)\nAMIDE_TOTAL = 6955\nAMIDE = [\n (\"O=C(O)[*:1]\", 2781, 1, 447, 1334, 1000, 0.60, -5.79, 0.99),\n (\"N#C[*:1]\", 2384, 2, 586, 1189, 609, 1.04, -19.30, -18.01),\n (\"CNC(=O)[*:1]\", 2202, 3, 192, 1591, 419, 0.26, -13.99, 14.03),\n (\"O[*:1]\", 1622, 4, 398, 1003, 221, 0.11, -22.86, -27.02),\n (\"COC(=O)[*:1]\", 1269, 5, 259, 699, 311, 0.68, -16.79, 15.02),\n (\"C[*:1]\", 1238, 6, 361, 668, 209, 1.21, -43.09, -28.99),\n (\"CN(C)C(=O)[*:1]\", 1112, 7, 128, 590, 394, 0.61, -22.78, 28.06),\n (\"OC[*:1]\", 1060, 8, 233, 651, 176, 0.51, -22.86, -13.00),\n (\"CO[*:1]\", 1003, 9, 235, 554, 214, 0.91, -33.86, -12.99),\n (\"F[*:1]\", 720, 10, 195, 360, 165, 1.04, -43.09, -25.04),\n (\"Cl[*:1]\", 669, 11, 262, 281, 126, 1.55, -43.09, -8.58),\n (\"N[*:1]\", 646, 12, 151, 332, 163, 0.08, -17.07, -28.01),\n (\"FC(F)(F)[*:1]\", 635, 13, 194, 272, 169, 1.92, -43.09, 24.97),\n (\"CCOC(=O)[*:1]\", 612, 14, 146, 344, 122, 1.08, -16.79, 29.03),\n (\"CS(=O)(=O)[*:1]\", 599, 15, 90, 403, 106, 0.17, -8.95, 35.06),\n (\"CC(=O)N[*:1]\", 546, 16, 80, 349, 117, 0.26, -13.99, 14.03),\n (\"O=C(NCCO)[*:1]\", 532, 17, 42, 385, 105, -0.38, 6.24, 44.05),\n (\"c1ccc([*:1])cc1\", 476, 18, 128, 246, 102, 2.37, -43.09, 33.07),\n (\"O=C(N1CCOCC1)[*:1]\", 443, 19, 45, 250, 148, 0.37, -13.55, 70.09),\n (\"CC(=O)[*:1]\", 441, 20, 112, 235, 94, 1.10, -26.02, -0.98),\n (\"CCNC(=O)[*:1]\", 438, 21, 65, 269, 104, 0.65, -13.99, 28.06),\n (\"O=C(NC1CC1)[*:1]\", 431, 22, 94, 246, 91, 0.80, -13.99, 40.06),\n (\"NS(=O)(=O)[*:1]\", 382, 23, 72, 232, 78, -0.45, 17.07, 36.06),\n (\"CS(=O)(=O)N[*:1]\", 359, 24, 58, 225, 76, 0.27, 3.08, 50.08),\n (\"Br[*:1]\", 339, 25, 137, 146, 56, 1.66, -43.09, 35.88),\n]\n\nQUERIES = {\n    \"carboxylic acid\": dict(query=\"[*:1]C(=O)O\", total=ACID_TOTAL, rows=ACID),\n    \"primary amide\":   dict(query=\"[*:1]C(=O)N\", total=AMIDE_TOTAL, rows=AMIDE),\n}\n\n\ndef table(which=\"carboxylic acid\"):\n    q = QUERIES[which]\n    return [dict(zip(FIELDS, r), name=NAMES.get(r[0], r[0])) for r in q[\"rows\"]]\n\n\ndef success_rate(row):\n    \"\"\"Fraction of matched pairs where activity did not get worse.\"\"\"\n    n = row[\"better\"] + row[\"similar\"] + row[\"worse\"]\n    return (row[\"better\"] + row[\"similar\"]) / n if n else 0.0\n", "swissbio.py", "exec"), _sb.__dict__)
sys.modules["swissbio"] = _sb
swissbio = _sb

_cr = types.ModuleType("crystal"); _cr.__dict__["__name__"] = "crystal"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"PHS 312 — real-structure activities built on crystallographic coordinates.\n\nEverything here operates on a PDB entry embedded in the notebook itself.\nNo network, no install beyond NumPy and matplotlib. Optional 3D viewing via\npy3Dmol when it is available; otherwise matplotlib draws the structure.\n\nStructure: PDB 1ACJ, Harel et al., acetylcholinesterase with tacrine (THA)\nbound in the active-site gorge.\n\"\"\"\nimport base64, gzip, itertools, math\nimport numpy as np\nimport matplotlib\nimport matplotlib.ticker\nimport matplotlib.pyplot as plt\nfrom mpl_toolkits.mplot3d import Axes3D  # noqa: F401\n\nPRIMARY, SECOND, MUTED, OK = \"#0B6E8F\", \"#C2410C\", \"#6B7280\", \"#15803D\"\nPDB_B64 = \"...\"          # filled in by the notebook builder\n_cache = {}\n\n# covalent radii, enough to infer bonds from coordinates\n_COV = {\"C\": 0.76, \"N\": 0.71, \"O\": 0.66, \"S\": 1.05, \"H\": 0.31, \"P\": 1.07}\nVDW = {\"C\": 1.70, \"N\": 1.55, \"O\": 1.52, \"S\": 1.80, \"H\": 1.20, \"P\": 1.80}\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\ndef pdb_text():\n    if \"txt\" not in _cache:\n        _cache[\"txt\"] = gzip.decompress(base64.b64decode(PDB_B64)).decode()\n    return _cache[\"txt\"]\n\n\ndef _parse():\n    if \"parsed\" in _cache:\n        return _cache[\"parsed\"]\n    prot, lig = [], []\n    for l in pdb_text().splitlines():\n        if l.startswith((\"ATOM\", \"HETATM\")):\n            rec = dict(name=l[12:16].strip(), resn=l[17:20].strip(),\n                       chain=l[21].strip(), resi=l[22:26].strip(),\n                       x=float(l[30:38]), y=float(l[38:46]), z=float(l[46:54]),\n                       elem=(l[76:78].strip() or l[12:16].strip()[0]).upper())\n            (lig if l.startswith(\"HETATM\") else prot).append(rec)\n    _cache[\"parsed\"] = (prot, lig)\n    return prot, lig\n\n\ndef ligand(resn=\"THA\"):\n    _, lig = _parse()\n    if not isinstance(resn, str):\n        raise TypeError(\n            f\"ligand() wants a residue name such as 'THA', got {resn!r}. \"\n            \"If you meant a distance, pass it as cutoff=...\")\n    a = [r for r in lig if r[\"resn\"] == resn.upper()]\n    if not a:\n        have = sorted({r[\"resn\"] for r in lig})\n        raise ValueError(f\"no residue {resn!r} in this structure; available: {have}\")\n    X = np.array([[r[\"x\"], r[\"y\"], r[\"z\"]] for r in a])\n    return a, X\n\n\ndef protein():\n    prot, _ = _parse()\n    X = np.array([[r[\"x\"], r[\"y\"], r[\"z\"]] for r in prot])\n    return prot, X\n\n\ndef bonds_of(atoms, X, tol=0.45):\n    \"\"\"Infer bonds from covalent radii. Returns (i, j, distance).\"\"\"\n    out = []\n    for i, j in itertools.combinations(range(len(X)), 2):\n        d = float(np.linalg.norm(X[i] - X[j]))\n        lim = _COV.get(atoms[i][\"elem\"], 0.77) + _COV.get(atoms[j][\"elem\"], 0.77) + tol\n        if d <= lim:\n            out.append((i, j, d))\n    return out\n\n\ndef angles_of(atoms, X, bonds):\n    adj = {}\n    for i, j, _ in bonds:\n        adj.setdefault(i, []).append(j); adj.setdefault(j, []).append(i)\n    out = []\n    for c, ns in adj.items():\n        for a, b in itertools.combinations(sorted(ns), 2):\n            v1, v2 = X[a] - X[c], X[b] - X[c]\n            cos = float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2)))\n            out.append((a, c, b, math.degrees(math.acos(max(-1, min(1, cos))))))\n    return out\n\n\n# ---------------------------------------------------------------- L22\ndef measure_geometry(term=\"bond lengths\", resn=\"THA\", show=True):\n    \"\"\"Measure what a force field parameterises, from the real crystal structure.\"\"\"\n    atoms, X = ligand(resn)\n    bl = bonds_of(atoms, X)\n    if term == \"bond lengths\":\n        vals = [d for _, _, d in bl]\n        labels = [f\"{atoms[i]['name']}-{atoms[j]['name']}\" for i, j, _ in bl]\n        unit, ideal, title = \"angstrom\", None, \"every bond in the ligand\"\n    else:\n        ang = angles_of(atoms, X, bl)\n        vals = [v for *_, v in ang]\n        labels = [f\"{atoms[a]['name']}-{atoms[c]['name']}-{atoms[b]['name']}\"\n                  for a, c, b, _ in ang]\n        unit, ideal, title = \"degrees\", (109.5, 120.0), \"every valence angle\"\n\n    if show:\n        f, ax = _fig(10.0, 3.8)\n        order = np.argsort(vals)\n        ax.bar(range(len(vals)), [vals[i] for i in order], color=PRIMARY)\n        ax.set_xticks(range(len(vals)))\n        ax.set_xticklabels([labels[i] for i in order], rotation=90, fontsize=7)\n        ax.set_ylabel(unit)\n        ax.set_ylim(min(vals) * 0.97, max(vals) * 1.02)\n        if ideal:\n            for v, lab in zip(ideal, (\"sp3  109.5\", \"sp2  120\")):\n                ax.axhline(v, color=SECOND, ls=\"--\", lw=1.4)\n                ax.text(len(vals) - .5, v, \"  \" + lab, va=\"center\",\n                        color=SECOND, fontsize=9)\n        ax.set_title(f\"{title}, measured in the crystal structure\")\n        plt.tight_layout(); plt.show()\n\n    v = np.array(vals)\n    if not show:\n        out = dict(term=term, n=len(v), lo=round(float(v.min()), 3),\n                   hi=round(float(v.max()), 3), mean=round(float(v.mean()), 3))\n        return out\n    print(f\"{len(v)} measurements from the real structure\")\n    print(f\"  range  {v.min():.3f} to {v.max():.3f} {unit}\")\n    print(f\"  mean   {v.mean():.3f}   spread {v.std():.3f}\")\n    out = dict(term=term, n=len(v), lo=round(float(v.min()), 3),\n               hi=round(float(v.max()), 3), mean=round(float(v.mean()), 3))\n    if term == \"bond lengths\":\n        ar = [d for _, _, d in bl if 1.33 <= d <= 1.43]\n        sp3 = [d for _, _, d in bl if d > 1.48]\n        print(f\"  aromatic-like (1.33-1.43 A): {len(ar)}\")\n        print(f\"  single sp3-like (> 1.48 A) : {len(sp3)}\")\n        print(\"\\n  One force-field parameter cannot cover both. That is why a\")\n        print(\"  force field carries a different constant for each bond type.\")\n        out.update(aromatic=len(ar), sp3=len(sp3))\n    else:\n        n120 = int(((v > 112) & (v < 128)).sum()); n109 = int(((v > 103) & (v < 116)).sum())\n        print(f\"  near 120 deg (sp2): {n120}      near 109.5 deg (sp3): {n109}\")\n        print(\"\\n  The angles cluster around the two hybridisation ideals, which\")\n        print(\"  is exactly what the bending term is fitted to reproduce.\")\n        out.update(near_120=n120, near_109=n109)\n    return out\n\n\ndef contacts(cutoff=4.5, resn=\"THA\", show=True):\n    \"\"\"Real non-bonded contacts between the ligand and the protein.\"\"\"\n    atoms, L = ligand(resn)\n    prot, P = protein()\n    d = np.linalg.norm(L[:, None, :] - P[None, :, :], axis=2)\n    pairs = []\n    for i in range(len(L)):\n        for j in np.where(d[i] <= cutoff)[0]:\n            r = prot[int(j)]\n            s = (VDW.get(atoms[i][\"elem\"], 1.7) + VDW.get(r[\"elem\"], 1.7))\n            pairs.append((float(d[i, int(j)]), atoms[i][\"name\"], r[\"resn\"],\n                          r[\"resi\"], r[\"name\"], s))\n    pairs.sort()\n    if show:\n        gap = np.array([p[0] - p[5] for p in pairs])\n        # four bands, the same ones the printout counts:\n        #   clash: more than 0.4 A inside the vdW sum\n        #   at or just inside: 0 to 0.4 A inside\n        #   near contact: 0 to 0.5 A outside\n        #   further out\n        col = [\"#DC2626\" if g < -0.4 else (OK if g < 0 else (\"#86EFAC\" if g < 0.5 else MUTED))\n               for g in gap]\n        f, ax = _fig(9.4, 4.2)\n        ax.bar(range(len(gap)), gap, color=col, width=1.0)\n        ax.axhline(0, color=\"#1F2937\", lw=1.6)\n        ax.set_xlabel(f\"ligand-protein atom pairs within {cutoff} A, closest first\")\n        ax.set_ylabel(\"distance minus sum of vdW radii (A)\")\n        ax.set_title(\"zero means exactly at van der Waals contact;\\n\"\n                     \"a real clash sits well below zero, and there are none here\",\n                     fontsize=10)\n        nc = int((gap < -0.4).sum()); nt = int(((gap >= -0.4) & (gap < 0)).sum())\n        nn = int(((gap >= 0) & (gap < 0.5)).sum())\n        ax.legend(handles=[\n            plt.Line2D([], [], color=\"#DC2626\", lw=6, label=f\"clash, > 0.4 A inside ({nc})\"),\n            plt.Line2D([], [], color=OK, lw=6, label=f\"at or just inside, 0-0.4 A ({nt})\"),\n            plt.Line2D([], [], color=\"#86EFAC\", lw=6, label=f\"near contact, 0-0.5 A outside ({nn})\"),\n            plt.Line2D([], [], color=MUTED, lw=6, label=\"further out\")],\n            fontsize=8, loc=\"upper left\")\n        plt.tight_layout(); plt.show()\n    tight = [p for p in pairs if p[0] < p[5]]\n    clash = [p for p in pairs if p[0] < p[5] - 0.4]\n    near = [p for p in pairs if p[5] <= p[0] < p[5] + 0.5]\n    if show:\n        print(f\"{len(pairs)} ligand-protein atom pairs within {cutoff} A\")\n        print(f\"  {'dist':>6}  {'sum vdW':>8}  contact\")\n        for p in pairs[:10]:\n            mark = \"  at/inside vdW contact\" if p[0] < p[5] else \"\"\n            print(f\"  {p[0]:>6.2f}  {p[5]:>8.2f}  {p[1]:<4} ... \"\n                  f\"{p[2]}{p[3]} {p[4]:<4}{mark}\")\n        print(f\"\\n  genuinely clashing (more than 0.4 A inside) : {len(clash)}\")\n        print(f\"  at or just inside the vdW sum (0-0.4 A in) : {len(tight) - len(clash)}\")\n        print(f\"  near contact (0-0.5 A outside the sum)     : {len(near)}\")\n        print()\n        if not clash:\n            print(\"  Nothing clashes. That is what a real binding mode looks like:\")\n            print(\"  the ligand sits AT van der Waals contact with the protein,\")\n            print(\"  never inside it. A docking pose that scores well while pushing\")\n            print(\"  atoms closer than this is wrong, however good its number.\")\n    resid = sorted({(p[2], p[3]) for p in pairs})\n    return dict(cutoff=cutoff, pairs=len(pairs), at_or_inside_vdw=len(tight),\n                clashing=len(clash), residues=len(resid),\n                nearest=round(pairs[0][0], 2) if pairs else None)\n\n\ndef dihedral(p0, p1, p2, p3):\n    \"\"\"Standard IUPAC torsion angle in degrees.\"\"\"\n    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2\n    b1n = b1 / np.linalg.norm(b1)\n    v = b0 - (b0 @ b1n) * b1n\n    w = b2 - (b2 @ b1n) * b1n\n    x = v @ w\n    y = np.cross(b1n, v) @ w\n    return math.degrees(math.atan2(y, x))\n\n\nCHI1 = {\"ARG\": \"CG\", \"ASN\": \"CG\", \"ASP\": \"CG\", \"CYS\": \"SG\", \"GLN\": \"CG\",\n        \"GLU\": \"CG\", \"HIS\": \"CG\", \"ILE\": \"CG1\", \"LEU\": \"CG\", \"LYS\": \"CG\",\n        \"MET\": \"CG\", \"PHE\": \"CG\", \"PRO\": \"CG\", \"SER\": \"OG\", \"THR\": \"OG1\",\n        \"TRP\": \"CG\", \"TYR\": \"CG\", \"VAL\": \"CG1\"}\n\n\ndef sidechain_torsions(near_ligand=8.0, resn=\"THA\"):\n    \"\"\"Real chi1 torsions of residues lining the pocket.\"\"\"\n    prot, P = protein()\n    _, L = ligand(resn)\n    byres = {}\n    for i, r in enumerate(prot):\n        byres.setdefault((r[\"chain\"], r[\"resi\"], r[\"resn\"]), {})[r[\"name\"]] = P[i]\n    out = []\n    for (ch, ri, rn), at in byres.items():\n        g = CHI1.get(rn)\n        if not g or not all(k in at for k in (\"N\", \"CA\", \"CB\", g)):\n            continue\n        d = float(np.min(np.linalg.norm(L - at[\"CA\"], axis=1)))\n        if d > near_ligand:\n            continue\n        chi = dihedral(at[\"N\"], at[\"CA\"], at[\"CB\"], at[g])\n        out.append(dict(resn=rn, resi=ri, chain=ch, chi1=chi, dist=d))\n    out.sort(key=lambda r: r[\"dist\"])\n    return out\n\n\ndef rotamers(near_ligand=8.0, show=True):\n    \"\"\"Real side-chain torsions cluster at the three staggered minima.\"\"\"\n    rows = sidechain_torsions(near_ligand)\n    chi = np.array([r[\"chi1\"] for r in rows])\n    def bucket(a):\n        a = (a + 360) % 360\n        if 0 <= a < 120: return \"gauche+ (~60)\"\n        if 120 <= a < 240: return \"trans (~180)\"\n        return \"gauche- (~300)\"\n    buckets = [bucket(c) for c in chi]\n    if show:\n        f, ax = _fig(9.4, 4.0)\n        ax.hist(((chi + 360) % 360), bins=36, range=(0, 360), color=PRIMARY)\n        for v, lab in ((60, \"gauche+\"), (180, \"trans\"), (300, \"gauche-\")):\n            ax.axvline(v, color=SECOND, ls=\"--\", lw=1.5)\n            ax.text(v, ax.get_ylim()[1] * .92, \"  \" + lab, color=SECOND, fontsize=9)\n        ax.set_xlim(0, 360); ax.set_xticks(range(0, 361, 60))\n        ax.set_xlabel(\"chi-1 torsion angle (degrees)\")\n        ax.set_ylabel(\"side chains\")\n        ax.set_title(f\"{len(rows)} residues within {near_ligand} A of the ligand, \"\n                     \"measured in the crystal\")\n        plt.tight_layout(); plt.show()\n        print(f\"{'residue':<10}{'chi-1':>9}   rotamer                 dist to ligand\")\n        for r in rows[:12]:\n            print(f\"  {r['resn']}{r['resi']:<6}{r['chi1']:>9.1f}   \"\n                  f\"{bucket(r['chi1']):<22}  {r['dist']:.1f} A\")\n        from collections import Counter\n        c = Counter(buckets)\n        print(f\"\\n  {c}\")\n        print(\"\\n  Side chains do not sit at random angles. They occupy three\")\n        print(\"  staggered wells, which is the same local-minimum structure you\")\n        print(\"  see in a torsion profile - measured here in a real protein.\")\n    from collections import Counter\n    c = Counter(buckets)\n    return dict(n=len(rows), counts=dict(c),\n                near_stagger=int(sum(1 for a in chi\n                                     if min(abs(((a + 360) % 360) - v)\n                                            for v in (60, 180, 300)) < 40)))\n\n\ndef search_cost(n_rot=7, step=30, seconds_each=0.05, show=True):\n    \"\"\"Arithmetic, not simulation: how many structures a grid search needs.\"\"\"\n    per_bond = int(round(360 / step))\n    n = per_bond ** n_rot\n    secs = n * seconds_each\n    yrs = secs / 3600 / 24 / 365\n    if show:\n        print(f\"settings per bond        {per_bond}   ({step} degree steps)\")\n        print(f\"rotatable bonds          {n_rot}\")\n        print(f\"conformations to score   {n:,}\")\n        print(f\"at {seconds_each} s each            {secs/3600:,.1f} hours\", end=\"\")\n        print(f\"   =  {yrs:,.0f} years\" if yrs >= 1 else \"\")\n        if yrs >= 1:\n            print(\"\\n  This is why no docking program does a systematic search.\")\n    return dict(conformations=n, per_bond=per_bond, hours=secs / 3600,\n                years=yrs)\n\n\n# ---------------------------------------------------------------- L24\ndef rmsd(A, B):\n    return float(np.sqrt(((A - B) ** 2).sum(axis=1).mean()))\n\n\ndef move_ligand(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, axis=\"z\", resn=\"THA\"):\n    \"\"\"Rigid-body move of the real ligand. Returns the moved coordinates.\"\"\"\n    _, X = ligand(resn)\n    c = X.mean(axis=0)\n    t = math.radians(rot_deg)\n    ax = {\"x\": np.array([1., 0, 0]), \"y\": np.array([0, 1., 0]),\n          \"z\": np.array([0, 0, 1.])}[axis]\n    K = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]])\n    R = np.eye(3) + math.sin(t) * K + (1 - math.cos(t)) * (K @ K)\n    return (X - c) @ R.T + c + np.array([dx, dy, dz])\n\n\ndef pose_rmsd(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, resn=\"THA\", show=True):\n    \"\"\"Move the real crystal pose and measure the real RMSD you created.\"\"\"\n    atoms, X = ligand(resn)\n    Y = move_ligand(dx, dy, dz, rot_deg, resn=resn)\n    r = rmsd(X, Y)\n    per = np.linalg.norm(X - Y, axis=1)\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.4, 4.2))\n        a1.bar(range(len(per)), per, color=PRIMARY)\n        a1.set_xticks(range(len(per)))\n        a1.set_xticklabels([a[\"name\"] for a in atoms], rotation=90, fontsize=7)\n        a1.set_ylabel(\"displacement (A)\")\n        a1.axhline(r, color=SECOND, ls=\"--\", lw=1.6, label=f\"RMSD = {r:.2f} A\")\n        a1.legend(fontsize=9)\n        a1.set_title(\"how far each atom moved\")\n        a2.scatter(X[:, 0], X[:, 1], s=60, color=MUTED, label=\"crystal pose\")\n        a2.scatter(Y[:, 0], Y[:, 1], s=60, color=SECOND, label=\"your pose\")\n        for i in range(len(X)):\n            a2.plot([X[i, 0], Y[i, 0]], [X[i, 1], Y[i, 1]], color=\"#AAB\", lw=.8)\n        a2.set_aspect(\"equal\"); a2.legend(fontsize=9)\n        a2.set_xlabel(\"x (A)\"); a2.set_ylabel(\"y (A)\")\n        a2.set_title(\"looking down z\")\n        for ax_ in (a1, a2):\n            ax_.grid(alpha=.25)\n            for s in (\"top\", \"right\"): ax_.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n    verdict = (\"counts as the correct pose (under 2 A)\" if r < 2.0\n               else \"would be scored as a wrong pose (2 A or more)\")\n    print(f\"RMSD to the crystal pose : {r:.2f} A   -> {verdict}\")\n    print(f\"furthest atom moved      : {per.max():.2f} A\")\n    print(f\"closest atom moved       : {per.min():.2f} A\")\n    return dict(rmsd=round(r, 2), max_move=round(float(per.max()), 2),\n                correct_pose=bool(r < 2.0))\n\n\ndef pharmacophore(resn=\"THA\", show=True):\n    \"\"\"The real 3-point geometry of the bound ligand.\"\"\"\n    atoms, X = ligand(resn)\n    idx = {a[\"name\"]: i for i, a in enumerate(atoms)}\n    ring = [idx[n] for n in (\"C1\", \"C2\", \"C3\", \"C4\", \"C5\", \"C6\") if n in idx]\n    pts = {}\n    if ring: pts[\"aromatic centroid\"] = X[ring].mean(axis=0)\n    for nm, lab in ((\"N15\", \"amine N (donor)\"), (\"N7\", \"ring N-H+ (donor)\")):\n        if nm in idx: pts[lab] = X[idx[nm]]\n    names = list(pts)\n    print(\"Three-point pharmacophore, measured in the crystal structure:\")\n    out = {}\n    for a, b in itertools.combinations(names, 2):\n        d = float(np.linalg.norm(pts[a] - pts[b]))\n        out[f\"{a} - {b}\"] = round(d, 2)\n        print(f\"   {a:<22} to {b:<22} {d:5.2f} A\")\n    if show and len(names) == 3:\n        f, ax = _fig(6.4, 5.2)\n        P2 = np.array([pts[n] for n in names])\n        ax.scatter(P2[:, 0], P2[:, 1], s=180,\n                   color=[PRIMARY, SECOND, OK][:len(names)], zorder=3)\n        for i, n in enumerate(names):\n            ax.annotate(n, (P2[i, 0], P2[i, 1]), fontsize=9,\n                        xytext=(8, 6), textcoords=\"offset points\")\n        for i, j in itertools.combinations(range(len(names)), 2):\n            ax.plot(P2[[i, j], 0], P2[[i, j], 1], color=MUTED, lw=1.4)\n            m = P2[[i, j]].mean(axis=0)\n            ax.text(m[0], m[1], f\"{np.linalg.norm(P2[i]-P2[j]):.2f} A\",\n                    fontsize=8, color=MUTED)\n        ax.set_aspect(\"equal\"); ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.margins(0.28)\n        ax.set_title(\"the pharmacophore is this triangle, not the skeleton\",\n                     pad=16)\n        plt.tight_layout(); plt.show()\n    return out\n\n\n# ---------------------------------------------------------------- L25 / L26\ndef score_pose(Y, resn=\"THA\", cutoff=6.0):\n    \"\"\"A deliberately simple score over REAL coordinates.\n\n    contact reward - clash penalty - desolvation-ish term. It is transparent\n    on purpose: every piece is printed, so the point is not that this is a\n    good scoring function but that you can see exactly what it rewards.\n    \"\"\"\n    atoms, _ = ligand(resn)\n    prot, P = protein()\n    d = np.linalg.norm(Y[:, None, :] - P[None, :, :], axis=2)\n    ser = np.array([VDW.get(a[\"elem\"], 1.7) for a in atoms])[:, None]\n    spr = np.array([VDW.get(r[\"elem\"], 1.7) for r in prot])[None, :]\n    s = ser + spr\n    near = d <= cutoff\n    gap = d - s\n    contact = float(np.sum(np.exp(-((gap[near]) ** 2) / 0.8)))\n    overlap = np.clip(-gap, 0, None)\n    clash = float(np.sum(overlap[near] ** 2) * 20)\n    polar_l = np.array([a[\"elem\"] in (\"N\", \"O\") for a in atoms])[:, None]\n    polar_p = np.array([r[\"elem\"] in (\"N\", \"O\") for r in prot])[None, :]\n    hb_mask = near & polar_l & polar_p & (d > 2.4) & (d < 3.5)\n    hbond = float(hb_mask.sum()) * 2.5\n    total = -(contact + hbond) + clash\n    return dict(score=round(total, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int((overlap > 0.4).any(axis=1).sum()),\n                overlapping_pairs=int((near & (overlap > 0)).sum()))\n\n\ndef dock_pose(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, axis=\"z\", resn=\"THA\",\n              show=True):\n    \"\"\"Move the real ligand in the real gorge and score it.\"\"\"\n    atoms, X = ligand(resn)\n    Y = move_ligand(dx, dy, dz, rot_deg, axis, resn)\n    sc = score_pose(Y, resn)\n    r = rmsd(X, Y)\n    if show:\n        prot, P = protein()\n        d = np.linalg.norm(Y[:, None, :] - P[None, :, :], axis=2)\n        nearby = np.where(d.min(axis=0) <= 8.0)[0]\n        f, ax = _fig(7.2, 6.0)\n        ax.scatter(P[nearby, 0], P[nearby, 1], s=14, color=\"#CBD5E1\",\n                   label=\"gorge atoms within 8 A\")\n        ax.scatter(X[:, 0], X[:, 1], s=190, facecolors=\"none\", edgecolors=\"#111827\",\n                   linewidths=1.6, zorder=4, label=\"crystal pose (open rings)\")\n        bad = np.zeros(len(Y), bool)\n        ov = np.clip((np.array([VDW.get(a[\"elem\"],1.7) for a in atoms])[:,None]\n                      + np.array([VDW.get(p[\"elem\"],1.7) for p in prot])[None,:]) - d, 0, None)\n        bad = (ov > 0.4).any(axis=1)\n        ax.scatter(Y[~bad, 0], Y[~bad, 1], s=70, color=SECOND, zorder=3,\n                   label=\"your pose (filled)\")\n        if bad.any():\n            ax.scatter(Y[bad, 0], Y[bad, 1], s=130, color=\"#DC2626\", marker=\"X\",\n                       zorder=3, label=\"clashing atom (> 0.4 A overlap)\")\n        ax.set_aspect(\"equal\"); ax.legend(fontsize=8, loc=\"best\")\n        ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.set_title(f\"score {sc['score']:+.1f}    RMSD to crystal {r:.2f} A\")\n        plt.tight_layout(); plt.show()\n    print(f\"score            {sc['score']:+8.2f}   (more negative is better)\")\n    print(f\"  contact        {sc['contact']:8.2f}\")\n    print(f\"  hydrogen bonds {sc['hbond']:8.2f}\")\n    print(f\"  clash penalty  {sc['clash']:8.2f}   = 20 x (sum of squared vdW overlaps)\")\n    print(f\"                            {sc['overlapping_pairs']} atom pairs inside their vdW sum; \"\n          f\"{sc['clashing_atoms']} atoms clash (> 0.4 A)\")\n    print(f\"RMSD to crystal  {r:8.2f} A\")\n    return dict(**sc, rmsd=round(r, 2))\n\n\ndef score_vs_truth(n=220, spread=4.0, seed=7, resn=\"THA\", show=True):\n    \"\"\"Generate real perturbed poses; plot score against RMSD to the crystal.\"\"\"\n    rng = np.random.default_rng(seed)\n    atoms, X = ligand(resn)\n    rows = []\n    for _ in range(n):\n        t = rng.normal(0, spread / 2, 3)\n        a = rng.uniform(-180, 180)\n        ax_ = \"xyz\"[rng.integers(0, 3)]\n        Y = move_ligand(*t, rot_deg=a, axis=ax_, resn=resn)\n        sc = score_pose(Y, resn)\n        rows.append((rmsd(X, Y), sc[\"score\"], sc[\"clashing_atoms\"]))\n    R = np.array([r[0] for r in rows]); S = np.array([r[1] for r in rows])\n    good_rmsd = R < 2.0\n    best = int(np.argmin(S))\n    if show:\n        f, ax = _fig(9.0, 5.4)\n        ax.scatter(R[~good_rmsd], S[~good_rmsd], s=26, color=MUTED,\n                   label=\"wrong pose (RMSD >= 2 A)\", alpha=.75)\n        ax.scatter(R[good_rmsd], S[good_rmsd], s=40, color=PRIMARY,\n                   label=\"correct pose (RMSD < 2 A)\")\n        ax.scatter([R[best]], [S[best]], s=150, marker=\"*\", color=SECOND,\n                   zorder=5, label=\"best-scoring pose\")\n        # every pose is plotted: linear scale between -100 and +100 (where the\n        # good poses are), logarithmic beyond it (heavily clashing poses)\n        lo = float(S.min()); hi = float(S.max())\n        ax.set_yscale(\"symlog\", linthresh=100.0, linscale=1.5)\n        ymin = min(-120.0, lo * 1.3)\n        ymax = max(200.0, hi * 1.6) if hi > 0 else 100.0\n        ax.set_ylim(ymin, ymax)\n        ticks = [t for t in (-1000, -300, -100, -50, 0, 50, 100, 300, 1000, 3000, 10000, 30000)\n                 if ymin <= t <= ymax]\n        ax.set_yticks(ticks); ax.set_yticklabels([f\"{t:,}\" for t in ticks])\n        ax.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())\n        ax.axhspan(100, ymax, color=MUTED, alpha=.06, lw=0)\n\n        ax.axvline(2.0, color=\"#B45309\", ls=\":\", lw=1.4)\n        ax.set_xlabel(\"RMSD to the crystal pose (A)\")\n        ax.set_ylabel(\"score (more negative = better)\\nlinear to +100, log scale above (shaded)\")\n        ax.legend(fontsize=8, loc=\"upper center\", bbox_to_anchor=(0.5, -0.14), ncol=3,\n                  frameon=False)\n        ax.set_title(f\"{n} real poses of tacrine in the real gorge (all poses shown)\")\n        plt.tight_layout(); plt.show()\n    picked_right = bool(R[best] < 2.0)\n    print(f\"{n} poses generated from the real crystal coordinates\")\n    print(f\"  poses within 2 A of the crystal : {int(good_rmsd.sum())}\")\n    print(f\"  best-scoring pose has RMSD      : {R[best]:.2f} A\")\n    print(f\"  did the score pick a correct pose? {'yes' if picked_right else 'NO'}\")\n    return dict(n=n, correct_available=int(good_rmsd.sum()),\n                best_rmsd=round(float(R[best]), 2),\n                best_score=round(float(S[best]), 2),\n                score_picked_correct=picked_right)\n\n\n# ---------------------------------------------------------------- L27\ndef empty_space(probe=1.6, grid=1.0, around=9.0, resn=\"THA\", show=True):\n    \"\"\"Find unfilled volume in the real gorge next to the bound ligand.\n\n    Only gorge atoms are considered and the grid is processed in chunks, so a\n    fine grid stays within memory.\n    \"\"\"\n    if not isinstance(resn, str):\n        raise TypeError(\"resn must be a residue name such as 'THA'; \"\n                        \"did you mean empty_space(probe=%r)?\" % resn)\n    if grid < 0.3:\n        raise ValueError(\"grid finer than 0.3 A is not useful here and is slow\")\n    atoms, X = ligand(resn)\n    prot, Pall = protein()\n    c = X.mean(axis=0)\n    # prune to atoms that could possibly matter\n    keep_p = np.linalg.norm(Pall - c, axis=1) <= (around + probe + 2.5)\n    P = Pall[keep_p]\n    rp = np.array([VDW.get(r[\"elem\"], 1.7) for r in np.array(prot)[keep_p]])[None, :]\n    rl = np.array([VDW.get(a[\"elem\"], 1.7) for a in atoms])[None, :]\n    rng_ = np.arange(-around, around + 1e-9, grid)\n    G = np.array([[x, y, z] for x in rng_ for y in rng_ for z in rng_]) + c\n    G = G[np.linalg.norm(G - c, axis=1) <= around]\n    free = np.empty(len(G), bool)\n    dlmin = np.empty(len(G))\n    notlig = np.empty(len(G), bool)\n    step = max(1, int(4e6 // max(len(P), 1)))\n    for i in range(0, len(G), step):\n        g = G[i:i + step]\n        dp = np.linalg.norm(g[:, None, :] - P[None, :, :], axis=2)\n        free[i:i + step] = (dp > (rp + probe)).all(axis=1)\n        dl = np.linalg.norm(g[:, None, :] - X[None, :, :], axis=2)\n        notlig[i:i + step] = (dl > (rl + probe)).all(axis=1)\n        dlmin[i:i + step] = dl.min(axis=1)\n    pocket = free & (dlmin < 8.0)\n    unfilled = pocket & notlig\n    if show:\n        zc = float(X[:, 2].mean()); slab = 2.0\n        sl = np.abs(G[:, 2] - zc) <= slab\n        lig = np.abs(X[:, 2] - zc) <= slab\n        f, ax = _fig(7.0, 6.8)\n        ax.scatter(G[pocket & ~notlig & sl, 0], G[pocket & ~notlig & sl, 1],\n                   s=9, color=\"#CBD5E1\",\n                   label=\"taken up by the ligand: probe centre within \"\n                         f\"(ligand vdW radius + {probe} A) of a tacrine atom\")\n        ax.scatter(G[unfilled & sl, 0], G[unfilled & sl, 1], s=12, color=OK,\n                   label=f\"free: a {probe} A probe centred here touches neither \"\n                         \"protein nor ligand\")\n        ax.scatter(X[lig, 0], X[lig, 1], s=80, color=SECOND, zorder=3,\n                   label=\"tacrine atoms in this slice\")\n        ax.set_aspect(\"equal\")\n        ax.legend(fontsize=8, loc=\"upper center\", bbox_to_anchor=(0.5, -0.11),\n                  frameon=False, ncol=1)\n        ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.set_title(f\"a {2*slab:.0f} A slice through the gorge, not the whole \"\n                     f\"volume\\n{int((unfilled & sl).sum())} of the \"\n                     f\"{int(unfilled.sum())} free points lie in this slice\",\n                     fontsize=10)\n        plt.tight_layout(); plt.show()\n    vol = float(unfilled.sum()) * grid ** 3\n    print(f\"probe radius             {probe} A\")\n    print(f\"grid points sampled      {len(G):,} at {grid} A spacing\")\n    print(f\"free and near the ligand {unfilled.sum():>8,}  -> about {vol:,.0f} A^3 (probe-centre space)\")\n    print(\"\\n  The volume is points x spacing^3: the space where the CENTRE of a\")\n    print(f\"  {probe} A probe fits, clear of every protein atom and of the ligand\")\n    print(f\"  (each atom inflated by its vdW radius + {probe} A). It is a probe-centre\")\n    print(\"  accessible volume, not the cavity volume: the empty cavity a new group\")\n    print(\"  could fill is larger, because every free point carries a probe-sized\")\n    print(\"  sphere around it.\")\n    print(\"\\n  That free space is what a de novo program would try to fill, and why\")\n    print(\"  a first design should NOT fill all of it.\")\n    return dict(probe=probe, grid=grid, grid_points=len(G),\n                unfilled=int(unfilled.sum()), volume_A3=round(vol, 1))\n\n\ndef score_ranking(n=220, spread=2.0, seed=7, top=10, resn=\"THA\"):\n    \"\"\"Rank real poses by score and show the RMSD of each. The question is not\n    only whether the top pose is right but how well the score SEPARATES.\"\"\"\n    rng = np.random.default_rng(seed)\n    atoms, X = ligand(resn)\n    rows = []\n    for _ in range(n):\n        t = rng.normal(0, spread / 2, 3)\n        Y = move_ligand(*t, rot_deg=rng.uniform(-180, 180),\n                        axis=\"xyz\"[rng.integers(0, 3)], resn=resn)\n        rows.append((rmsd(X, Y), score_pose(Y, resn)[\"score\"]))\n    rows.sort(key=lambda r: r[1])\n    print(f\"top {top} of {n} poses, ranked by score\")\n    print(f\"  {'rank':>4}  {'score':>9}  {'RMSD':>6}   verdict\")\n    for i, (r, sc) in enumerate(rows[:top], 1):\n        print(f\"  {i:>4}  {sc:>9.2f}  {r:>5.2f} A   \"\n              f\"{'correct' if r < 2 else 'WRONG POSE'}\")\n    wrong = sum(1 for r, _ in rows[:top] if r >= 2)\n    print(f\"\\n  {wrong} of the top {top} scoring poses are the wrong pose.\")\n    return dict(top=top, wrong_in_top=wrong,\n                rows=[(round(r, 2), round(s, 2)) for r, s in rows[:top]])\n\n\n_RING = {\"TRP\": {\"CD2\", \"CE2\", \"CE3\", \"CZ2\", \"CZ3\", \"CH2\"},\n         \"PHE\": {\"CG\", \"CD1\", \"CD2\", \"CE1\", \"CE2\", \"CZ\"},\n         \"TYR\": {\"CG\", \"CD1\", \"CD2\", \"CE1\", \"CE2\", \"CZ\"},\n         \"HIS\": {\"CG\", \"ND1\", \"CD2\", \"CE1\", \"NE2\"}}\n\n\ndef stacking(near=6.0, resn=\"THA\", show=True):\n    \"\"\"Is the ligand sandwiched? Measure each nearby aromatic ring's position\n    relative to the plane of the ligand itself.\"\"\"\n    atoms, X = ligand(resn)\n    prot, P = protein()\n    cen = X.mean(axis=0)\n    nrm = np.linalg.svd(X - cen)[2][2]\n    seen, rows = set(), []\n    for j, r in enumerate(prot):\n        kk = (r[\"resn\"], r[\"resi\"])\n        if r[\"resn\"] not in _RING or kk in seen:\n            continue\n        idx = [k for k, q in enumerate(prot)\n               if q[\"resn\"] == r[\"resn\"] and q[\"resi\"] == r[\"resi\"]\n               and q[\"name\"] in _RING[r[\"resn\"]]]\n        if len(idx) < 5:\n            continue\n        Q = P[idx]\n        qc = Q.mean(axis=0)\n        if np.linalg.norm(qc - cen) > near + 3.0:\n            continue\n        seen.add(kk)\n        qn = np.linalg.svd(Q - qc)[2][2]\n        signed = float(np.dot(qc - cen, nrm))\n        ang = float(np.degrees(np.arccos(min(1.0, abs(float(np.dot(nrm, qn)))))))\n        rows.append((f\"{r['resn']}{r['resi']}\", signed,\n                     float(np.linalg.norm(qc - cen)), ang))\n    rows.sort(key=lambda t: t[2])\n    rows = [t for t in rows if t[2] <= near + 1.0]\n    if show:\n        f, ax = _fig(7.6, 4.4)\n        for nm, sg, dd, an in rows:\n            st = an < 30\n            ax.barh(nm, sg, height=.6, color=(PRIMARY if sg > 0 else SECOND),\n                    alpha=(1.0 if st else 0.35),\n                    hatch=(None if st else \"///\"), edgecolor=\"white\")\n            ax.text(sg + (0.2 if sg > 0 else -0.2), nm,\n                    f\"{an:.0f} deg\" + (\"\" if st else \"  edge-on\"),\n                    va=\"center\", ha=(\"left\" if sg > 0 else \"right\"),\n                    fontsize=8, color=(\"#1F2937\" if st else MUTED))\n        ax.set_xlim(min(r[1] for r in rows) - 3.0, max(r[1] for r in rows) + 3.0)\n        ax.axvline(0, color=\"#1F2937\", lw=1.4)\n        ax.set_xlabel(\"signed distance from the ligand's plane (A)\")\n        ax.set_title(\"opposite sides of zero = sandwiched;\\nsolid bar = near-parallel = genuinely stacked\", fontsize=10)\n        plt.tight_layout(); plt.show()\n    print(\"aromatic rings near the ligand, measured against the ligand's own plane\")\n    print(f\"  {'residue':<9}{'side':>6}{'perp (A)':>10}{'centroid (A)':>14}\"\n          f\"{'planes (deg)':>14}\")\n    for nm, sg, dd, an in rows:\n        print(f\"  {nm:<9}{('above' if sg > 0 else 'below'):>6}{sg:>10.2f}\"\n              f\"{dd:>14.2f}{an:>14.1f}\")\n    above = [r for r in rows if r[1] > 0]\n    below = [r for r in rows if r[1] < 0]\n    print(f\"\\n  {len(above)} above the plane, {len(below)} below.\")\n    if above and below:\n        print(\"  Rings on both sides: the ligand is sandwiched.\")\n    print(\"  A plane angle under about 30 degrees means the rings are \"\n          \"roughly parallel,\\n  which is face-to-face stacking rather than \"\n          \"edge-on contact.\")\n    return dict(rows=[(n, round(s, 2), round(d, 2), round(a, 1))\n                      for n, s, d, a in rows],\n                above=len(above), below=len(below),\n                sandwiched=bool(above and below))\n", "crystal.py", "exec"), _cr.__dict__)
sys.modules["crystal"] = _cr
crystal = _cr

_sm = types.ModuleType("sims"); _sm.__dict__["__name__"] = "sims"
exec(compile("\"\"\"PHS 312 simulations. NumPy + matplotlib only - no install, no network.\n\nEvery function returns the numbers it plotted so the notebook can quiz on them.\n\"\"\"\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nPRIMARY, SECOND, MUTED = \"#0B6E8F\", \"#C2410C\", \"#6B7280\"\n\n\ndef _tw(text, width=76, indent=\"\"):\n    import textwrap\n    return \"\\n\".join(textwrap.fill(p, width, initial_indent=indent,\n                                   subsequent_indent=indent)\n                     for p in text.split(\"\\n\"))\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\n# ---------------------------------------------------------------- L22\ndef force_field_terms(bond_k=300., bond_r0=1.52, ang_k=60., ang_t0=109.5,\n                      tor_V=14., vdw_eps=0.3, vdw_sigma=3.6, q1q2=-0.3):\n    \"\"\"Each force-field term plotted over its own coordinate, in kcal/mol.\n\n    The van der Waals term uses the sigma form, 4 eps [(sigma/r)^12 - (sigma/r)^6]:\n    sigma is the separation where the vdW energy is ZERO, and the vdW minimum\n    (depth -eps) lies at r_m = 2^(1/6) sigma, about 1.12 sigma. The same curve\n    written with r_m is eps [(r_m/r)^12 - 2 (r_m/r)^6].\n    \"\"\"\n    f, axes = plt.subplots(1, 4, figsize=(15, 3.6))\n    r = np.linspace(1.1, 2.1, 200); Es = bond_k * (r - bond_r0) ** 2\n    t = np.linspace(80, 140, 200);  Eb = ang_k * np.radians(t - ang_t0) ** 2\n    d = np.linspace(-180, 180, 361); Et = tor_V / 2 * (1 + np.cos(np.radians(3 * d)))\n    x = np.linspace(3.0, 8.0, 300)\n    Ev = 4 * vdw_eps * ((vdw_sigma / x) ** 12 - (vdw_sigma / x) ** 6)\n    Ec = 332. * q1q2 / x\n    r_m = 2 ** (1 / 6.) * vdw_sigma\n    for ax, (xx, yy, xl, ti) in zip(axes, [\n            (r, Es, \"bond length (A)\", \"stretch\"),\n            (t, Eb, \"bond angle (deg)\", \"bend\"),\n            (d, Et, \"torsion (deg)\", \"torsion\"),\n            (x, Ev + Ec, \"separation (A)\", \"vdW + electrostatic\")]):\n        ax.plot(xx, yy, color=PRIMARY, lw=2)\n        ax.set_title(ti, fontsize=10); ax.set_xlabel(xl, fontsize=8)\n        ax.set_ylabel(\"energy (kcal/mol)\", fontsize=8)\n        ax.grid(alpha=.25); ax.axhline(0, color=MUTED, lw=.8)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    a4 = axes[3]\n    a4.plot(x, Ev, color=SECOND, lw=1.4, ls=\"--\", label=\"vdW term alone\")\n    for xv, lab, ha in ((vdw_sigma, \"sigma \", \"right\"), (r_m, \" r_m\", \"left\")):\n        if x[0] <= xv <= x[-1]:\n            a4.axvline(xv, color=SECOND, ls=\":\", lw=1)\n            a4.text(xv, 0.97, lab, transform=a4.get_xaxis_transform(),\n                    fontsize=7.5, color=SECOND, va=\"top\", ha=ha)\n    a4.legend(fontsize=7, loc=\"lower right\", frameon=False)\n    a4.set_title(f\"vdW + electrostatic\\nsigma {vdw_sigma:.2f} A (vdW = 0), \"\n                 f\"r_m {r_m:.2f} A (vdW min)\", fontsize=9)\n    plt.tight_layout(); plt.show()\n    k = int((Ev + Ec).argmin())\n    return dict(stretch_min=float(r[Es.argmin()]), torsion_barrier=float(Et.max()),\n                vdw_min_sep=float(x[k]), vdw_min_E=float((Ev + Ec).min()),\n                vdw_sigma=float(vdw_sigma), vdw_r_m=float(r_m),\n                vdw_alone_min_E=float(-vdw_eps), units=\"kcal/mol\")\n\n\n# ---------------------------------------------------------------- L23\ndef torsion_profile(V3=14.0, gauche_strain=3.8, step=10, start_angle=100.):\n    \"\"\"Butane-like relaxed torsion profile, plus where minimisation stops.\"\"\"\n    d = np.arange(-180, 181, step)\n    E = V3 / 2 * (1 + np.cos(np.radians(3 * d)))\n    E += gauche_strain * np.exp(-((d - 60) / 32.) ** 2)\n    E += gauche_strain * np.exp(-((d + 60) / 32.) ** 2)\n    E += 2.2 * gauche_strain * np.exp(-(d / 30.) ** 2)\n    E = E - E.min()\n    # steepest descent from start_angle on this surface\n    fine = np.arange(-180, 180.01, 0.5)\n    Ef = np.interp(fine, d, E)\n    i = int(np.argmin(np.abs(fine - start_angle)))\n    while True:\n        lo, hi = max(i - 1, 0), min(i + 1, len(fine) - 1)\n        j = lo if Ef[lo] < Ef[i] else (hi if Ef[hi] < Ef[i] else i)\n        if j == i: break\n        i = j\n    f, ax = _fig()\n    ax.plot(d, E, \"-o\", ms=3.5, color=PRIMARY, label=\"relaxed scan\")\n    ax.axvline(start_angle, color=MUTED, ls=\":\", label=f\"start {start_angle:.0f}\")\n    ax.plot([fine[i]], [Ef[i]], \"v\", ms=12, color=SECOND,\n            label=f\"minimisation stops at {fine[i]:.0f} deg\")\n    ax.set_xlabel(\"C1-C2-C3-C4 dihedral (degrees)\")\n    ax.set_ylabel(\"relative steric energy (kJ/mol)\")\n    ax.set_xticks(range(-180, 181, 60)); ax.legend(fontsize=8)\n    plt.tight_layout(); plt.show()\n    mins = [float(d[k]) for k in range(1, len(d) - 1)\n            if E[k] < E[k - 1] and E[k] < E[k + 1]]\n    return dict(global_min=float(d[E.argmin()]), local_minima=mins,\n                barrier_at_0=float(E[d == 0][0]), stops_at=float(fine[i]))\n\n\ndef search_cost(n_rot=7, step=30):\n    n = int(round(360 / step)) ** n_rot\n    secs = n * 0.05\n    return dict(conformations=n, hours=secs / 3600,\n                years=secs / 3600 / 24 / 365)\n\n\n# ---------------------------------------------------------------- L24\ndef anneal(T=300., n_steps=4000, seed=0):\n    \"\"\"Metropolis walk on a double-well surface: does it escape the local min?\n\n    Global minima sit at +/-180 deg (0 kJ/mol); a local minimum sits at\n    +60 deg (4 kJ/mol) behind a ~26 kJ/mol barrier. At low T the walker is\n    trapped; raising T lets it cross.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    PLATEAU, DEEP, SHALLOW, W = 30.0, 30.0, 26.0, 26.0\n\n    def E(a):\n        a = np.asarray(a, float)\n        return (PLATEAU\n                - DEEP * np.exp(-((a - 180) / W) ** 2)\n                - DEEP * np.exp(-((a + 180) / W) ** 2)\n                - SHALLOW * np.exp(-((a - 60) / W) ** 2))\n\n    pos = 60.0\n    visited = [pos]\n    kT = max(T, 1.) * 0.0083145                      # kJ/mol per K\n    for _ in range(n_steps):\n        trial = ((pos + rng.normal(0, 20) + 180) % 360) - 180\n        dE = float(E(trial) - E(pos))\n        if dE < 0 or rng.random() < np.exp(-min(dE / kT, 700)):\n            pos = trial\n        visited.append(pos)\n    visited = np.array(visited)\n\n    second_half = visited[len(visited) // 2:]\n    near_global = float(np.mean(np.abs(np.abs(second_half) - 180) < 50) * 100)\n    ever = bool(np.any(np.abs(np.abs(visited) - 180) < 50))\n\n    x = np.linspace(-180, 180, 721)\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))\n    a1.plot(x, E(x), color=PRIMARY, lw=2)\n    a1.set_xlabel(\"dihedral (deg)\"); a1.set_ylabel(\"energy (kJ/mol)\")\n    a1.set_title(\"energy surface\")\n    a1.plot([60], [E(60)], \"o\", color=SECOND, ms=10)\n    a1.annotate(\"start (local, 4 kJ/mol)\", (60, E(60) + 1.5), fontsize=8, ha=\"center\")\n    a1.annotate(\"global (0 kJ/mol)\", (180, E(180) + 1.5), fontsize=8, ha=\"right\")\n    bins = np.linspace(-180, 180, 73)\n    a2.hist(visited, bins=bins, color=\"#CBD5E1\", label=\"whole run (all steps)\")\n    a2.hist(second_half, bins=bins, color=PRIMARY, alpha=.9,\n            label=f\"second half (counted): {near_global:.1f}% near global\")\n    for lo_, hi_ in ((-180, -130), (130, 180)):\n        a2.axvspan(lo_, hi_, color=SECOND, alpha=.08, lw=0)\n    a2.text(155, 0.97, \"near\\nglobal\", transform=a2.get_xaxis_transform(),\n            ha=\"center\", va=\"top\", fontsize=7.5, color=SECOND)\n    a2.text(-155, 0.97, \"near\\nglobal\", transform=a2.get_xaxis_transform(),\n            ha=\"center\", va=\"top\", fontsize=7.5, color=SECOND)\n    a2.legend(fontsize=7.5, loc=\"upper center\", frameon=False)\n    a2.set_ylim(0, a2.get_ylim()[1] * 1.25)\n    a2.set_xlabel(\"dihedral visited (deg)\"); a2.set_ylabel(\"count of steps\")\n    a2.set_title(f\"T = {T:.0f} K   -   \"\n                 + (\"escaped the local minimum\" if near_global > 15\n                    else \"stayed trapped\")\n                 + \"\\n% near global = share of SECOND-HALF steps within 50 deg of +/-180\",\n                 fontsize=9.5)\n    for ax in (a1, a2):\n        ax.grid(alpha=.25)\n        for sp in (\"top\", \"right\"): ax.spines[sp].set_visible(False)\n    plt.tight_layout(); plt.show()\n    return dict(escaped=near_global > 15, pct_near_global=round(near_global, 1),\n                ever_reached_global=ever, final=round(float(visited[-1]), 1))\n\n\ndef rmsd_overlay(dx=0.0, dy=0.0, rot_deg=0.0):\n    \"\"\"Rigid-body move of a probe onto a template; reports RMSD.\"\"\"\n    tmpl = np.array([[0., 0.], [1.4, 0.3], [2.6, -0.4], [3.9, 0.2], [1.2, 1.7]])\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    probe = (tmpl - tmpl.mean(0)) @ R.T + tmpl.mean(0) + np.array([dx, dy])\n    rmsd = float(np.sqrt(((probe - tmpl) ** 2).sum(1).mean()))\n    f, ax = _fig(6.4, 4.6)\n    ax.plot(*tmpl.T, \"o-\", color=PRIMARY, ms=11, lw=2, label=\"template (rigid active)\")\n    ax.plot(*probe.T, \"s--\", color=SECOND, ms=10, lw=2, label=\"your analogue\")\n    for p, q in zip(tmpl, probe):\n        ax.plot([p[0], q[0]], [p[1], q[1]], color=MUTED, lw=.9, ls=\":\")\n    ax.set_title(f\"RMSD = {rmsd:.2f} A\"); ax.legend(fontsize=8); ax.set_aspect(\"equal\")\n    plt.tight_layout(); plt.show()\n    return dict(rmsd=rmsd, good=rmsd < 0.5)\n\n\n# ---------------------------------------------------------------- L25\n# A snug pocket, 12.0 x 8.0 A, holding a ligand that spans 5.0 A, with a\n# gap in the wall where a polar residue sits. Orientation matters: turned\n# across the pocket the ligand cannot fit without clashing.\n_A, _B = 6.0, 4.1\n_TH = np.linspace(0, 2 * np.pi, 25)[:-1]\n_TH = np.array([t for t in _TH\n                if abs(np.degrees(t) - 90) > 16])          # gap for the residue\nSITE = np.c_[_A * np.cos(_TH), _B * np.sin(_TH)]           # cavity wall atoms\nHBOND_POINT = np.array([0.0, 3.7])                         # polar partner\nLIGAND = np.array([[-2.5, -0.3], [-0.8, 0.3], [0.9, 0.2],\n                   [2.5, -0.4], [0.0, 1.3]])               # last atom is polar\nCLASH_AT, CONTACT_AT, HBOND_AT = 3.0, 3.9, 2.7\n\n\ndef dock_pose(x=0.0, y=0.0, rot_deg=0.0, show=True):\n    \"\"\"Place a rigid 5-atom ligand in a 2-D pocket it very nearly fills.\n\n    Score = -(shape contacts + H-bond) + steric clash penalty, so a MORE\n    NEGATIVE score is a better pose. A pose outside the pocket earns nothing\n    and tends to zero; a pose driven into the wall goes sharply positive.\n    \"\"\"\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    pose = LIGAND @ R.T + np.array([x, y])\n\n    d = np.linalg.norm(pose[:, None, :] - SITE[None, :, :], axis=2)\n    nearest = d.min(1)\n    bad = nearest < CLASH_AT\n    clash = float(np.sum(np.clip(CLASH_AT - nearest, 0, None) ** 2) * 12)\n    contact = float(np.sum(np.exp(-((nearest - CONTACT_AT) / 1.25) ** 2)) * 2.6)\n    hb_dist = float(np.linalg.norm(pose[4] - HBOND_POINT))\n    hbond = float(6.0 * np.exp(-((hb_dist - HBOND_AT) / 0.95) ** 2))\n    score = -(contact + hbond) + clash\n\n    if show:\n        f, ax = _fig(7.0, 5.4)\n        _full = np.linspace(0, 2 * np.pi, 160)\n        ax.fill(_A * np.cos(_full), _B * np.sin(_full), color=\"#DCE9EF\",\n                zorder=0)\n        ax.plot(np.append(SITE[:, 0], SITE[0, 0]),\n                np.append(SITE[:, 1], SITE[0, 1]), \"o\", color=PRIMARY,\n                ms=15, label=\"binding-site wall\", zorder=1)\n        ax.plot(*HBOND_POINT, \"*\", ms=22, color=\"#B45309\",\n                label=\"polar residue\", zorder=3)\n        chain = pose[:4]\n        ax.plot(chain[:, 0], chain[:, 1], \"-\", color=SECOND, lw=2.6,\n                alpha=.8, zorder=4)\n        ax.plot([chain[1, 0], pose[4, 0]], [chain[1, 1], pose[4, 1]], \"-\",\n                color=SECOND, lw=2.6, alpha=.8, zorder=4)\n        ok = ~bad\n        if ok.any():\n            ax.plot(pose[ok][:, 0], pose[ok][:, 1], \"o\", ms=14,\n                    color=SECOND, label=\"ligand atom\", zorder=5)\n        if bad.any():\n            ax.plot(pose[bad][:, 0], pose[bad][:, 1], \"X\", ms=19,\n                    color=\"#DC2626\", label=\"steric clash\", zorder=6)\n        ax.set_aspect(\"equal\")\n        ax.set_xlim(-9.5, 9.5); ax.set_ylim(-7, 7)\n        ax.legend(fontsize=8, loc=\"lower left\")\n        ax.set_title(f\"score {score:+.1f}    \"\n                     f\"contacts {contact:.1f}   H-bond {hbond:.1f}   \"\n                     f\"clash {clash:.1f}\")\n        plt.tight_layout(); plt.show()\n    return dict(score=round(score, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int(bad.sum()))\n\n\n# ---------------------------------------------------------------- L26\ndef score_vs_truth(cutoff=-7.0, seed=3):\n    \"\"\"Docking scores against crystallographic truth: the ranking problem.\"\"\"\n    rng = np.random.default_rng(seed)\n    n = 60\n    active = rng.random(n) < 0.25\n    rmsd = np.where(active, rng.gamma(2, .9, n), rng.gamma(5, 1.1, n))\n    score = -4.5 - 3.2 * active - rng.normal(0, 1.5, n)\n    picked = score <= cutoff\n    tp = int(np.sum(picked & active)); fp = int(np.sum(picked & ~active))\n    tot_act = int(active.sum())\n    hit = 100 * tp / max(picked.sum(), 1)\n    base = 100 * tot_act / n\n    f, ax = _fig(8.6, 4.6)\n    ax.scatter(score[~active], rmsd[~active], s=46, color=MUTED,\n               label=\"decoy\", alpha=.75)\n    ax.scatter(score[active], rmsd[active], s=56, color=PRIMARY,\n               label=\"true active\", alpha=.9)\n    ax.axvline(cutoff, color=SECOND, lw=2, ls=\"--\",\n               label=f\"your cutoff {cutoff:+.1f}\")\n    ax.axhline(2.0, color=\"#B45309\", lw=1.2, ls=\":\",\n               label=\"2 A - pose counts as correct\")\n    ax.set_xlabel(\"docking score (more negative = better)\")\n    ax.set_ylabel(\"RMSD to crystal pose (A)\")\n    ax.legend(fontsize=8)\n    ax.set_title(f\"picked {int(picked.sum())} compounds: {tp} actives, {fp} decoys\"\n                 f\"   |   hit rate {hit:.0f}% vs {base:.0f}% at random\")\n    plt.tight_layout(); plt.show()\n    return dict(picked=int(picked.sum()), true_actives_found=tp, decoys=fp,\n                total_actives=tot_act, hit_rate=round(hit, 1),\n                random_rate=round(base, 1),\n                enrichment=round(hit / base, 2) if base else 0.0)\n\n\n# ---------------------------------------------------------------- L27\ndef de_novo(h_bond_group=\"amidine\", near_group=\"phenyl\", far_group=\"isopropyl\",\n            linker_bonds=5, fill_site=False):\n    \"\"\"Grow and link fragments into a site with one polar and two greasy pockets.\"\"\"\n    HB = {\"amidine\": 5.0, \"tetrazole\": 4.2, \"phenol\": 3.0, \"methyl\": 0.2,\n          \"isopropyl\": 0.0}\n    GREASE = {\"phenyl\": 3.6, \"cyclohexyl\": 3.9, \"isopropyl\": 2.6,\n              \"methyl\": 1.2, \"amidine\": -1.5, \"tetrazole\": -1.8, \"phenol\": 0.6}\n    SYNTH = {\"amidine\": 1.0, \"tetrazole\": 1.6, \"phenol\": 0.3, \"phenyl\": 0.2,\n             \"cyclohexyl\": 0.5, \"isopropyl\": 0.1, \"methyl\": 0.0}\n    inter = HB.get(h_bond_group, 0) + GREASE.get(near_group, 0) + GREASE.get(far_group, 0)\n    ideal = 6\n    strain = 0.9 * abs(linker_bonds - ideal)\n    synth = SYNTH.get(h_bond_group, 0) + SYNTH.get(near_group, 0) + SYNTH.get(far_group, 0) \\\n        + 0.25 * max(linker_bonds - 8, 0)\n    overfill = 3.0 if fill_site else 0.0\n    total = inter - strain - synth - overfill\n    labels = [\"H-bond region\", \"near pocket\", \"far pocket\",\n              \"linker strain\", \"synthetic cost\", \"over-fill\"]\n    vals = [HB.get(h_bond_group, 0), GREASE.get(near_group, 0),\n            GREASE.get(far_group, 0), -strain, -synth, -overfill]\n    f, ax = _fig(9.2, 3.8)\n    cols = [PRIMARY if v >= 0 else SECOND for v in vals]\n    ax.barh(labels, vals, color=cols)\n    ax.axvline(0, color=\"k\", lw=.9)\n    ax.set_xlabel(\"contribution to design score\")\n    ax.set_title(f\"design score {total:+.1f}\"\n                 + (\"   (over-filled: no room left to optimise)\" if fill_site else \"\"))\n    plt.tight_layout(); plt.show()\n    return dict(score=round(total, 2), interactions=round(inter, 2),\n                strain=round(strain, 2), synthetic_cost=round(synth, 2),\n                overfill_penalty=overfill)\n\n\n# ---------------------------------------------------------------- L29 / L30\nCOUMARIN = dict(\n    X=[\"H\", \"Me\", \"OH\", \"OMe\", \"OCF3\", \"NH2\", \"F\", \"Cl\", \"CF3\", \"CN\", \"NO2\"],\n    pIC50=np.array([8.36, 8.38, 8.01, 8.44, 7.94, 7.46, 8.55, 8.48, 8.24, 7.97, 8.59]),\n    pi=np.array([0., .49, -.67, -.09, 1.02, -1.23, .14, .71, .88, -.57, -.26]),\n    Vdw=np.array([.08, 1.01, .49, 1.49, 1.60, .67, .36, 1.07, 1.11, 1.14, 1.20]),\n    F=np.array([0., -.04, .28, .25, .37, .02, .42, .40, .37, .50, .66]))\n\n_PI = {\"H\": 0., \"Me\": .56, \"Cl\": .71, \"NO2\": -.28, \"OMe\": -.02, \"NH2\": -1.23, \"F\": .14}\n_MR = {\"H\": .103, \"Me\": .565, \"Cl\": .603, \"NO2\": .736, \"OMe\": .787, \"NH2\": .542, \"F\": .092}\n_F = {\"H\": 0., \"Me\": .01, \"Cl\": .42, \"NO2\": .65, \"OMe\": .29, \"NH2\": .08, \"F\": .45}\n_R = {\"H\": 0., \"Me\": -.18, \"Cl\": -.19, \"NO2\": .13, \"OMe\": -.56, \"NH2\": -.74, \"F\": -.39}\n_TAC = [(\"tacrine\", \"H\", \"H\", 6.59), (\"I\", \"Me\", \"H\", 5.09), (\"II\", \"H\", \"Me\", 7.00),\n        (\"III\", \"Cl\", \"H\", 6.26), (\"IV\", \"H\", \"Cl\", 8.00), (\"V\", \"NO2\", \"H\", 5.52),\n        (\"VI\", \"H\", \"NO2\", 7.55), (\"VII\", \"H\", \"OMe\", 6.45), (\"VIII\", \"NH2\", \"H\", 5.42),\n        (\"IX\", \"H\", \"F\", 7.06), (\"X\", \"Cl\", \"Cl\", 6.32), (\"XI\", \"OMe\", \"OMe\", 5.29)]\n\ndef tacrine_table():\n    rows = []\n    for cid, X, Y, p in _TAC:\n        rows.append(dict(cpd=cid, X=X, Y=Y, pIC50=p, piX=_PI[X], piY=_PI[Y],\n                         MRX=_MR[X], MRY=_MR[Y], F_XY=_F[X] + _F[Y], R_XY=_R[X] + _R[Y]))\n    return rows\n\n\ndef _lstsq(X, y):\n    X = np.column_stack([np.asarray(X, float).reshape(len(y), -1), np.ones(len(y))])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    pred = X @ beta\n    ss_res = float(((y - pred) ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())\n    r2 = 1 - ss_res / ss_tot\n    n, p = len(y), X.shape[1]\n    s = float(np.sqrt(ss_res / (n - p))) if n > p else float(\"nan\")\n    return beta, pred, r2, s\n\n\ndef qsar_single(param=\"pi\", quadratic=False):\n    \"\"\"One descriptor at a time on the coumarin MAO-B series.\"\"\"\n    y = COUMARIN[\"pIC50\"]; x = COUMARIN[param]\n    cols = np.column_stack([x, x ** 2]) if quadratic else x\n    beta, pred, r2, s = _lstsq(cols, y)\n    f, ax = _fig(7.6, 4.4)\n    ax.scatter(x, y, s=70, color=PRIMARY, zorder=3)\n    for xi, yi, t in zip(x, y, COUMARIN[\"X\"]):\n        ax.annotate(t, (xi, yi), fontsize=8, xytext=(4, 4), textcoords=\"offset points\")\n    xs = np.linspace(x.min(), x.max(), 200)\n    ys = beta[0] * xs + beta[1] * xs ** 2 + beta[2] if quadratic else beta[0] * xs + beta[1]\n    ax.plot(xs, ys, color=SECOND, lw=2,\n            label=(\"quadratic\" if quadratic else \"linear\") + f\"   r = {np.sqrt(max(r2,0)):.3f}   s = {s:.3f}\")\n    ax.set_xlabel({\"pi\": \"substituent hydrophobicity  pi\",\n                   \"Vdw\": \"van der Waals volume\", \"F\": \"inductive factor  F\"}[param])\n    ax.set_ylabel(\"pIC50 (MAO-B)\"); ax.legend(fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(param=param, quadratic=quadratic,\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3))\n\n\ndef qsar_multi(use=(\"MRX\",), show=True, lim=None):\n    \"\"\"Add descriptors and watch r rise while leave-one-out gets worse.\n    lim: optional (low, high) axis range, so that two plots can share the same scale.\"\"\"\n    rows = tacrine_table()\n    y = np.array([r[\"pIC50\"] for r in rows])\n    X = np.column_stack([[r[c] for r in rows] for c in use])\n    beta, pred, r2, s = _lstsq(X, y)\n    loo = []\n    for i in range(len(y)):\n        m = np.ones(len(y), bool); m[i] = False\n        b, _, _, _ = _lstsq(X[m], y[m])\n        loo.append(float(np.dot(np.append(X[i], 1.0), b)))\n    loo = np.array(loo)\n    q2 = 1 - float(((y - loo) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())\n    own = [float(min(y.min(), pred.min(), loo.min()) - .4), float(max(y.max(), pred.max(), loo.max()) + .4)]\n    if show:\n        f, ax = _fig(5.6, 5.2)\n        ax.scatter(pred, y, s=72, color=PRIMARY, label=\"fitted\", zorder=3)\n        ax.scatter(loo, y, s=52, color=SECOND, marker=\"s\",\n                   label=\"left-out prediction\", alpha=.85, zorder=3)\n        lim = list(lim) if lim is not None else own\n        ax.plot(lim, lim, \"k--\", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)\n        ax.set_xlabel(\"predicted pIC50\"); ax.set_ylabel(\"observed pIC50\")\n        ax.set_title(\" + \".join(use) + f\"\\nr = {np.sqrt(max(r2,0)):.3f}    \"\n                     f\"s = {s:.3f}    q2 = {q2:.3f}\")\n        ax.legend(fontsize=8); plt.tight_layout(); plt.show()\n    return dict(descriptors=list(use), n=len(y), k=len(use),\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3),\n                q2=round(q2, 3), compounds_per_parameter=round(len(y) / len(use), 1),\n                plot_range=own)\n\n\n# ---------------------------------------------------------------- L31\nBIOISOSTERE = {\n    \"carboxylic acid (COOH)\": dict(pKa=4.2, logP=0.0, MW=45, HBD=1, HBA=2, note=\"parent\"),\n    \"tetrazole\":              dict(pKa=4.9, logP=0.5, MW=69, HBD=1, HBA=4, note=\"classic acid mimic\"),\n    \"acylsulfonamide\":        dict(pKa=4.5, logP=0.6, MW=121, HBD=1, HBA=4, note=\"acid mimic, bulkier\"),\n    \"2,6-difluorophenol\":     dict(pKa=7.1, logP=1.9, MW=130, HBD=1, HBA=1, note=\"lipophilic acid mimic\"),\n    \"hydroxamic acid\":        dict(pKa=9.0, logP=-0.4, MW=61, HBD=2, HBA=3, note=\"weaker acid, metal binder\"),\n    \"primary amide (CONH2)\":  dict(pKa=None, logP=-1.5, MW=44, HBD=2, HBA=2, note=\"neutral\"),\n    \"methyl ester\":           dict(pKa=None, logP=0.6, MW=59, HBD=0, HBA=2, note=\"neutral, hydrolysable\"),\n}\n\ndef bioisostere_swap(replacement=\"tetrazole\", need_acidic=True, need_permeable=True):\n    p = BIOISOSTERE[\"carboxylic acid (COOH)\"]; r = BIOISOSTERE[replacement]\n    keeps_acid = r[\"pKa\"] is not None and r[\"pKa\"] < 6.5\n    more_lipo = r[\"logP\"] > p[\"logP\"]\n    verdict = []\n    if need_acidic:\n        verdict.append((\"keeps the ionised acid at pH 7.4\", keeps_acid))\n    if need_permeable:\n        verdict.append((\"raises lipophilicity for permeability\", more_lipo))\n    verdict.append((\"molecular weight increase under 60 Da\", r[\"MW\"] - p[\"MW\"] < 60))\n    f, ax = _fig(8.8, 3.6)\n    labels = [\"pKa\", \"logP\", \"MW (Da)\", \"HBD\", \"HBA\"]\n    par = [p[\"pKa\"] if p[\"pKa\"] else 0, p[\"logP\"], p[\"MW\"] / 20, p[\"HBD\"], p[\"HBA\"]]\n    rep = [r[\"pKa\"] if r[\"pKa\"] else 0, r[\"logP\"], r[\"MW\"] / 20, r[\"HBD\"], r[\"HBA\"]]\n    i = np.arange(len(labels)); w = .38\n    ax.bar(i - w / 2, par, w, label=\"carboxylic acid (parent)\", color=MUTED)\n    ax.bar(i + w / 2, rep, w, label=replacement, color=PRIMARY)\n    ax.set_xticks(i); ax.set_xticklabels(labels)\n    ax.set_title(f\"{replacement} - {r['note']}   (MW shown /20)\")\n    ax.legend(fontsize=8); ax.axhline(0, color=\"k\", lw=.8)\n    plt.tight_layout(); plt.show()\n    for text, ok in verdict:\n        print((\"  MET      \" if ok else \"  NOT MET  \") + text)\n    return dict(replacement=replacement, pKa=r[\"pKa\"], logP=r[\"logP\"],\n                dMW=r[\"MW\"] - p[\"MW\"], requirements_met=sum(1 for _, o in verdict if o),\n                requirements_total=len(verdict))\n\n\n# ---------------------------------------------------------------- L32\ndef prodrug_kinetics(promoiety=\"ethyl ester\", esterase_activity=1.0, hours=8.0):\n    \"\"\"First-order activation: prodrug -> active drug, and patient variability.\"\"\"\n    K = {\"ethyl ester\": 0.45, \"5-indanyl ester (EWG)\": 1.60,\n         \"pivaloyl ester (hindered)\": 0.10, \"amide\": 0.02, \"no promoiety\": 0.0}\n    k = K[promoiety] * max(esterase_activity, 0.0)\n    t = np.linspace(0, hours, 400)\n    pro = np.exp(-k * t); act = 1 - pro\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))\n    a1.plot(t, pro * 100, color=MUTED, lw=2, label=\"prodrug (inactive)\")\n    a1.plot(t, act * 100, color=PRIMARY, lw=2.4, label=\"active drug\")\n    a1.set_xlabel(\"hours\"); a1.set_ylabel(\"% of dose\"); a1.legend(fontsize=8)\n    a1.set_title(f\"{promoiety}   k = {k:.2f} /h\")\n    for lab, mult, col in [(\"low enzyme (x0.3)\", .3, SECOND),\n                           (\"typical (x1)\", 1., PRIMARY),\n                           (\"high enzyme (x3)\", 3., \"#15803D\")]:\n        a2.plot(t, (1 - np.exp(-K[promoiety] * mult * t)) * 100, lw=2.2,\n                color=col, label=lab)\n    a2.set_xlabel(\"hours\"); a2.set_ylabel(\"% converted to active drug\")\n    a2.set_title(\"same prodrug, three patients\"); a2.legend(fontsize=8)\n    for ax in (a1, a2):\n        ax.grid(alpha=.25); ax.set_ylim(-3, 103)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    plt.tight_layout(); plt.show()\n    half = float(np.log(2) / k) if k > 0 else float(\"inf\")\n    at2 = float((1 - np.exp(-k * 2)) * 100)\n    spread = float(abs((1 - np.exp(-K[promoiety] * 3 * 2)) -\n                       (1 - np.exp(-K[promoiety] * .3 * 2))) * 100)\n    return dict(promoiety=promoiety, k_per_h=round(k, 3),\n                half_life_h=(round(half, 2) if np.isfinite(half) else None),\n                pct_active_at_2h=round(at2, 1),\n                patient_spread_at_2h_pct=round(spread, 1))\n\n\n# ---------------------------------------------------------------- L22 extra\ndef method_choice(atoms=20, want=\"partial charges and an electrostatic map\"):\n    \"\"\"Which level of theory answers the question at an affordable cost.\"\"\"\n    needs_electrons = want in (\"partial charges and an electrostatic map\",\n                               \"heat of formation\",\n                               \"transition state of a reaction\")\n    if not needs_electrons:\n        rec = \"Molecular mechanics\"\n        why = (\"The question is about geometry and sterics only. A force field \"\n               \"treats atoms as spheres on springs and never computes an \"\n               \"electron distribution, which is all the speed comes from.\")\n    elif atoms <= 30:\n        rec = \"Ab initio quantum mechanics\"\n        why = (\"Electron distribution is required and the molecule is small \"\n               \"enough that a first-principles calculation is affordable.\")\n    elif atoms <= 150:\n        rec = \"Semi-empirical quantum mechanics\"\n        why = (\"Electron distribution is required but the molecule is too large \"\n               \"for ab initio, so experimentally fitted parameters are used to \"\n               \"simplify the integrals.\")\n    else:\n        rec = \"Molecular mechanics, and reconsider the question\"\n        why = (\"No quantum method is affordable at this size. Either cut the \"\n               \"molecule down to the fragment that matters, or accept that \"\n               \"electronic properties are out of reach.\")\n    # illustrative scaling, the same model as the lecture-notes cost calculator:\n    # relative time = prefactor x N^p / 20^2  (MM for a 20-atom molecule = 1)\n    model = {\"Molecular mechanics\": (1.0, 2), \"Semi-empirical quantum mechanics\": (1e2, 3),\n             \"Ab initio quantum mechanics\": (1e4, 4)}\n    cost = {k: pf * atoms ** p / 20. ** 2 for k, (pf, p) in model.items()}\n    f, ax = _fig(8.6, 3.1)\n    names = list(cost); vals = [cost[n] for n in names]\n    cols = [PRIMARY if n in rec else MUTED for n in names]\n    ax.barh(names, vals, color=cols)\n    for i_, v_ in enumerate(vals):\n        ax.text(v_, i_, f\"  {v_:.3g}\", va=\"center\", fontsize=8, color=\"#1F2937\")\n    ax.set_xscale(\"log\"); ax.set_xlim(min(vals) / 3, max(vals) * 60)\n    ax.set_xlabel(\"relative time = prefactor x N^p / 20^2 (log scale; MM at 20 atoms = 1)\")\n    ax.set_title(f\"{atoms} atoms   -   {want}\")\n    plt.tight_layout(); plt.show()\n    print(\"RECOMMEND:\", rec); print()\n    print(_tw(why))\n    print()\n    print(\"relative time (illustrative scaling MM N^2, semi-empirical 100 N^3,\")\n    print(\"ab initio 10^4 N^4, divided by 20^2 - the same model as the cost calculator):\")\n    for k_ in names:\n        print(f\"  {k_:<34} {cost[k_]:>12.3g}\")\n    return dict(atoms=atoms, question=want, recommend=rec,\n                needs_electron_distribution=needs_electrons,\n                relative_cost={k: float(v) for k, v in cost.items()})\n\n\n# ---------------------------------------------------------------- L25 extra\ndef site_surface(probe_radius=1.4, show_probe=True):\n    \"\"\"van der Waals surface versus the solvent-accessible (Connolly) surface.\n\n    Two gate atoms are placed so the opening between their van der Waals\n    surfaces is exactly GAP angstroms. A spherical probe passes only if its\n    radius does not exceed GAP/2, so the narration below is derived from the\n    coordinates rather than asserted.\n    \"\"\"\n    R = 1.7\n    GAP = 1.60                       # surface-to-surface opening, angstroms\n    d = 2 * R + GAP                  # required centre-to-centre separation\n    gate = [(-d / 2, 0.0, R), (d / 2, 0.0, R)]\n    wall = [(-5.6, -1.0, R), (-5.0, -3.0, R), (-3.4, -4.4, R),\n            (-1.2, -5.0, R), (1.2, -5.0, R), (3.4, -4.4, R),\n            (5.0, -3.0, R), (5.6, -1.0, R)]\n    atoms = gate + wall\n    half_gap = GAP / 2\n    fits = probe_radius <= half_gap\n\n    gx, gy = np.meshgrid(np.linspace(-8, 8, 460), np.linspace(-7.5, 4.5, 360))\n    vdw = np.zeros_like(gx, bool); excluded = np.zeros_like(gx, bool)\n    for (cx, cy, r) in atoms:\n        dd = (gx - cx) ** 2 + (gy - cy) ** 2\n        vdw |= dd <= r ** 2\n        excluded |= dd <= (r + probe_radius) ** 2\n\n    # The probe centre can only reach free points CONNECTED to the outside.\n    # A pocket of free points sealed behind the gate is a void the probe could\n    # sit in but never get into, so it is treated as unreachable (flood fill\n    # from the edge of the box; NumPy only).\n    free = ~excluded\n    reach = np.zeros_like(free)\n    reach[0, :] = free[0, :]; reach[-1, :] = free[-1, :]\n    reach[:, 0] = free[:, 0]; reach[:, -1] = free[:, -1]\n    while True:\n        grow = reach.copy()\n        grow[1:, :] |= reach[:-1, :]; grow[:-1, :] |= reach[1:, :]\n        grow[:, 1:] |= reach[:, :-1]; grow[:, :-1] |= reach[:, 1:]\n        grow &= free\n        if (grow == reach).all():\n            break\n        reach = grow\n    if fits:                                 # the gate is open (decided from the\n        reach = free                         # coordinates above), so the space behind\n    sealed = free & ~reach                   # it counts as reachable even when the grid\n    blocked = ~reach                         # pinches at exactly GAP/2\n\n    f, ax = _fig(7.4, 6.0)\n    ax.contourf(gx, gy, blocked.astype(float), levels=[.5, 1.5],\n                colors=[\"#CBD5E1\"], alpha=.8)\n    ax.contour(gx, gy, blocked.astype(float), levels=[.5], colors=[SECOND],\n               linewidths=2.2)\n    ax.contourf(gx, gy, vdw.astype(float), levels=[.5, 1.5], colors=[MUTED],\n                alpha=.95)\n    ax.contour(gx, gy, vdw.astype(float), levels=[.5], colors=[\"#334155\"],\n               linewidths=1.3)\n    if show_probe:\n        ax.add_patch(plt.Circle((0.0, 0.0), probe_radius, fill=False,\n                                ec=PRIMARY, lw=2.4, ls=\"--\"))\n        ax.plot([0], [0], \"x\", color=PRIMARY, ms=9)\n    ax.annotate(\"\", xy=(-d / 2 + R, 0), xytext=(d / 2 - R, 0),\n                arrowprops=dict(arrowstyle=\"<->\", color=PRIMARY, lw=1.6))\n    ax.text(0, .45, f\"{GAP:.2f} A opening\", ha=\"center\", color=PRIMARY,\n            fontsize=9)\n    ax.set_aspect(\"equal\"); ax.set_xlim(-8, 8); ax.set_ylim(-7.5, 4.5)\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.set_title(f\"probe radius {probe_radius:.2f} A\\n\"\n                 \"dark grey = atoms (van der Waals)    \"\n                 \"orange outline = surface this probe can reach\\n\"\n                 \"pale grey = where the probe centre cannot get to, including any \"\n                 \"space sealed behind the gate\", fontsize=9.5)\n    plt.tight_layout(); plt.show()\n\n    print(f\"opening between the gate atoms : {GAP:.2f} A\")\n    print(f\"largest probe that passes      : {half_gap:.2f} A radius\")\n    print(f\"this probe                     : {probe_radius:.2f} A radius\")\n    print()\n    print(_tw(\n        (\"The probe fits through the opening, so the cavity behind the gate is \"\n         \"mapped as accessible and a ligand atom may be placed there.\"\n         if fits else\n         \"The probe is too large to pass, so the opening is sealed over. The \"\n         \"cavity behind it never appears on the docking surface, which is the \"\n         \"point: no ligand atom could reach it either.\")))\n    if sealed.any():\n        print()\n        print(_tw(\"Behind the gate there is still a little free space the probe \"\n                  \"centre would fit into, but the probe cannot get there from \"\n                  \"outside, so it is shaded as unreachable and has no orange \"\n                  \"outline.\"))\n    return dict(probe_radius=probe_radius, opening_angstrom=GAP,\n                max_probe_radius_that_fits=half_gap,\n                probe_passes=bool(fits),\n                excluded_volume_grid_points=int(excluded.sum()))\n\n\n# ------------------------------------------------- L31, real SwissBioisostere\ndef bioisostere_real(parent=\"carboxylic acid\", replacement=\"primary amide\",\n                     need_not_worse=0.80, max_mw_increase=60.0, show=True):\n    \"\"\"Judge a replacement against real matched-pair data.\n\n    Data: SwissBioisostere (CC-BY 4.0), ChEMBL 36. See swissbio.py for the\n    full citation and for what was changed on retrieval.\n    \"\"\"\n    import swissbio as sb\n    rows = {r[\"name\"]: r for r in sb.table(parent)}\n    if replacement not in rows:\n        print(f\"'{replacement}' is not in the curated {parent} list.\")\n        print(\"Available:\", \", \".join(sorted(rows)[:12]), \"...\")\n        return None\n    r = rows[replacement]\n    n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n    not_worse = (r[\"better\"] + r[\"similar\"]) / n\n    total = sb.QUERIES[parent][\"total\"]\n\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.6, 4.2))\n        # how activity changed across the real pairs\n        a1.barh([\"worse\", \"similar\", \"better\"],\n                [r[\"worse\"], r[\"similar\"], r[\"better\"]],\n                color=[SECOND, MUTED, \"#15803D\"])\n        a1.set_xlabel(f\"matched molecular pairs (n = {n})\")\n        a1.set_title(f\"{replacement}\\nreplacing {parent} in real compounds\")\n        for s in (\"top\", \"right\"): a1.spines[s].set_visible(False)\n\n        # where this replacement sits against the others by frequency\n        tab = sb.table(parent)\n        ranks = [x[\"rank\"] for x in tab]; freqs = [x[\"freq\"] for x in tab]\n        cols = [PRIMARY if x[\"name\"] == replacement else \"#CBD5E1\" for x in tab]\n        a2.bar(ranks, freqs, color=cols)\n        a2.set_xlabel(f\"rank by frequency (top {len(tab)} of {total:,})\")\n        a2.set_ylabel(\"matched pairs\")\n        a2.set_title(f\"this replacement is rank {r['rank']} of {total:,}\")\n        for s in (\"top\", \"right\"): a2.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n\n    print(f\"SMILES              {r['smiles']}\")\n    print(f\"frequency           {r['freq']:,} matched pairs  (rank {r['rank']} of {total:,})\")\n    print(f\"activity outcome    {r['better']} better / {r['similar']} similar / {r['worse']} worse\")\n    print(f\"not worse           {not_worse*100:.0f}% of pairs\")\n    print(f\"mean change         logP {r['dlogp']:+.2f}   tPSA {r['dtpsa']:+.1f}   MW {r['dmw']:+.1f}\")\n    print()\n    checks = [(f\"at least {need_not_worse*100:.0f}% of pairs not worse\", not_worse >= need_not_worse),\n              (f\"molecular weight increase under {max_mw_increase:.0f} Da\",\n               r[\"dmw\"] < max_mw_increase)]\n    for text, okd in checks:\n        print((\"  MET      \" if okd else \"  NOT MET  \") + text)\n    return dict(name=replacement, smiles=r[\"smiles\"], freq=r[\"freq\"], rank=r[\"rank\"],\n                not_worse=round(not_worse, 3), dlogp=r[\"dlogp\"], dtpsa=r[\"dtpsa\"],\n                dmw=r[\"dmw\"], met=sum(1 for _, o in checks if o), total_checks=len(checks))\n\n\ndef _label_points(f, ax, pts, names, s=70, fontsize=8):\n    \"\"\"Label every point without overlaps: try several offsets around each marker and keep the first\n    that clears the other markers, the labels already placed and the axes edge. A point with no free\n    slot gets its number instead, and the numbered names are returned for a key under the plot.\"\"\"\n    f.canvas.draw()\n    rend = f.canvas.get_renderer()\n    disp = ax.transData.transform(pts)\n    rad = (s ** 0.5) / 2 * f.dpi / 72 + 1.5\n    marks = [(x - rad, y - rad, x + rad, y + rad) for x, y in disp]\n    abox = ax.get_window_extent(rend)\n    placed, key = [], []\n    offs = [(6, 3, \"left\", \"bottom\"), (6, -3, \"left\", \"top\"), (-6, 3, \"right\", \"bottom\"),\n            (-6, -3, \"right\", \"top\"), (0, 8, \"center\", \"bottom\"), (0, -8, \"center\", \"top\"),\n            (10, 12, \"left\", \"bottom\"), (10, -12, \"left\", \"top\"), (-10, 12, \"right\", \"bottom\"),\n            (-10, -12, \"right\", \"top\"), (0, 18, \"center\", \"bottom\"), (0, -18, \"center\", \"top\")]\n    def clash(b, own):\n        for k, (x0, y0, x1, y1) in enumerate(placed + marks):\n            if k == len(placed) + own:\n                continue\n            if b.x0 < x1 and b.x1 > x0 and b.y0 < y1 and b.y1 > y0:\n                return True\n        return b.x0 < abox.x0 or b.x1 > abox.x1 or b.y0 < abox.y0 or b.y1 > abox.y1\n    order = sorted(range(len(pts)), key=lambda k: -disp[k][1])\n    for k in order:\n        done = False\n        for dx, dy, ha, va in offs:\n            t = ax.annotate(names[k], pts[k], fontsize=fontsize, xytext=(dx, dy), textcoords=\"offset points\",\n                            ha=ha, va=va)\n            b = t.get_window_extent(rend)\n            if not clash(b, k):\n                placed.append((b.x0 - 1, b.y0 - 1, b.x1 + 1, b.y1 + 1)); done = True\n                if (dx, dy) != offs[0][:2]:\n                    px, py = disp[k]\n                    qx, qy = min(max(px, b.x0), b.x1), min(max(py, b.y0), b.y1)\n                    sc = 72.0 / f.dpi\n                    ax.annotate(\"\", pts[k], xytext=((qx - px) * sc, (qy - py) * sc), textcoords=\"offset points\",\n                                arrowprops=dict(arrowstyle=\"-\", color=\"#94A3B8\", lw=0.6, shrinkA=0, shrinkB=3))\n                break\n            t.remove()\n        if not done:\n            num = str(len(key) + 1)\n            for dx, dy, ha, va in offs:\n                t = ax.annotate(num, pts[k], fontsize=fontsize, xytext=(dx, dy), textcoords=\"offset points\",\n                                ha=ha, va=va, fontweight=\"bold\")\n                b = t.get_window_extent(rend)\n                if not clash(b, k):\n                    break\n                t.remove()\n            else:\n                t = ax.annotate(num, pts[k], fontsize=fontsize, xytext=(0, 0), textcoords=\"offset points\",\n                                ha=\"center\", va=\"center\", color=\"white\", fontweight=\"bold\")\n                b = t.get_window_extent(rend)\n            placed.append((b.x0, b.y0, b.x1, b.y1))\n            key.append(f\"{num} = {names[k]}\")\n    return key\n\n\ndef bioisostere_ranking(parent=\"carboxylic acid\", min_freq=300, show=True):\n    \"\"\"Frequency is not success: rank the common replacements by outcome.\"\"\"\n    import swissbio as sb\n    tab = [r for r in sb.table(parent) if r[\"freq\"] >= min_freq]\n    for r in tab:\n        n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n        r[\"not_worse\"] = (r[\"better\"] + r[\"similar\"]) / n if n else 0\n    by_freq = sorted(tab, key=lambda r: -r[\"freq\"])\n    by_ok = sorted(tab, key=lambda r: -r[\"not_worse\"])\n    if show:\n        f, ax = _fig(10.4, 6.2)\n        pts = [(r[\"freq\"], r[\"not_worse\"] * 100) for r in tab]\n        ax.scatter([p[0] for p in pts], [p[1] for p in pts], s=50, color=PRIMARY, zorder=3)\n        ax.set_xscale(\"log\")\n        ax.set_xlim(min(p[0] for p in pts) / 1.12, max(p[0] for p in pts) * 1.6)\n        ys = [p[1] for p in pts]\n        ax.set_ylim(min(ys) - 2.5, max(ys) + 2.5)\n        ax.set_xlabel(f\"how often it was done (matched pairs, log scale)\")\n        ax.set_ylabel(\"% of pairs where activity was not worse\")\n        ax.set_title(f\"replacing {parent}: popularity against outcome\")\n        plt.tight_layout()\n        names = [r[\"name\"] for r in tab]\n        key = _label_points(f, ax, pts, names, s=50, fontsize=7.5)\n        if key:   # make room for the key, then place the labels again on the final layout\n            for t in list(ax.texts):\n                t.remove()\n            rows = (len(key) + 3) // 4\n            plt.tight_layout(rect=(0, 0.035 * rows, 1, 1))\n            key = _label_points(f, ax, pts, names, s=50, fontsize=7.5)\n            for i in range(0, len(key), 4):\n                f.text(0.01, 0.035 * rows - 0.035 * (i // 4 + 1) + 0.008, (\"Numbered points:  \" if i == 0 else \"\") +\n                       \";   \".join(key[i:i + 4]), fontsize=7.5, ha=\"left\", va=\"bottom\")\n        plt.show()\n    print(f\"{'most common':<30}{'most reliable':<30}\")\n    print(\"-\" * 60)\n    for i in range(min(6, len(tab))):\n        print(f\"{by_freq[i]['name'][:28]:<30}{by_ok[i]['name'][:28]:<30}\")\n    return dict(n=len(tab),\n                most_common=by_freq[0][\"name\"], most_reliable=by_ok[0][\"name\"],\n                same=by_freq[0][\"name\"] == by_ok[0][\"name\"])\n", "sims.py", "exec"), _sm.__dict__)
sys.modules["sims"] = _sm
sims = _sm

ITEMS = [
    Item("P1", "What distinguishes de novo design from virtual screening?",
         {"A": "Structures are restricted to compounds already on the market", "B": "Structures are retrieved from a database rather than generated", "C": "Structures are tested experimentally before any scoring is done", "D": "Structures are generated to fit the site rather than retrieved"},
         "D", {"B": "That describes virtual screening, which is the method being contrasted.", "C": "Scoring comes first in both approaches; testing follows afterwards.", "A": "De novo design is not restricted to known compounds in any way.", "_correct": "De novo design builds new structures from the shape and chemistry of the site, whereas screening can only find what already exists in a database."},
         bloom="Understand", silo="", section="precheck"),
    Item("P2", "Why should a first-generation de novo ligand not fill the binding site completely?",
         {"A": "Because a large ligand cannot be scored by any docking function", "B": "Because site coordinates carry error and optimization needs room", "C": "Because filling the site would raise the molecular weight too far", "D": "Because complete filling always produces an insoluble compound"},
         "B", {"C": "Molecular weight matters later, but it is not the primary reason here.", "D": "Solubility is a real concern but does not follow from filling a site.", "A": "Large ligands are scored routinely; size is not a scoring obstacle.", "_correct": "The crystal coordinates are approximate, so a ligand that already fills the cavity leaves no room to adjust when the model proves slightly wrong."},
         bloom="Understand", silo="", section="precheck"),
    Item("P3", "A designed structure scores superbly but a chemist says it cannot be made. What does that establish?",
         {"A": "That the scoring function must have been applied incorrectly", "B": "That the binding site coordinates were of insufficient quality", "C": "That a design score and synthetic feasibility are independent", "D": "That the compound will bind but will not reach its target site"},
         "C", {"A": "The score may be perfectly correct; it simply does not assess synthesis.", "D": "Nothing about distribution follows from a synthetic objection.", "B": "Site quality affects the score, not whether a chemist can make the molecule.", "_correct": "Scoring functions assess interactions with the site and contain no information about whether a synthetic route exists."},
         bloom="Understand", silo="", section="precheck"),
    Item("C1", "At a 1.6 angstrom probe your output reports about 31 cubic angstrom of free space; at 1.4 angstrom about 67; at 1.2 angstrom about 118. Refining the grid from 1.0 to 0.5 angstrom changes each value by only a few per cent. What do the two patterns together establish?",
         {"A": "Unreliable, since two separate choices change it and neither is justified", "B": "Converged on the grid but probe-dependent, so the probe must be reported", "C": "Grid spacing dominates the error and ought to be made finer still", "D": "The probe has no physical meaning, so the smallest value should be used"},
         "B", {"C": "Changing the grid by a factor of two altered the answer by a few per cent, which is the definition of converged.", "D": "The probe represents the size of the atom or group you intend to place, so it is chosen to match the design.", "A": "Grid convergence shows the calculation is sound; probe dependence is a real physical effect, not an error.", "_correct": "A cavity volume is only defined relative to a probe, because whether a gap counts as empty depends on what you intend to put in it. Grid convergence is a numerical check you must pass; probe dependence is chemistry you must report."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("C2", "Measured with a 1.4 angstrom probe, the free space beside tacrine comes to roughly 67 cubic angstrom, which is on the order of a few heavy atoms. Historically, potent acetylcholinesterase inhibitors were obtained by linking two binding units with a long chain. What does the measured volume suggest about why?",
         {"A": "The volume allows a large extension, so the linked design was arbitrary", "B": "Too little room beside the ligand, so designers reached to a second site", "C": "Linking two units lowers the volume needed, which is why it succeeds", "D": "The measurement is irrelevant, since affinity ignores molecular shape"},
         "B", {"C": "A linked bivalent compound is larger, not smaller, and occupies additional space outside the gorge.", "D": "Shape complementarity is a primary determinant of affinity and is what this measurement probes.", "A": "Sixty-seven cubic angstrom is roughly four to five heavy atoms; a substantial extension needs far more.", "_correct": "The gorge around tacrine is nearly filled, so growing the ligand in place offers little. The successful strategy was to extend along the gorge to the peripheral anionic site, which is why bivalent inhibitors carry long linkers. The measurement explains the design decision."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("C3", "Your output warns that a first design should not fill all the free volume. Why would filling it completely be a poor starting point?",
         {"A": "A fully filled design leaves no tolerance for error in pose or model", "B": "A filled design is too heavy for oral availability whatever its shape", "C": "Filling the volume is impossible, so the attempt wastes computing time", "D": "Filled designs bind too tightly, and excess affinity is undesirable"},
         "A", {"B": "Molecular weight matters, but 67 cubic angstrom of extension is a small addition; the pose risk is the immediate problem.", "C": "Programs will readily generate a space-filling design; the problem is that it is brittle, not unreachable.", "D": "Affinity is rarely the limiting problem at this stage, and a space-filling design is more likely to lose affinity than to gain too much.", "_correct": "The free volume was computed from one pose of one structure with one probe. A design that fits it exactly fails if any of those is slightly wrong. Leaving tolerance means the compound survives the inevitable differences between the model and the real complex."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("Q1", "What distinguishes de novo design from virtual screening?",
         {"A": "Candidate structures are restricted to compounds already in catalogues", "B": "Candidate structures are ranked by measured affinity rather than score", "C": "Candidate structures are generated from the site rather than retrieved", "D": "Candidate structures are built without reference to the target protein"},
         "C", {"B": "confuses computed scores with assay data", "D": "removes the defining input", "A": "describes screening", "_correct": "De novo design builds new structures to fit the site, whereas virtual screening only retrieves compounds that already exist in a database."},
         bloom="Understand", silo="6", section="post"),
    Item("Q2", "Why is a space-filling first-generation de novo ligand usually a poor choice?",
         {"A": "Site coordinates carry error and no room is left to optimize", "B": "Larger ligands are always metabolized faster than smaller ones", "C": "Programs cannot build structures above about thirty heavy atoms", "D": "Full occupancy prevents the ligand from forming hydrogen bonds"},
         "A", {"C": "is a false software limit", "D": "has no mechanistic basis", "B": "is an overgeneralized metabolic claim", "_correct": "Site coordinates carry experimental error, and a ligand that already fills the cavity leaves no room to adjust or optimize it."},
         bloom="Evaluate", silo="6", section="post"),
    Item("Q3", "How does LUDI assemble a candidate structure?",
         {"A": "It adds single atoms along a grid, adjusting each for local fit", "B": "It joins amino acid residues until the pocket is filled in", "C": "It mutates and recombines torsion strings over many generations", "D": "It fits library fragments to interaction sites, then bridges them"},
         "D", {"A": "describes LEGEND", "B": "describes GROW", "C": "describes a genetic algorithm", "_correct": "LUDI places library fragments at identified interaction sites, then searches for bridging groups to join them into one molecule."},
         bloom="Understand", silo="6", section="post"),
    Item("Q4", "In SPROUT, a fragment template is specified by",
         {"A": "three interaction points forming a pharmacophoric triangle", "B": "a ranked list of commercially purchasable starting materials", "C": "vertices for hybridization states and edges for bond orders", "D": "partial atomic charges assigned to every atom in the fragment"},
         "C", {"B": "describes SYNOPSIS", "A": "describes FlexX docking", "D": "is irrelevant to templates", "_correct": "A SPROUT template is a graph whose vertices carry hybridization states and whose edges carry bond orders."},
         bloom="Remember", silo="6", section="post"),
    Item("Q5", "What does the CAESA program contribute to a de novo workflow?",
         {"A": "It estimates how accessible a designed structure is synthetically", "B": "It calculates the desolvation penalty of the designed ligand", "C": "It generates conformations of each designed structure for docking", "D": "It aligns designed structures onto a reference pharmacophore"},
         "A", {"B": "Desolvation penalties are handled by docking scoring functions, not by this program", "C": "Conformation generation is a separate step that this program does not carry out", "D": "Pharmacophore alignment is a different task from estimating synthetic accessibility", "_correct": "CAESA judges synthetic accessibility, flagging designs that would be impractical for a chemist to make."},
         bloom="Remember", silo="6", section="post"),
]

gb = Gradebook("L27", "De Novo Drug Design")
gb.add_many(ITEMS)
print("Ready - 11 questions loaded. Scroll down and begin.")

# ===== BEGIN structure-drawing and calculator kit (lecture notes) =====
# ---------------------------------------------------------------------------
# PHS 511 notebook helper library (pure Python + matplotlib; no RDKit, no network)
# Draws textbook-oriented 2D skeletal structures from explicit coordinates so
# every figure has a fixed, reviewed orientation:
#   * acyl chains: C1 (carbonyl) on the LEFT, omega (methyl) end on the RIGHT
#   * glycerol / sphingosine backbones vertical (sn-1 top, sn-3 bottom)
#   * steroids: rings A-B-C-D left to right, C3-OH lower left, C18/C19 up
# ---------------------------------------------------------------------------
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch, FancyBboxPatch, Circle

COL = {
    "ink": "#1f2430", "muted": "#6b7280",
    "ester": "#1d6fb8", "amide": "#c2410c", "ether": "#7c3aed",
    "phos": "#15803d", "oh": "#b91c1c", "dbl": "#d97706",
    "delta": "#1d6fb8", "omega": "#b91c1c", "drug": "#be123c",
    "polar": "#0e7490", "nonpolar": "#a16207", "hl": "#fde68a",
}
S30, C30 = 0.5, math.cos(math.radians(30))
UNIT_IN = 0.36   # inches per bond length -> identical scale in every figure


class Mol:
    """Atoms with explicit 2D coordinates; bonds with order/style/highlight."""

    def __init__(self, name=""):
        self.name, self.atoms, self.bonds, self.notes, self.boxes = name, [], [], [], []

    def add(self, x, y, label="", color=None, ha="center", size=None, anchor=None):
        """anchor: index of the label character that sits on the atom (default: first C/N/O/P/S)."""
        self.atoms.append(dict(x=x, y=y, label=label, color=color, ha=ha, size=size, anchor=anchor))
        return len(self.atoms) - 1

    def bond(self, i, j, order=1, style="plain", hl=None, color=None, side=None):
        self.bonds.append(dict(i=i, j=j, order=order, style=style, hl=hl, color=color, side=side))
        return len(self.bonds) - 1

    def note(self, x, y, text, color=None, size=8.5, ha="center", va="center", weight="normal"):
        self.notes.append(dict(x=x, y=y, text=text, color=color, size=size, ha=ha, va=va, weight=weight))

    def pos(self, i):
        a = self.atoms[i]
        return a["x"], a["y"]



def _turtle(m, start, n, d0, turns, orders, hl=None):
    idx = [start]
    x, y = m.pos(start)
    d = d0
    for k in range(1, n + 1):
        x, y = x + math.cos(math.radians(d)), y + math.sin(math.radians(d))
        idx.append(m.add(x, y))
        m.bond(idx[-2], idx[-1], order=orders.get(k, 1), hl=hl)
        if k < n:
            d += turns[k]
    return idx


def chain(m, start, n, heading=0.0, first=+1, cis=(), trans=(), hl=None, flip=()):
    """Zigzag chain of n carbons grown from atom `start` (start = carbon 0).
    Bond k joins chain carbons k-1 and k. The turn at carbon k normally
    alternates sign (zigzag). If bond k is a cis C=C, the turn at carbon k
    repeats the previous sign so both substituents sit on the same side.
    `flip` lists carbons where a single bond is drawn rotated (same-sign turn)
    to steer a long chain back into its row; it does not change the molecule."""
    turns, orders = {}, {}
    sign = -first
    for k in range(1, n):
        if k >= 2:
            sign = sign if (k in cis or k in flip) else -sign
        turns[k] = 60 * sign
    for k in list(cis) + list(trans):
        orders[k] = 2
    return _turtle(m, start, n, heading + 30 * first, turns, orders, hl)


# ---------------------------------------------------------------------------
# Rendering
# ---------------------------------------------------------------------------
SUBS = set("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089\u207a\u207b\u00b9\u00b2\u00b3\u2074")
EM = 1.0 / (72 * UNIT_IN)   # data units per point at the fixed drawing scale


def _cw(ch):
    """Approximate glyph advance (em) for DejaVu Sans."""
    if ch in SUBS:
        return 0.40
    if ch in "()":
        return 0.39
    if ch.isupper():
        return 0.72
    if ch.isdigit():
        return 0.64
    return 0.56


def _label_box(lab, fs, anchor=None):
    """(left, right, half-height) of a label in data units, measured from the
    anchor glyph centre (first C/N/O/P/S), which sits exactly on the atom."""
    em = fs * EM
    ws = [_cw(c) * em for c in lab]
    k = anchor if anchor is not None else next((i for i, c in enumerate(lab) if c in "CNOPS"), 0)
    left = sum(ws[:k]) + ws[k] / 2
    right = sum(ws[k + 1:]) + ws[k] / 2
    return left, right, 0.36 * fs * EM * 1.0


def _label_radius(lab, ux, uy, fs=10.5, anchor=None):
    """Distance from atom centre to the label box edge along (ux, uy), plus a gap."""
    if not lab:
        return 0.0
    left, right, hh = _label_box(lab, fs, anchor)
    tx = (right if ux > 0 else left) / abs(ux) if abs(ux) > 1e-9 else 1e9
    ty = hh / abs(uy) if abs(uy) > 1e-9 else 1e9
    return min(tx, ty) + 0.08


def draw(ax, m, fs=10.5):
    A = m.atoms
    nb = {i: [] for i in range(len(A))}
    for b in m.bonds:
        nb[b["i"]].append(b["j"]); nb[b["j"]].append(b["i"])
    # highlights first (under everything)
    for b in m.bonds:
        if b["hl"]:
            (x1, y1), (x2, y2) = m.pos(b["i"]), m.pos(b["j"])
            ax.plot([x1, x2], [y1, y2], color=b["hl"], lw=9, alpha=0.30,
                    solid_capstyle="round", zorder=1)
    for b in m.bonds:
        i, j = b["i"], b["j"]
        (x1, y1), (x2, y2) = m.pos(i), m.pos(j)
        L = math.hypot(x2 - x1, y2 - y1)
        ux, uy = (x2 - x1) / L, (y2 - y1) / L
        r1 = _label_radius(A[i]["label"], ux, uy, A[i]["size"] or fs, A[i]["anchor"])
        r2 = _label_radius(A[j]["label"], ux, uy, A[j]["size"] or fs, A[j]["anchor"])
        ax1, ay1 = x1 + ux * r1, y1 + uy * r1
        ax2, ay2 = x2 - ux * r2, y2 - uy * r2
        c = b["color"] or COL["ink"]
        st = b["style"]
        if st == "wedge":
            px, py = -uy * 0.11, ux * 0.11
            ax.add_patch(Polygon([(ax1, ay1), (ax2 + px, ay2 + py), (ax2 - px, ay2 - py)],
                                 closed=True, fc=c, ec=c, lw=0.6, zorder=3))
            continue
        if st == "hash":
            nseg = 7
            for s in range(nseg + 1):
                f = s / nseg
                cx, cy = ax1 + (ax2 - ax1) * f, ay1 + (ay2 - ay1) * f
                w = 0.02 + 0.10 * f
                ax.plot([cx - uy * w, cx + uy * w], [cy + ux * w, cy - ux * w],
                        color=c, lw=1.1, zorder=3, solid_capstyle="butt")
            continue
        if st == "wavy":
            n = 40
            xs = [ax1 + (ax2 - ax1) * k / n - uy * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ys = [ay1 + (ay2 - ay1) * k / n + ux * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ax.plot(xs, ys, color=c, lw=1.4, zorder=3)
            continue
        px, py = -uy, ux
        if b["order"] == 1:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round",
                    ls="--" if st == "dashed" else "-")
            continue
        side = b["side"]
        if side is None:
            if A[i]["label"] or A[j]["label"]:
                side = 0                      # C=O, C=N+: centred pair
            else:
                sx = sy = 0.0
                for k in nb[i]:
                    if k != j:
                        sx += A[k]["x"] - x1; sy += A[k]["y"] - y1
                for k in nb[j]:
                    if k != i:
                        sx += A[k]["x"] - x2; sy += A[k]["y"] - y2
                cross = ux * sy - uy * sx
                side = 1 if cross >= -1e-6 else -1
        if side == 0:
            for sgn in (+1, -1):
                o = 0.075 * sgn
                ax.plot([ax1 + px * o, ax2 + px * o], [ay1 + py * o, ay2 + py * o],
                        color=c, lw=1.5, zorder=3, solid_capstyle="round")
        else:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round")
            o, sh = 0.15 * side, 0.16
            ax.plot([ax1 + px * o + ux * sh, ax2 + px * o - ux * sh],
                    [ay1 + py * o + uy * sh, ay2 + py * o - uy * sh],
                    color=c, lw=1.5, zorder=3, solid_capstyle="round")
    for a in A:
        if a["label"]:
            l_, r_, _ = _label_box(a["label"], a["size"] or fs, a["anchor"])
            ax.text(a["x"] + (r_ - l_) / 2, a["y"], a["label"], ha="center", va="center",
                    fontsize=a["size"] or fs, color=a["color"] or COL["ink"],
                    zorder=5, fontweight="normal")
    for n_ in m.notes:
        ax.text(n_["x"], n_["y"], n_["text"], ha=n_["ha"], va=n_["va"], fontsize=n_["size"],
                color=n_["color"] or COL["muted"], zorder=6, fontweight=n_["weight"])


def extent(m, pad=0.7):
    xs = [a["x"] for a in m.atoms] + [n["x"] for n in m.notes]
    ys = [a["y"] for a in m.atoms] + [n["y"] for n in m.notes]
    return min(xs) - pad, max(xs) + pad, min(ys) - pad, max(ys) + pad




def show(m, title=None, unit=UNIT_IN, extra=None, caption=None, show_now=True):
    """One structure at the fixed scale (UNIT_IN inches per bond)."""
    x0, x1, y0, y1 = extent(m)
    if extra and hasattr(extra, "bounds"):
        bx0, bx1, by0, by1 = extra.bounds
        x0, x1, y0, y1 = min(x0, bx0), max(x1, bx1), min(y0, by0), max(y1, by1)
    th = 0.42 if title else 0.0
    ch = 0.40 if caption else 0.0
    W, H = (x1 - x0) * unit, (y1 - y0) * unit
    fig = plt.figure(figsize=(W, H + th + ch))
    ax = fig.add_axes([0, ch / (H + th + ch), 1, H / (H + th + ch)])
    draw(ax, m)
    if extra:
        extra(ax)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        fig.text(0.0, 1 - (th / 2) / (H + th + ch), title, fontsize=11.5, fontweight="bold",
                 color=COL["ink"], va="center", ha="left")
    if caption:
        fig.text(0.0, (ch / 2) / (H + th + ch), caption, fontsize=9, color=COL["muted"],
                 va="center", ha="left")
    if show_now:
        plt.show()
    return fig


def grid(mols, titles, ncols=2, unit=UNIT_IN, align="left", suptitle=None, captions=None, show_now=True, bottom=0.0):
    """Panels drawn at one common scale; each panel box has the same width and
    the row height of its tallest member, so structures line up cleanly."""
    exts = [extent(m) for m in mols]
    W = max(max(e[1] - e[0] for e in exts), 3.6 / unit)   # panels at least 3.6 in wide
    nrows = math.ceil(len(mols) / ncols)
    import textwrap as _tw
    maxch = max(12, int(W * unit * 72 / (10.5 * 0.60)) - 1)
    wrapped = [_tw.fill(t, maxch) for t in titles]
    rowH = []
    for r in range(nrows):
        rowH.append(max(exts[k][3] - exts[k][2] for k in range(r * ncols, min(len(mols), (r + 1) * ncols))))
    row_th = []
    for r in range(nrows):
        nl = max(wrapped[k].count("\n") + 1 for k in range(r * ncols, min(len(mols), (r + 1) * ncols)))
        row_th.append(0.27 + 0.19 * nl)
    fig_w = W * unit * ncols
    fig_h = sum(h * unit + th for h, th in zip(rowH, row_th)) + (0.5 if suptitle else 0) + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    top = 1.0 - (0.5 / fig_h if suptitle else 0)
    y_cursor = top
    for r in range(nrows):
        h_frac = (rowH[r] * unit) / fig_h
        t_frac = row_th[r] / fig_h
        y_cursor -= (h_frac + t_frac)
        for c in range(ncols):
            k = r * ncols + c
            if k >= len(mols):
                break
            ax = fig.add_axes([c / ncols, y_cursor, 1 / ncols, h_frac])
            m = mols[k]
            x0, x1, y0, y1 = exts[k]
            draw(ax, m)
            if align == "left":
                ax.set_xlim(x0, x0 + W)
            else:
                cx = (x0 + x1) / 2
                ax.set_xlim(cx - W / 2, cx + W / 2)
            cy = (y0 + y1) / 2
            ax.set_ylim(cy - rowH[r] / 2, cy + rowH[r] / 2)
            ax.set_aspect("equal"); ax.axis("off")
            ax.text(0.01, 1.0 + 0.02, wrapped[k], transform=ax.transAxes, fontsize=10.5, linespacing=1.15,
                    fontweight="bold", color=COL["ink"], va="bottom", ha="left")
    if suptitle:
        fig.text(0.005, 1 - 0.25 / fig_h, suptitle, fontsize=12.5, fontweight="bold", va="center")
    if show_now:
        plt.show()
    return fig


# ---------------------------------------------------------------------------
# Structure builders (coordinates chosen to match textbook orientation)
# ---------------------------------------------------------------------------
def _outer(m, i, nbrs, dist=0.55):
    """Point on the open (outer) side of atom i, away from its neighbours."""
    x, y = m.pos(i)
    sx = sum(m.atoms[k]["x"] - x for k in nbrs); sy = sum(m.atoms[k]["y"] - y for k in nbrs)
    L = math.hypot(sx, sy) or 1.0
    return x - sx / L * dist, y - sy / L * dist


def fatty_acid(n, cis=(), trans=(), name="", numbers=None, ionized=False, origin=(0.0, 0.0),
               hl_double=True, flip=()):
    """Free fatty acid, C1 (COOH) on the LEFT, methyl (omega) end on the RIGHT.
    cis/trans: Delta positions (bond between C_p and C_p+1).
    numbers: None | 'delta' | 'omega' | 'both'"""
    m = Mol(name)
    ox, oy = origin
    c1 = m.add(ox, oy)
    o_dbl = m.add(ox, oy + 1.0, "O")
    o_h = m.add(ox - C30, oy - 0.5, "⁻O" if ionized else "HO")
    m.bond(c1, o_dbl, 2); m.bond(c1, o_h, 1)
    hl = COL["hl"] if hl_double else None
    ids = chain(m, c1, n - 1, heading=0, first=-1, cis=cis, trans=trans, flip=flip)
    if hl_double:
        for b in m.bonds:
            if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
                b["hl"] = COL["dbl"]
    m.carbons = ids  # carbons[0] = C1
    if numbers:
        dbs = sorted(list(cis) + list(trans))
        last = dbs[-1] if dbs else n - 2
        for p, a in enumerate(ids, start=1):
            if p == 1:
                nb = [o_dbl, o_h, ids[1]]
            elif p == n:
                nb = [ids[p - 2]]
            else:
                nb = [ids[p - 2], ids[p]]
            if numbers in ("delta", "both"):
                x, y = _outer(m, a, nb, 0.50)
                m.note(x, y, str(p), color=COL["delta"], size=7.5)
            w = n - p + 1
            if numbers == "omega" or (numbers == "both" and p >= last):
                if p == n:
                    x, y = _outer(m, a, nb, 0.50)
                    y -= 0.42 if numbers == "both" else 0.0
                else:
                    x, y = _outer(m, a, nb, 0.50 if numbers == "omega" else -0.62)
                m.note(x, y, ("\u03c9" + str(w)) if w == 1 else str(w), color=COL["omega"], size=7.5,
                       weight="bold")
    return m


def acyl_row(m, o_atom, n, cis=(), trans=(), carbonyl_up=True, flip=None):
    """Acyl group C(=O)-chain attached to ester/amide atom o_atom, chain to the right."""
    x, y = m.pos(o_atom)
    c1 = m.add(x + C30, y + (0.5 if carbonyl_up else -0.5))
    m.bond(o_atom, c1, 1)
    od = m.add(x + C30, y + (1.5 if carbonyl_up else -1.5), "O")
    m.bond(c1, od, 2)
    if flip is None:   # keep long chains in their row: rotate the bond 3 carbons past the last cis C=C
        flip = tuple(p + 2 for p in cis)
    ids = chain(m, c1, n - 1, heading=0, first=-1 if carbonyl_up else +1, cis=cis, trans=trans, flip=flip)
    for b in m.bonds:
        if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
            b["hl"] = COL["dbl"]
    return c1, od, ids


ROW = 2.6  # vertical spacing of backbone carbons in E-shaped lipids


def backbone(m, labels, x=0.0, y0=0.0):
    """Vertical 3-carbon backbone (glycerol or sphingosine C1-C3), top->bottom."""
    ids = []
    for k, lab in enumerate(labels):
        ids.append(m.add(x, y0 - k * ROW, lab))
        if k:
            m.bond(ids[k - 1], ids[k], 1)
    return ids


def ester(m, c_backbone, n, cis=(), trans=(), hl=True, cut=None):
    x, y = m.pos(c_backbone)
    o = m.add(x + 1.15, y, "O")
    b0 = m.bond(c_backbone, o, 1, hl=COL["ester"] if hl else None)
    c1, od, ids = acyl_row(m, o, n, cis, trans)
    m.bonds[-(len(ids)) - 1]["hl"] = None
    # highlight O-C(=O) (the ester bond lipases cut)
    for b in m.bonds:
        if b["i"] == o and b["j"] == c1:
            b["hl"] = COL["ester"] if hl else None
            if cut:
                b["color"] = COL["drug"]
    return o, c1, ids


def triacylglycerol(acyls=((16, ()), (18, (9,)), (18, (9, 12))), name="Triacylglycerol",
                    sn=True, cuts=None):
    """E-shaped TAG; glycerol vertical, sn-1 top. acyls: ((n, cis-positions), ...)."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    rows = []
    for k, (n, cis) in enumerate(acyls):
        rows.append(ester(m, g[k], n, cis=cis))
    if sn:
        for k in range(3):
            x, y = m.pos(g[k])
            m.note(x - 0.95, y, "sn-%d" % (k + 1), color=COL["muted"], size=8.5, ha="right")
    m.g, m.rows = g, rows
    return m


def mark_cut(m, o_idx, c_idx, text, color=None, dy=-0.62, dx=0.0):
    x1, y1 = m.pos(o_idx); x2, y2 = m.pos(c_idx)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    m.note(mx, my - 0.05, "✂", color=color or COL["drug"], size=13)
    m.note(mx + dx, my + dy, text, color=color or COL["drug"], size=7.5, ha="left" if dx else "center")


def phosphocholine(m, c_attach, head="choline"):
    """-O-P(=O)(O-)-O-CH2-CH2-N+(CH3)3 to the right of c_attach (horizontal)."""
    x, y = m.pos(c_attach)
    o1 = m.add(x + 1.15, y, "O"); m.bond(c_attach, o1, 1)
    p = m.add(x + 2.3, y, "P", color=COL["phos"]); m.bond(o1, p, 1, hl=COL["phos"])
    po = m.add(x + 2.3, y + 1.05, "O"); m.bond(p, po, 2)
    pm = m.add(x + 2.3, y - 1.05, "O⁻"); m.bond(p, pm, 1)
    o2 = m.add(x + 3.45, y, "O"); m.bond(p, o2, 1, hl=COL["phos"])
    if head == "choline":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "N⁺(CH₃)₃"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.6, y - 1.0, "choline", color=COL["muted"], size=8)
    elif head == "ethanolamine":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "NH₃⁺"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.4, y - 1.0, "ethanolamine", color=COL["muted"], size=8)
    return p


def phosphatidylcholine(name="Phosphatidylcholine (lecithin)", a1=(16, ()), a2=(18, (9,))):
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    ester(m, g[0], *a1); ester(m, g[1], *a2)
    phosphocholine(m, g[2], "choline")
    for k in range(3):
        x, y = m.pos(g[k]); m.note(x - 0.95, y, "sn-%d" % (k + 1), size=8.5, ha="right")
    m.g = g
    return m


def plasmalogen(name="Plasmalogen (ethanolamine)", n1=16, a2=(18, (9,))):
    """sn-1 vinyl ether: CH2-O-CH=CH-R (cis), no carbonyl."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    x, y = m.pos(g[0])
    o = m.add(x + 1.15, y, "O"); m.bond(g[0], o, 1, hl=COL["ether"])
    ids = chain(m, o, n1, heading=0, first=+1, cis=(2,), flip=(4,))
    # bond O-C1' is ether; C1'=C2' vinyl (cis, as in natural plasmalogens)
    for b in m.bonds:
        if b["i"] == o and b["j"] == ids[1]:
            b["hl"] = COL["ether"]
        if b["order"] == 2:
            b["hl"] = COL["ether"]
    ester(m, g[1], *a2)
    phosphocholine(m, g[2], "ethanolamine")
    for k in range(3):
        xx, yy = m.pos(g[k]); m.note(xx - 0.95, yy, "sn-%d" % (k + 1), size=8.5, ha="right")
    return m


def sphingo_core(m, fa_n=18, fa_cis=()):
    """Sphingosine drawn as the same vertical 3-carbon backbone:
    C3 top (CH-OH + trans CH=CH tail), C2 middle (CH-NH-acyl, amide), C1 bottom (CH2-O-head)."""
    s = backbone(m, ["CH", "CH", "CH₂"])
    x, y = m.pos(s[0])
    oh = m.add(x - 1.15, y, "HO"); m.bond(s[0], oh, 1, hl=COL["oh"])
    ids = chain(m, s[0], 15, heading=0, first=+1, trans=(1,))
    # C4=C5 trans: chain bond 1 is C3-C4? -> rebuild explicitly
    m.bonds[[k for k, b in enumerate(m.bonds) if b["i"] == s[0] and b["j"] == ids[1]][0]]["order"] = 1
    for b in m.bonds:
        if b["i"] == ids[1] and b["j"] == ids[2]:
            b["order"] = 2; b["hl"] = COL["dbl"]
    # amide at C2
    x2, y2 = m.pos(s[1])
    nh = m.add(x2 + 1.4, y2, "NH"); m.bond(s[1], nh, 1)
    c1, od, acyl = acyl_row(m, nh, fa_n, cis=fa_cis)
    for b in m.bonds:
        if b["i"] == nh and b["j"] == c1:
            b["hl"] = COL["amide"]
    for k, lab in enumerate(["C3", "C2", "C1"]):
        xx, yy = m.pos(s[k]); m.note(xx - 0.25, yy - 0.55, lab, size=7.5)
    return s


def sphingomyelin(name="Sphingomyelin"):
    m = Mol(name)
    s = sphingo_core(m, 18)
    phosphocholine(m, s[2], "choline")
    return m


def galactosylceramide(name="Galactosylceramide (a cerebroside)"):
    m = Mol(name)
    s = sphingo_core(m, 24, fa_cis=(15,))
    x, y = m.pos(s[2])
    o = m.add(x + 1.15, y, "O"); m.bond(s[2], o, 1, hl=COL["polar"])
    # beta-D-galactopyranose, Haworth-free chair-less hexagon (ring O upper right)
    cx, cy, r = x + 3.0, y - 0.1, 1.0
    ang = [180, 120, 60, 0, -60, -120]
    pts = [m.add(cx + r * math.cos(math.radians(a)), cy + r * math.sin(math.radians(a)),
                 "O" if k == 1 else "") for k, a in enumerate(ang)]
    for k in range(6):
        m.bond(pts[k], pts[(k + 1) % 6])
    m.bond(o, pts[0], 1, hl=COL["polar"])
    m.note(cx, cy - 0.05, "Gal", color=COL["polar"], size=9, weight="bold")
    return m


# --- steroids --------------------------------------------------------------
def _hex(cx, cy):
    return [(cx + math.cos(math.radians(a)), cy + math.sin(math.radians(a))) for a in (90, 30, -30, -90, -150, 150)]


def steroid_core(m, ox=0.0, oy=0.0, dbl=()):
    """Gonane nucleus in standard orientation. Returns dict C1..C17 -> atom index.
    Unit hexagon geometry with vertical shared A/B edge (C5-C10)."""
    r3 = math.sqrt(3)
    P = {}
    A = _hex(ox, oy)                      # top, ur, lr, bottom, ll, ul
    P.update({1: A[0], 10: A[1], 5: A[2], 4: A[3], 3: A[4], 2: A[5]})
    B = _hex(ox + r3, oy)
    P.update({9: B[0], 8: B[1], 6: B[3], 7: B[2]})
    C = _hex(ox + 1.5 * r3, oy + 1.5)
    P.update({12: C[0], 13: C[1], 14: C[2], 11: C[5]})
    # ring D: regular pentagon on edge C13-C14, outward (to the right)
    (x13, y13), (x14, y14) = P[13], P[14]
    mx, my = (x13 + x14) / 2, (y13 + y14) / 2
    s = 1.0
    apo = s / (2 * math.tan(math.radians(36)))
    cx, cy = mx + apo, my
    R = s / (2 * math.sin(math.radians(36)))
    base = math.degrees(math.atan2(y13 - cy, x13 - cx))
    pent = [(cx + R * math.cos(math.radians(base - 72 * k)), cy + R * math.sin(math.radians(base - 72 * k)))
            for k in range(5)]
    # pent[0]=C13, going clockwise: C17, C16, C15, C14
    P.update({17: pent[1], 16: pent[2], 15: pent[3]})
    idx = {k: m.add(v[0], v[1]) for k, v in P.items()}
    ring_bonds = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 10), (10, 1), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
                  (9, 11), (11, 12), (12, 13), (13, 14), (14, 8), (13, 17), (17, 16), (16, 15), (15, 14)]
    for a, b in ring_bonds:
        m.bond(idx[a], idx[b], 2 if (a, b) in dbl or (b, a) in dbl else 1)
    m.st = idx
    return idx


def subst(m, at, dx, dy, label="", style="plain", order=1, hl=None, color=None):
    x, y = m.pos(at)
    k = m.add(x + dx, y + dy, label, color=color)
    m.bond(at, k, order, style=style, hl=hl)
    return k


def side_chain_cholestane(m, st, extra_ethyl=False):
    """C20-C27 side chain on C17 (drawn up-right, zigzag)."""
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 4, heading=0, first=-1)
    c25 = ids[-1]
    x25, y25 = m.pos(c25)
    c26 = m.add(x25 + C30, y25 + 0.5); m.bond(c25, c26)
    c27 = m.add(x25, y25 - 1.0); m.bond(c25, c27)
    if extra_ethyl:
        x24, y24 = m.pos(ids[3])
        ex, ey = _outer(m, ids[3], [ids[2], ids[4]], 1.0)
        e1 = m.add(ex, ey); m.bond(ids[3], e1, hl=COL["nonpolar"])
        e2 = m.add(ex + C30, ey + (0.5 if ey > y24 else -0.5)); m.bond(e1, e2, hl=COL["nonpolar"])
    return ids


def cholesterol(name="Cholesterol", sitosterol=False):
    m = Mol(name)
    st = steroid_core(m, dbl=((5, 6),))
    ho = subst(m, st[3], -C30, -0.5, "HO", style="wedge", hl=COL["oh"])
    subst(m, st[10], 0.0, 0.95, "", style="wedge")     # C19 methyl
    subst(m, st[13], 0.0, 0.95, "", style="wedge")     # C18 methyl
    side_chain_cholestane(m, st, extra_ethyl=sitosterol)
    m.note(m.pos(st[3])[0] - 0.2, m.pos(st[3])[1] + 0.45, "3β", size=7.5)
    for k, lab in ((1, "A"), (2, "B"), (3, "C")):
        pass
    r3 = math.sqrt(3)
    for (cx, cy, lab) in ((0, 0, "A"), (r3, 0, "B"), (1.5 * r3, 1.5, "C")):
        m.note(cx, cy, lab, color="#9aa3b2", size=9, weight="bold")
    x13, y13 = m.pos(st[13]); x14, y14 = m.pos(st[14])
    m.note((x13 + x14) / 2 + 0.68, (y13 + y14) / 2, "D", color="#9aa3b2", size=9, weight="bold")
    m.st = st
    return m


def bile_acid(name="Glycocholate", oh=(("3", "a"), ("7", "a"), ("12", "a")), conj="glycine", ionized=True):
    """5-beta bile acid; oh entries (position, 'a' alpha=hash | 'b' beta=wedge)."""
    m = Mol(name)
    st = steroid_core(m)
    subst(m, st[10], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[13], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[5], 0.0, -0.85, "H", style="wedge")      # 5-beta H (A/B cis)
    vec = {"3": (-C30, -0.5), "7": (C30 * 0.0, -1.0), "12": (0.0, 1.0)}
    for pos, face in oh:
        dx, dy = vec[pos]
        if pos == "12":
            dx, dy = (-0.35, 0.9)
        if pos == "7":
            dx, dy = (0.35, -0.95)
        subst(m, st[int(pos)], dx, dy, "HO" if pos == "3" else "OH",
              style="hash" if face == "a" else "wedge", hl=COL["polar"])
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 3, heading=0, first=-1)
    c24 = ids[-1]
    x24, y24 = m.pos(c24)
    o24 = m.add(x24, y24 + 1.0, "O"); m.bond(c24, o24, 2)
    if conj == "glycine":
        nh = m.add(x24 + C30, y24 - 0.5, "NH"); m.bond(c24, nh, 1, hl=COL["amide"])
        ca = m.add(x24 + 2 * C30, y24); m.bond(nh, ca)
        cc = m.add(x24 + 3 * C30, y24 - 0.5); m.bond(ca, cc)
        m.bond(cc, m.add(x24 + 3 * C30, y24 - 1.5, "O"), 2)
        m.bond(cc, m.add(x24 + 4 * C30, y24, "O⁻" if ionized else "OH"), 1, hl=COL["polar"])
    else:
        m.bond(c24, m.add(x24 + C30, y24 - 0.5, "OH"), 1, hl=COL["polar"])
    m.st = st
    return m


# ---------------------------------------------------------------------------
# General 2D sketch helpers: rings, fused rings, substituents, reaction schemes
# (built on draw.py; every coordinate is explicit so orientation is fixed)
# ---------------------------------------------------------------------------
def _nbrs(m, i):
    out = []
    for b in m.bonds:
        if b["i"] == i:
            out.append(b["j"])
        elif b["j"] == i:
            out.append(b["i"])
    return out


def bond_between(m, i, j):
    for b in m.bonds:
        if (b["i"], b["j"]) in ((i, j), (j, i)):
            return b
    return None


def ring(m, cx, cy, n=6, rot=90.0, labels=None, side=1.0):
    """Regular n-ring with bond length `side`, first vertex at angle `rot` (deg),
    vertices counter-clockwise. labels: list of n labels ('' = carbon).
    Returns atom indices in ring order."""
    R = side / (2 * math.sin(math.pi / n))
    ids = []
    for k in range(n):
        a = math.radians(rot + 360.0 * k / n)
        ids.append(m.add(cx + R * math.cos(a), cy + R * math.sin(a), (labels[k] if labels else "")))
    for k in range(n):
        m.bond(ids[k], ids[(k + 1) % n], 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def fuse(m, a, b, n=6, labels=None, side=1.0):
    """New regular n-ring sharing edge a-b, built on the side away from the
    existing neighbours. labels: for the n-2 NEW atoms, in order from b onward.
    Returns ring atoms [a, b, new...]."""
    (xa, ya), (xb, yb) = m.pos(a), m.pos(b)
    mx, my = (xa + xb) / 2, (ya + yb) / 2
    ex, ey = xb - xa, yb - ya
    L = math.hypot(ex, ey)
    px, py = -ey / L, ex / L
    # decide which side is free: away from centroid of current neighbours
    nb = [k for k in _nbrs(m, a) + _nbrs(m, b) if k not in (a, b)]
    if nb:
        cx_ = sum(m.atoms[k]["x"] for k in nb) / len(nb); cy_ = sum(m.atoms[k]["y"] for k in nb) / len(nb)
        if (cx_ - mx) * px + (cy_ - my) * py > 0:
            px, py = -px, -py
    apo = side / (2 * math.tan(math.pi / n))
    cx, cy = mx + px * apo, my + py * apo
    R = side / (2 * math.sin(math.pi / n))
    # walk from b around the new ring (away from a)
    ang_b = math.atan2(yb - cy, xb - cx)
    ang_a = math.atan2(ya - cy, xa - cx)
    step = 2 * math.pi / n
    # direction: going from b, the next vertex must not be a
    d = (ang_b - ang_a + math.pi) % (2 * math.pi) - math.pi
    sgn = 1 if d > 0 else -1
    ids = [a, b]
    prev = b
    for k in range(1, n - 1):
        ang = ang_b + sgn * step * k
        lab = labels[k - 1] if labels else ""
        i = m.add(cx + R * math.cos(ang), cy + R * math.sin(ang), lab)
        m.bond(prev, i, 1)
        ids.append(i); prev = i
    m.bond(prev, a, 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def set_double(m, pairs):
    """pairs: iterable of (i, j) atom-index pairs to make double bonds."""
    for i, j in pairs:
        b = bond_between(m, i, j)
        if b is None:
            raise ValueError("no bond %d-%d" % (i, j))
        b["order"] = 2


def kekule(m, ids, start=0):
    """Alternate double bonds around ring `ids` beginning with bond start->start+1."""
    n = len(ids)
    for k in range(start, start + n, 2):
        bond_between(m, ids[k % n], ids[(k + 1) % n])["order"] = 2


def outward(m, i, dist=1.0):
    """Unit outward direction from atom i (away from its neighbours)."""
    x, y = m.pos(i)
    nb = _nbrs(m, i)
    sx = sum(m.atoms[k]["x"] - x for k in nb); sy = sum(m.atoms[k]["y"] - y for k in nb)
    L = math.hypot(sx, sy)
    if L < 1e-9:
        return (dist, 0.0)
    return (-sx / L * dist, -sy / L * dist)


def sub(m, i, label="", angle=None, dist=1.0, order=1, style="plain", hl=None, color=None, anchor=None):
    """Attach a substituent to atom i: outward by default, or at an absolute angle (deg)."""
    x, y = m.pos(i)
    if angle is None:
        dx, dy = outward(m, i, dist)
    else:
        dx, dy = dist * math.cos(math.radians(angle)), dist * math.sin(math.radians(angle))
    k = m.add(x + dx, y + dy, label, color=color, anchor=anchor)
    m.bond(i, k, order, style=style, hl=hl)
    return k


def grow(m, i, n, angle=None, first=+1, cis=(), trans=(), flip=()):
    """Zigzag chain of n atoms from atom i; heading = outward unless angle given."""
    if angle is None:
        dx, dy = outward(m, i)
        angle = math.degrees(math.atan2(dy, dx))
    return chain(m, i, n, heading=angle, first=first, cis=cis, trans=trans, flip=flip)


def label(m, i, text, anchor=None):
    m.atoms[i]["label"] = text
    m.atoms[i]["anchor"] = anchor


def hl_bonds(m, pairs, color):
    for i, j in pairs:
        bond_between(m, i, j)["hl"] = color


def hl_atoms_note(m, i, text, color=None, dist=0.75, size=8):
    x, y = m.pos(i)
    dx, dy = outward(m, i, dist)
    m.note(x + dx, y + dy, text, color=color or COL["muted"], size=size)


def shift(m, dx, dy):
    for a in m.atoms:
        a["x"] += dx; a["y"] += dy
    for n in m.notes:
        n["x"] += dx; n["y"] += dy
    return m


def rotate(m, deg, about=None):
    """Rotate a whole drawing (use to fix an orientation, e.g. to put a ring flat)."""
    if about is None:
        xs = [a["x"] for a in m.atoms]; ys = [a["y"] for a in m.atoms]
        about = ((min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2)
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    for obj in m.atoms + m.notes:
        x, y = obj["x"] - about[0], obj["y"] - about[1]
        obj["x"], obj["y"] = about[0] + c * x - s * y, about[1] + s * x + c * y
    return m


def mirror(m, axis="x"):
    """Mirror left-right (axis='x') or top-bottom ('y'); wedges keep their sense."""
    for obj in m.atoms + m.notes:
        if axis == "x":
            obj["x"] = -obj["x"]
        else:
            obj["y"] = -obj["y"]
    return m


def merge(dst, src, dx=0.0, dy=0.0):
    """Copy all atoms/bonds/notes of src into dst with an offset; returns index map."""
    base = len(dst.atoms)
    for a in src.atoms:
        b = dict(a); b["x"] += dx; b["y"] += dy
        dst.atoms.append(b)
    for b in src.bonds:
        c = dict(b); c["i"] += base; c["j"] += base
        dst.bonds.append(c)
    for n in src.notes:
        c = dict(n); c["x"] += dx; c["y"] += dy
        dst.notes.append(c)
    return base


# ---------------------------------------------------------------------------
# Reaction / pathway scheme: molecules in a row (wrapping), arrows with labels
# ---------------------------------------------------------------------------
def scheme(items, title=None, per_row=3, unit=UNIT_IN, gap=3.2, caption=None, show_now=True):
    """items: [Mol, ("enzyme", "cofactor note"), Mol, (...), Mol ...]
    Molecules are drawn at one scale; arrows carry an enzyme label above and
    an optional cofactor/regulation note below. Rows wrap after per_row molecules."""
    mols = [x for x in items if isinstance(x, Mol)]
    arrows = [x for x in items if not isinstance(x, Mol)]
    exts = [extent(mm, pad=0.4) for mm in mols]
    rows = [list(range(k, min(k + per_row, len(mols)))) for k in range(0, len(mols), per_row)]
    row_h = [max(exts[k][3] - exts[k][2] for k in r) for r in rows]
    row_w = [sum(exts[k][1] - exts[k][0] for k in r) + gap * (len(r) - 1) for r in rows]
    W = max(row_w) + (1.6 if len(rows) > 1 else 0)
    H = sum(row_h) + 1.6 * (len(rows) - 1)
    th = 0.45 if title else 0.0
    ch = 0.4 if caption else 0.0
    fig = plt.figure(figsize=(W * unit, H * unit + th + ch))
    ax = fig.add_axes([0, ch / (H * unit + th + ch), 1, H * unit / (H * unit + th + ch)])
    ax.set_xlim(0, W); ax.set_ylim(-H, 0); ax.set_aspect("equal"); ax.axis("off")
    y_top = 0.0
    pos = {}
    for r, ids in enumerate(rows):
        x = 0.0 if r == 0 else 1.6
        cy = y_top - row_h[r] / 2
        for k in ids:
            x0, x1, y0, y1 = exts[k]
            dx = x - x0; dy = cy - (y0 + y1) / 2
            mm = Mol(); merge(mm, mols[k], dx, dy)
            draw(ax, mm)
            pos[k] = (x, x + (x1 - x0), cy, y1 - y0)
            x += (x1 - x0) + gap
        y_top -= row_h[r] + 1.6
    for k, lab in enumerate(arrows):
        if k + 1 >= len(mols):
            break
        enz, note = (lab, "") if isinstance(lab, str) else (lab[0], lab[1] if len(lab) > 1 else "")
        xa0, xa1, ya, ha = pos[k]; xb0, xb1, yb, hb = pos[k + 1]
        same = abs(ya - yb) < 1e-6
        if same:
            p, q = (xa1 + 0.25, ya), (xb0 - 0.25, yb)
        else:  # wrap: from under the last molecule of a row to above the first of the next
            p, q = ((xa0 + xa1) / 2, ya - ha / 2 + 0.1), ((xb0 + xb1) / 2, yb + hb / 2 - 0.1)
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=14, lw=1.4, color=COL["ink"],
                                     connectionstyle="arc3,rad=0" if same else "arc3,rad=-0.12"))
        mx, my = (p[0] + q[0]) / 2, (p[1] + q[1]) / 2
        box = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9)
        if same:
            ax.text(mx, my + 0.3, enz, ha="center", va="bottom", fontsize=8.5, color=COL["ester"],
                    fontweight="bold", bbox=box)
            if note:
                ax.text(mx, my - 0.3, note, ha="center", va="top", fontsize=8, color=COL["muted"], bbox=box)
        else:
            ax.text(mx, my, enz + ("\n" + note if note else ""), ha="center", va="center", fontsize=8.5,
                    color=COL["ester"], bbox=box)
    if title:
        fig.text(0.0, 1 - (th / 2) / (H * unit + th + ch), title, fontsize=11.5, fontweight="bold", va="center")
    if caption:
        fig.text(0.0, (ch / 2) / (H * unit + th + ch), caption, fontsize=9, color=COL["muted"], va="center")
    if show_now:
        plt.show()
    return fig


try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    def Markdown(s): return s


def _md(lines):
    display(Markdown("\n".join(lines) if isinstance(lines, (list, tuple)) else lines))


# ---------------------------------------------------------------------------
# Calculator engines (no UI). Two families:
#   RPNEngine  - HP-15C / HP 35s RPN behaviour: 4-level stack X Y Z T, LAST X,
#                stack-lift rules, f/g (or left/right) shifted functions.
#   AlgEngine  - algebraic entry line with textbook precedence (TI-30XS,
#                TI-84 Plus CE, Casio fx-991EX/115ES, HP 35s ALG mode).
# Keys are canonical function ids (see FUNCS); the layout tables map each
# physical key + shift state to one of these ids.
# ---------------------------------------------------------------------------
import math

_FACT_MAX = 170


def _fact(x):
    if x < 0 or abs(x - round(x)) > 1e-12:
        return math.gamma(x + 1)                       # HP-15C x! gives Γ(x+1) for non-integers
    if x > _FACT_MAX:
        raise OverflowError("overflow")
    return float(math.factorial(int(round(x))))


def _perm(n, r):
    return math.perm(int(n), int(r))


def _comb(n, r):
    return math.comb(int(n), int(r))


def _hms(h):          # decimal hours -> H.MMSS
    sign = -1 if h < 0 else 1
    h = abs(h)
    H = int(h); m = (h - H) * 60; M = int(m); s = (m - M) * 60
    return sign * (H + M / 100 + s / 10000)


def _hr(hms):         # H.MMSS -> decimal hours
    sign = -1 if hms < 0 else 1
    x = abs(hms)
    H = int(x); MM = int(round((x - H) * 100, 8)); SS = ((x - H) * 100 - MM) * 100
    return sign * (H + MM / 60 + SS / 3600)


class AngleMixin:
    angle = "DEG"

    def to_rad(self, x):
        return {"DEG": math.radians(x), "RAD": x, "GRD": x * math.pi / 200}[self.angle]

    def from_rad(self, r):
        return {"DEG": math.degrees(r), "RAD": r, "GRD": r * 200 / math.pi}[self.angle]


def fmt_number(x, mode="FIX", digits=4, width=10):
    """HP-style display formatting (FIX/SCI/ENG/ALL)."""
    if isinstance(x, str):
        return x
    if x != x:
        return "Error"
    if math.isinf(x):
        return "Overflow"
    if mode == "ALL":
        s = "%.10g" % x
        return s
    if mode == "FIX":
        if x != 0 and (abs(x) >= 1e10 or abs(x) < 10 ** (-digits)):
            return "%.*e" % (digits, x)
        return "%.*f" % (digits, x)
    if mode == "SCI":
        return "%.*e" % (digits, x)
    if mode == "ENG":
        if x == 0:
            return "%.*fe+00" % (digits, 0.0)
        e = int(math.floor(math.log10(abs(x)) / 3) * 3)
        m = x / 10 ** e
        return "%.*fe%+03d" % (max(0, digits - (len(str(int(abs(m)))) - 1)), m, e)
    return str(x)


# =============================================================================
class RPNEngine(AngleMixin):
    """HP-15C-compatible RPN core. Unsupported programming keys report a message."""

    def __init__(self, model="HP-15C"):
        self.model = model
        self.X = self.Y = self.Z = self.T = 0.0
        self.LASTX = 0.0
        self.lift = True
        self.entry = None          # string being typed, or None
        self.exp_entry = None      # exponent digits being typed after EEX
        self.regs = {}
        self.pending = None        # 'STO' | 'RCL' | 'FIX' | 'SCI' | 'ENG' (+ optional op)
        self.pending_op = None
        self.disp_mode, self.disp_digits = "FIX", 4
        self.angle = "DEG"
        self.hyp = None            # None | 'HYP' | 'AHYP'
        self.message = ""
        self.stats_clear()

    # ---------------- stack primitives
    def _lift(self):
        self.T, self.Z, self.Y = self.Z, self.Y, self.X

    def _drop(self):
        self.Y, self.Z = self.Z, self.T

    def _finish_entry(self):
        if self.entry is not None:
            self.X = self._entry_value()
            self.entry = None
            self.exp_entry = None

    def _entry_value(self):
        s = self.entry or "0"
        if s in ("", "-", ".", "-."):
            s = s.replace(".", "") + "0"
        v = float(s if not s.endswith(".") else s[:-1] or "0")
        if self.exp_entry is not None:
            e = self.exp_entry or "0"
            if e in ("-", "+"):
                e = "0"
            v = v * 10 ** int(e)
        return v

    def stats_clear(self):
        self.S = dict(n=0.0, sx=0.0, sx2=0.0, sy=0.0, sy2=0.0, sxy=0.0)

    # ---------------- display
    def display(self):
        if self.message:
            return self.message
        if self.entry is not None:
            s = self.entry if self.entry else "0"
            if self.exp_entry is not None:
                e = self.exp_entry
                sign = "-" if e.startswith("-") else ""
                e = e.lstrip("-")
                s = "%s  %s%s" % (s, sign, e.rjust(2, "0") if e else "00")
            return s
        return fmt_number(self.X, self.disp_mode, self.disp_digits)

    def stack(self):
        x = self._entry_value() if self.entry is not None else self.X
        return dict(X=x, Y=self.Y, Z=self.Z, T=self.T, LASTX=self.LASTX)

    # ---------------- key dispatcher
    def press(self, key):
        self.message = ""
        if self.pending:
            return self._pending_key(key)
        if key in "0123456789" or key == ".":
            return self._digit(key)
        h = getattr(self, "k_" + key, None)
        if h is None:
            self._finish_entry()
            self.message = "%s: not simulated" % key
            return
        h()

    # ---------------- entry keys
    def _digit(self, d):
        if self.entry is None:
            if self.lift:
                self._lift()
            self.entry = ""
            self.exp_entry = None
        if self.exp_entry is not None:
            digits = self.exp_entry.lstrip("-")
            if d == ".":
                return
            digits = (digits + d)[-2:]
            self.exp_entry = ("-" if self.exp_entry.startswith("-") else "") + digits
            return
        if d == "." and "." in self.entry:
            return
        if len(self.entry.replace("-", "").replace(".", "")) >= 10 and d != ".":
            return
        self.entry += d
        self.lift = True

    def k_EEX(self):
        if self.entry is None:
            self._digit("1")
        if self.exp_entry is None:
            self.exp_entry = ""

    def k_CHS(self):
        if self.entry is not None:
            if self.exp_entry is not None:
                self.exp_entry = self.exp_entry[1:] if self.exp_entry.startswith("-") else "-" + self.exp_entry
            else:
                self.entry = self.entry[1:] if self.entry.startswith("-") else "-" + self.entry
        else:
            self.X = -self.X
            self.lift = True

    def k_BSP(self):
        if self.entry is not None:
            if self.exp_entry is not None:
                if self.exp_entry.lstrip("-"):
                    self.exp_entry = self.exp_entry[:-1]
                else:
                    self.exp_entry = None
                return
            self.entry = self.entry[:-1]
            if self.entry in ("", "-"):
                self.entry = None
                self.X = 0.0
                self.lift = False
            return
        self.k_CLX()

    def k_ENTER(self):
        self._finish_entry()
        self._lift()
        self.lift = False

    def k_CLX(self):
        self.entry = None
        self.exp_entry = None
        self.X = 0.0
        self.lift = False

    def k_CLSTK(self):
        self.entry = None
        self.X = self.Y = self.Z = self.T = 0.0
        self.lift = False

    # ---------------- generic helpers
    def _unary(self, f):
        self._finish_entry()
        try:
            r = f(self.X)
        except (ValueError, ZeroDivisionError, OverflowError) as e:
            self.message = "Error 0" if not isinstance(e, OverflowError) else "Overflow"
            return
        if isinstance(r, complex):
            self.message = "Error 0"
            return
        self.LASTX = self.X
        self.X = r
        self.lift = True

    def _binary(self, f):
        self._finish_entry()
        try:
            r = f(self.Y, self.X)
        except (ValueError, ZeroDivisionError, OverflowError) as e:
            self.message = "Error 0" if not isinstance(e, OverflowError) else "Overflow"
            return
        if isinstance(r, complex):
            self.message = "Error 0"
            return
        self.LASTX = self.X
        self.X = r
        self._drop()
        self.lift = True

    def _push(self, v):
        self._finish_entry()
        if self.lift:
            self._lift()
        self.X = v
        self.lift = True

    # ---------------- arithmetic
    def k_ADD(self): self._binary(lambda y, x: y + x)
    def k_SUB(self): self._binary(lambda y, x: y - x)
    def k_MUL(self): self._binary(lambda y, x: y * x)
    def k_DIV(self): self._binary(lambda y, x: y / x)
    def k_YX(self): self._binary(lambda y, x: y ** x)
    def k_XROOTY(self): self._binary(lambda y, x: y ** (1.0 / x))

    def k_PCT(self):        # % : X = Y*X/100, Y kept
        self._finish_entry()
        self.LASTX = self.X
        self.X = self.Y * self.X / 100.0
        self.lift = True

    def k_DPCT(self):       # Δ% : 100 (X - Y) / Y, Y kept
        self._finish_entry()
        try:
            r = 100.0 * (self.X - self.Y) / self.Y
        except ZeroDivisionError:
            self.message = "Error 0"; return
        self.LASTX = self.X
        self.X = r
        self.lift = True

    # ---------------- unary functions
    def k_SQRT(self): self._unary(lambda x: math.sqrt(x))
    def k_SQ(self): self._unary(lambda x: x * x)
    def k_EXP(self): self._unary(math.exp)
    def k_LN(self): self._unary(math.log)
    def k_10X(self): self._unary(lambda x: 10.0 ** x)
    def k_LOG(self): self._unary(math.log10)
    def k_INV(self): self._unary(lambda x: 1.0 / x)
    def k_ABS(self): self._unary(abs)
    def k_INT(self): self._unary(lambda x: float(math.trunc(x)))
    def k_FRAC(self): self._unary(lambda x: x - math.trunc(x))
    def k_FACT(self): self._unary(_fact)
    def k_RND(self): self._unary(lambda x: float(fmt_number(x, self.disp_mode, self.disp_digits).replace("e", "E")) if self.disp_mode != "ALL" else x)
    def k_HMS(self): self._unary(_hms)
    def k_HR(self): self._unary(_hr)
    def k_TORAD(self): self._unary(math.radians)
    def k_TODEG(self): self._unary(math.degrees)
    # HP 35s conversion keys (exact factors)
    def k_TOF(self): self._unary(lambda c: c * 9.0 / 5.0 + 32.0)
    def k_TOC(self): self._unary(lambda f: (f - 32.0) * 5.0 / 9.0)
    def k_TOLB(self): self._unary(lambda kg: kg / 0.45359237)
    def k_TOKG(self): self._unary(lambda lb: lb * 0.45359237)
    def k_TOMILE(self): self._unary(lambda km: km / 1.609344)
    def k_TOKM(self): self._unary(lambda mi: mi * 1.609344)
    def k_TOIN(self): self._unary(lambda cm: cm / 2.54)
    def k_TOCM(self): self._unary(lambda inch: inch * 2.54)
    def k_TOGAL(self): self._unary(lambda l: l / 3.785411784)
    def k_TOL(self): self._unary(lambda gal: gal * 3.785411784)

    def _trig(self, name):
        hyp = self.hyp
        self.hyp = None
        if hyp == "HYP":
            f = {"SIN": math.sinh, "COS": math.cosh, "TAN": math.tanh}[name]
            return self._unary(f)
        if hyp == "AHYP":
            f = {"SIN": math.asinh, "COS": math.acosh, "TAN": math.atanh}[name]
            return self._unary(f)
        f = {"SIN": math.sin, "COS": math.cos, "TAN": math.tan}[name]
        self._unary(lambda x: f(self.to_rad(x)))

    def k_SIN(self): self._trig("SIN")
    def k_COS(self): self._trig("COS")
    def k_TAN(self): self._trig("TAN")
    def k_ASIN(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("SIN")
        self._unary(lambda x: self.from_rad(math.asin(x)))
    def k_ACOS(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("COS")
        self._unary(lambda x: self.from_rad(math.acos(x)))
    def k_ATAN(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("TAN")
        self._unary(lambda x: self.from_rad(math.atan(x)))
    def k_HYP(self): self._finish_entry(); self.hyp = "HYP"
    def k_AHYP(self): self._finish_entry(); self.hyp = "AHYP"

    # ---------------- stack manipulation
    def k_SWAP(self):
        self._finish_entry()
        self.X, self.Y = self.Y, self.X
        self.lift = True

    def k_RDN(self):
        self._finish_entry()
        self.X, self.Y, self.Z, self.T = self.Y, self.Z, self.T, self.X
        self.lift = True

    def k_RUP(self):
        self._finish_entry()
        self.X, self.Y, self.Z, self.T = self.T, self.X, self.Y, self.Z
        self.lift = True

    def k_LASTX(self): self._push(self.LASTX)
    def k_PI(self): self._push(math.pi)
    def k_E(self): self._push(math.e)

    # ---------------- polar / rectangular (15C: X=x? -> f→R uses X=r, Y=θ)
    def k_TOP(self):     # rectangular (x in X, y in Y) -> polar: X = r, Y = θ
        self._finish_entry()
        x, y = self.X, self.Y
        self.LASTX = self.X
        self.X = math.hypot(x, y)
        self.Y = self.from_rad(math.atan2(y, x))
        self.lift = True

    def k_TOR(self):     # polar (r in X, θ in Y) -> rectangular: X = x, Y = y
        self._finish_entry()
        r, th = self.X, self.to_rad(self.Y)
        self.LASTX = self.X
        self.X, self.Y = r * math.cos(th), r * math.sin(th)
        self.lift = True

    def k_PERM(self): self._binary(_perm)
    def k_COMB(self): self._binary(_comb)

    # ---------------- statistics (Σ registers as on HP-15C)
    def k_SIGMAP(self, sign=1):
        self._finish_entry()
        x, y = self.X, self.Y
        S = self.S
        S["n"] += sign; S["sx"] += sign * x; S["sx2"] += sign * x * x
        S["sy"] += sign * y; S["sy2"] += sign * y * y; S["sxy"] += sign * x * y
        self.LASTX = x
        self.X = S["n"]
        self.lift = False

    def k_SIGMAM(self): self.k_SIGMAP(-1)

    def _need(self, k):
        if self.S["n"] < k:
            self.message = "Error 2"
            return False
        return True

    def k_MEAN(self):
        self._finish_entry()
        if not self._need(1): return
        n = self.S["n"]
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = self.S["sx"] / n, self.S["sy"] / n
        self.lift = True

    def k_SDEV(self):
        self._finish_entry()
        if not self._need(2): return
        n, S = self.S["n"], self.S
        sx = math.sqrt((n * S["sx2"] - S["sx"] ** 2) / (n * (n - 1)))
        sy = math.sqrt(max(0.0, (n * S["sy2"] - S["sy"] ** 2) / (n * (n - 1))))
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = sx, sy
        self.lift = True

    def _lr(self):
        n, S = self.S["n"], self.S
        M = n * S["sx2"] - S["sx"] ** 2
        N = n * S["sy2"] - S["sy"] ** 2
        P = n * S["sxy"] - S["sx"] * S["sy"]
        slope = P / M
        icpt = (M * S["sy"] - P * S["sx"]) / (n * M)
        r = P / math.sqrt(M * N) if N > 0 else float("nan")
        return slope, icpt, r

    def k_LR(self):         # X = intercept, Y = slope (HP-15C L.R.)
        self._finish_entry()
        if not self._need(2): return
        slope, icpt, _ = self._lr()
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = icpt, slope
        self.lift = True

    def k_YHATR(self):      # X = ŷ at x in X, Y = r
        self._finish_entry()
        if not self._need(2): return
        slope, icpt, r = self._lr()
        self.LASTX = self.X
        yhat = icpt + slope * self.X
        self._lift()
        self.X, self.Y = yhat, r
        self.lift = True

    def k_CLSIGMA(self):
        self.stats_clear()
        self.k_CLSTK()

    # ---------------- modes and prefixes
    def k_DEG(self): self.angle = "DEG"
    def k_RAD(self): self.angle = "RAD"
    def k_GRD(self): self.angle = "GRD"
    def k_FIX(self): self._finish_entry(); self.pending = "FIX"
    def k_SCI(self): self._finish_entry(); self.pending = "SCI"
    def k_ENG(self): self._finish_entry(); self.pending = "ENG"
    def k_ALL(self): self.disp_mode = "ALL"
    def k_STO(self): self._finish_entry(); self.pending = "STO"; self.pending_op = None
    def k_RCL(self): self._finish_entry(); self.pending = "RCL"; self.pending_op = None

    def _pending_key(self, key):
        p = self.pending
        if p in ("FIX", "SCI", "ENG"):
            if key in "0123456789":
                self.disp_mode, self.disp_digits = p, int(key)
            self.pending = None
            return
        if p in ("STO", "RCL"):
            if key in ("ADD", "SUB", "MUL", "DIV") and self.pending_op is None:
                self.pending_op = key
                return
            if key == ".":
                self.pending_op = (self.pending_op or "") + "."
                return
            if key in "0123456789":
                op = self.pending_op or ""
                dot = "." if op.endswith(".") else ""
                op = op.rstrip(".")
                name = dot + key
                self.pending, self.pending_op = None, None
                if p == "STO":
                    cur = self.regs.get(name, 0.0)
                    v = {"": self.X, "ADD": cur + self.X, "SUB": cur - self.X,
                         "MUL": cur * self.X, "DIV": (cur / self.X) if self.X else float("nan")}[op]
                    self.regs[name] = v
                    self.lift = True
                else:
                    v = self.regs.get(name, 0.0)
                    if op:
                        self.LASTX = self.X
                        self.X = {"ADD": self.X + v, "SUB": self.X - v, "MUL": self.X * v,
                                  "DIV": (self.X / v) if v else float("nan")}[op]
                        self.lift = True
                    else:
                        self._push(v)
                return
            self.pending, self.pending_op = None, None
            return
        self.pending = None


# =============================================================================
class AlgEngine(AngleMixin):
    """Algebraic entry-line calculator with textbook precedence.
    Keys append tokens; '=' (ENTER/EXE) evaluates. ANS holds the last result."""

    FUNCS1 = {"sin(": "sin", "cos(": "cos", "tan(": "tan", "sin⁻¹(": "asin", "cos⁻¹(": "acos", "tan⁻¹(": "atan",
              "sinh(": "sinh", "cosh(": "cosh", "tanh(": "tanh", "sinh⁻¹(": "asinh", "cosh⁻¹(": "acosh",
              "tanh⁻¹(": "atanh", "ln(": "ln", "log(": "log", "√(": "sqrt", "e^(": "exp", "10^(": "pow10",
              "abs(": "abs", "∛(": "cbrt"}

    def __init__(self, model="TI-30XS MultiView", implicit_mult=True, neg_power_first=True):
        self.model = model
        self.line = []            # list of tokens (strings)
        self.ans = 0.0
        self.result = None
        self.history = []
        self.vars = {}
        self.angle = "DEG"
        self.just_evaluated = False
        self.message = ""
        self.implicit = implicit_mult
        self.pending_sto = False
        self.disp_mode, self.disp_digits = "ALL", 10

    # ---------------- display helpers
    def entry_text(self):
        return "".join(self.line)

    def display(self):
        if self.message:
            return self.message
        return fmt_number(self.result, self.disp_mode, self.disp_digits) if self.result is not None else ""

    # ---------------- keys
    def press(self, key):
        self.message = ""
        if self.pending_sto:
            self.pending_sto = False
            if key.startswith("VAR_"):
                v = self._evaluate(store=False)
                if v is not None:
                    self.vars[key[4:]] = v
                    self.result = v
                    self.ans = v
                    self.history.append((list(self.line), v))
                    self.just_evaluated = True
                return
        if key in ("EQ", "ENTER", "EXE"):
            if self.line == [] and self.history:
                self.line = list(self.history[-1][0])
            v = self._evaluate()
            return
        if key in ("CLEAR", "AC"):
            if self.line:
                self.line = []
            else:
                self.result = None
            self.just_evaluated = False
            return
        if key in ("DEL", "BSP"):
            if self.line:
                self.line.pop()
            return
        if key == "STO":
            self.pending_sto = True
            return
        if key == "ENTRY":               # TI-84 2nd ENTRY: recall the last entry line
            if self.history:
                self.line = list(self.history[-1][0]); self.just_evaluated = False
            return
        if key == "DEG": self.angle = "DEG"; return
        if key == "RAD": self.angle = "RAD"; return
        if key == "GRD": self.angle = "GRD"; return
        tok = self._token_for(key)
        if tok is None:
            self.message = "%s: not simulated" % key
            return
        if self.just_evaluated:
            # an operator continues from Ans; anything else starts a new line
            if tok in ("+", "−", "×", "÷", "^", "²", "³", "⁻¹", "!", "E", "%"):
                self.line = ["Ans"]
            else:
                self.line = []
            self.just_evaluated = False
        self.line.append(tok)

    TOKENS = {"ADD": "+", "SUB": "−", "MUL": "×", "DIV": "÷", "NEG": "(−)", "LP": "(", "RP": ")",
              "YX": "^", "SQ": "²", "INV": "⁻¹", "SQRT": "√(", "SIN": "sin(", "COS": "cos(", "TAN": "tan(",
              "ASIN": "sin⁻¹(", "ACOS": "cos⁻¹(", "ATAN": "tan⁻¹(", "SINH": "sinh(", "COSH": "cosh(",
              "TANH": "tanh(", "ASINH": "sinh⁻¹(", "ACOSH": "cosh⁻¹(", "ATANH": "tanh⁻¹(",
              "LN": "ln(", "LOG": "log(", "EXP": "e^(", "10X": "10^(", "PI": "π", "E": "e", "EEX": "E",
              "ANS": "Ans", "FACT": "!", "PERM": " nPr ", "COMB": " nCr ", "ABS": "abs(", "COMMA": ",",
              "PCT": "%", "XROOTY": "ˣ√(", "CBRT": "∛(", "CUBE": "³"}

    def _token_for(self, key):
        if key in "0123456789" or key == ".":
            return key
        if key.startswith("VAR_"):
            return key[4:]
        return self.TOKENS.get(key)

    # ---------------- evaluation: tokenizer + recursive descent
    def _evaluate(self, store=True):
        text = self.entry_text()
        try:
            toks = self._lex(text)
            self._toks, self._i = toks, 0
            v = self._expr()
            if self._i != len(self._toks):
                raise SyntaxError("unexpected %r" % (self._toks[self._i],))
            if isinstance(v, complex) or v != v:
                raise ValueError("domain")
        except ZeroDivisionError:
            self.message = "DIVIDE BY 0"; return None
        except (ValueError, OverflowError):
            self.message = "DOMAIN" if True else ""; return None
        except (SyntaxError, IndexError, KeyError):
            self.message = "SYNTAX ERROR"; return None
        if store:
            self.history.append((list(self.line), v))
            self.ans = v
            self.result = v
            self.just_evaluated = True
        return v

    def _lex(self, s):
        out, i = [], 0
        words = sorted(list(self.FUNCS1) + ["Ans", " nPr ", " nCr ", "(−)", "⁻¹", "ˣ√("], key=len, reverse=True)
        while i < len(s):
            c = s[i]
            if c == " " and not any(s.startswith(w, i) for w in (" nPr ", " nCr ")):
                i += 1; continue
            m = next((w for w in words if s.startswith(w, i)), None)
            if m:
                out.append(("FN", self.FUNCS1[m]) if m in self.FUNCS1 else ("SYM", m.strip()))
                i += len(m); continue
            if c in "0123456789.":
                j = i
                while j < len(s) and s[j] in "0123456789.":
                    j += 1
                num = s[i:j]
                if num.count(".") > 1:
                    raise SyntaxError("number")
                out.append(("NUM", float(num if num != "." else "0")))
                i = j; continue
            if c in "+−×÷^²³!(),%E":
                out.append(("SYM", c)); i += 1; continue
            if c == "-":
                out.append(("SYM", "−")); i += 1; continue
            if c in "πe":
                out.append(("NUM", math.pi if c == "π" else math.e)); i += 1; continue
            if c.isalpha():
                out.append(("VAR", c)); i += 1; continue
            raise SyntaxError(c)
        return out

    def _peek(self):
        return self._toks[self._i] if self._i < len(self._toks) else (None, None)

    def _take(self):
        t = self._toks[self._i]; self._i += 1; return t

    def _expr(self):
        v = self._term()
        while self._peek() in (("SYM", "+"), ("SYM", "−")):
            op = self._take()[1]
            r = self._term()
            v = v + r if op == "+" else v - r
        return v

    def _starts_factor(self):
        k, t = self._peek()
        return k in ("NUM", "FN", "VAR") or (k == "SYM" and t in ("(", "(−)", "√")) or (k == "SYM" and t == "Ans")

    def _term(self):
        v = self._factor()
        while True:
            k, t = self._peek()
            if (k, t) in (("SYM", "×"), ("SYM", "÷")):
                self._take()
                r = self._factor()
                v = v * r if t == "×" else v / r
            elif (k, t) in (("SYM", "nPr"), ("SYM", "nCr")):
                self._take()
                r = self._factor()
                v = float(_perm(v, r) if t == "nPr" else _comb(v, r))
            elif self.implicit and self._starts_factor():
                v = v * self._factor()
            else:
                return v

    def _factor(self):
        k, t = self._peek()
        if (k, t) == ("SYM", "(−)") or (k, t) == ("SYM", "−"):
            self._take()
            return -self._factor()          # negation binds looser than ^ : (−)2² = −4
        return self._power()

    def _power(self):
        base = self._postfix()
        if self._peek() == ("SYM", "^"):
            self._take()
            expo = self._factor()            # right-associative, allows (−) in exponent
            return base ** expo
        return base

    def _postfix(self):
        v = self._primary()
        while True:
            k, t = self._peek()
            if (k, t) == ("SYM", "²"):
                self._take(); v = v * v
            elif (k, t) == ("SYM", "³"):
                self._take(); v = v * v * v
            elif (k, t) == ("SYM", "⁻¹"):
                self._take(); v = 1.0 / v
            elif (k, t) == ("SYM", "!"):
                self._take(); v = _fact(v)
            elif (k, t) == ("SYM", "ˣ√("):
                self._take()
                inner = self._expr()
                if self._peek() == ("SYM", ")"):
                    self._take()
                v = inner ** (1.0 / v)
            elif (k, t) == ("SYM", "%"):
                self._take(); v = v / 100.0
            elif (k, t) == ("SYM", "E"):
                self._take()
                sign = 1
                if self._peek() in (("SYM", "(−)"), ("SYM", "−")):
                    self._take(); sign = -1
                kk, ee = self._take()
                if kk != "NUM":
                    raise SyntaxError("EE")
                v = v * 10 ** (sign * ee)
            else:
                return v

    def _primary(self):
        k, t = self._take()
        if k == "NUM":
            return t
        if k == "VAR":
            return self.vars.get(t, 0.0)
        if (k, t) == ("SYM", "Ans"):
            return self.ans
        if (k, t) == ("SYM", "("):
            v = self._expr()
            if self._peek() == ("SYM", ")"):
                self._take()
            return v                          # closing parenthesis optional at the end, as on TI/Casio
        if (k, t) == ("SYM", "ˣ√("):
            raise SyntaxError("x-root needs index first")
        if k == "FN":
            v = self._expr()
            if self._peek() == ("SYM", ")"):
                self._take()
            return self._apply(t, v)
        raise SyntaxError(t)

    def _apply(self, fn, v):
        r = self.to_rad
        f = {"sin": lambda x: math.sin(r(x)), "cos": lambda x: math.cos(r(x)), "tan": lambda x: math.tan(r(x)),
             "asin": lambda x: self.from_rad(math.asin(x)), "acos": lambda x: self.from_rad(math.acos(x)),
             "atan": lambda x: self.from_rad(math.atan(x)), "sinh": math.sinh, "cosh": math.cosh,
             "tanh": math.tanh, "asinh": math.asinh, "acosh": math.acosh, "atanh": math.atanh,
             "ln": math.log, "log": math.log10, "sqrt": math.sqrt, "exp": math.exp,
             "pow10": lambda x: 10.0 ** x, "abs": abs, "cbrt": lambda x: math.copysign(abs(x) ** (1 / 3), x)}[fn]
        return f(v)


def run_keys(engine, keys):
    """Feed a space-separated key string to an engine; returns the engine."""
    for k in keys.split():
        engine.press(k)
    return engine


# ---------------------------------------------------------------------------
# Step-by-step calculation cards (equation -> case -> worked solution)
#
#   1. Equation: name, colour-coded LaTeX equation, table of variables and
#      constants (meaning, role, units, source).
#   2. Case: pharmacy / pharmaceutical-sciences narrative with each given
#      value highlighted in the same colour as its symbol.
#   3. Solution table: general equation | values from the case | unit check
#      (prefixes expanded, cancellations noted, conversions) | substitution |
#      arithmetic steps | answer | what the answer means.
# The same card renders static Markdown (build time) and an interactive
# solver (Colab), and logs every solve to the lab report.
# ---------------------------------------------------------------------------
import math
import re

PALETTE = ["#1d6fb8", "#c2410c", "#15803d", "#7c3aed", "#b91c1c", "#0e7490", "#a16207", "#be185d"]

# ---- units ---------------------------------------------------------------
PREFIX = {"p": (-12, "pico"), "n": (-9, "nano"), "µ": (-6, "micro"), "u": (-6, "micro"),
          "m": (-3, "milli"), "c": (-2, "centi"), "d": (-1, "deci"), "k": (3, "kilo"), "M": (6, "mega")}
BASE = {"g": "gram", "L": "litre", "mol": "mole", "M": "molar (mol/L)", "Da": "dalton", "s": "second",
        "J": "joule", "cal": "calorie", "eq": "equivalent", "m": "metre", "Pa": "pascal", "U": "enzyme unit",
        "kat": "katal", "Hz": "hertz", "V": "volt", "W": "watt"}
NO_PREFIX = {"min", "h", "day", "week", "yr", "%", "mmHg", "°C", "K", "bp", "kb", "Å", "ppm", "dL"}
SPECIAL = {"dL": (-1, "L"), "mmHg": None, "kb": None}


def parse_unit_token(tok):
    """'mg' -> (exp=-3, base='g', prefix='milli'); 'min' -> (0, 'min', '')."""
    t = tok.strip()
    p = 1
    m = re.match(r"^(.*?)(\^(-?\d+)|²|³)?$", t)
    core = m.group(1)
    if m.group(2):
        p = 2 if m.group(2) == "²" else 3 if m.group(2) == "³" else int(m.group(3))
    if core in NO_PREFIX and core not in SPECIAL:
        return 0, core, "", p
    if core == "dL":
        return -1, "L", "deci", p
    if core in BASE:
        return 0, core, "", p
    if len(core) > 1 and core[0] in PREFIX and core[1:] in BASE:
        e, name = PREFIX[core[0]]
        return e, core[1:], name, p
    return 0, core, "", p


def unit_factor(unit):
    """Return (power of ten, base-unit string) for a compound unit like 'mg/dL'."""
    if not unit:
        return 0, ""
    num, _, den = unit.partition("/")
    e_tot, parts_n, parts_d = 0, [], []
    for tok in re.split(r"[·*\s]+", num.strip()):
        if tok:
            e, b, _, p = parse_unit_token(tok)
            e_tot += e * p; parts_n.append(b + ("^%d" % p if p != 1 else ""))
    for tok in re.split(r"[·*/\s]+", den.strip()):
        if tok:
            e, b, _, p = parse_unit_token(tok)
            e_tot -= e * p; parts_d.append(b + ("^%d" % p if p != 1 else ""))
    base = "·".join(parts_n) + ("/" + "·".join(parts_d) if parts_d else "")
    return e_tot, base


def _plain(x, sig=4):
    """sig significant figures without exponent notation (1e-3 <= |x| < 1e5)."""
    d = int(math.floor(math.log10(abs(x))))
    v = round(x, sig - 1 - d)
    dec = max(0, sig - 1 - d)
    t = ("%.*f" % (dec, v))
    if "." in t:
        t = t.rstrip("0").rstrip(".")
    return t


def sci(x, sig=4):
    """LaTeX scientific notation with sig figs: 1.23 \\times 10^{-4}."""
    if x == 0:
        return "0"
    if isinstance(x, int) and abs(x) < 1e5 or (abs(x) >= 1e-3 and abs(x) < 1e5):
        return _plain(x, sig)
    e = int(math.floor(math.log10(abs(x))))
    m = x / 10 ** e
    return r"%s \times 10^{%d}" % (("%.*g" % (sig, m)), e)


def fmt(x, sig=4):
    if isinstance(x, str):
        return x
    if x == 0:
        return "0"
    if abs(x) >= 1e5 or abs(x) < 1e-3:
        return sci(x, sig)
    return _plain(x, sig)


def utex(u):
    """Unit string to LaTeX (upright)."""
    if not u:
        return ""
    u = u.replace("µ", r"\mu ").replace("·", r"\cdot ").replace("%", r"\%")
    u = re.sub(r"\^(-?\d+)", r"^{\1}", u)
    u = u.strip().replace(" ", r"\ ")
    return r"\,\mathrm{%s}" % u


def expand_prefix(value, unit):
    """LaTeX showing the prefix written out, e.g. 5 mg/dL = 5 × 10⁻³ g / 10⁻¹ L."""
    e, base = unit_factor(unit)
    if e == 0:
        return None
    return r"%s%s = %s \times 10^{%d}\,\mathrm{%s} = %s%s" % (
        fmt(value), utex(unit), fmt(value), e, base.replace("µ", r"\mu "), sci(value * 10 ** e), utex(base))


def _katex_safe(s):
    """KaTeX (Colab) rejects a superscript on a spacing command: \\,^{\\circ} -> \\,{}^{\\circ}."""
    return re.sub(r"(\\[,;:! ]|\\quad|\\qquad)\s*\^", r"\1{}^", s) if isinstance(s, str) else s


# ---- card model -------------------------------------------------------------
CARDS = {}


class Var:
    def __init__(self, key, sym, name, unit="", role="input", source="", value=None, color=None, const=False):
        self.key, self.sym, self.name, self.unit, self.role = key, sym, name, unit, role
        self.source, self.value, self.color, self.const = source, value, color, const

    def tex(self):
        return r"{\color{%s}{%s}}" % (self.color, self.sym)

    def hl(self, value=None, unit=None):
        v = self.value if value is None else value
        u = self.unit if unit is None else unit
        return r"$\mathbf{{\color{%s}{%s%s}}}$" % (self.color, fmt(v), utex(u))


class Card:
    """A worked calculation. `solve(card, vals)` must return a Solution."""

    def __init__(self, id, title, equation, variables, case, solve, constants=(), conversions=(),
                 result_name="", result_unit="", source="", inputs=None):
        self.id, self.title, self.equation = id, title, equation
        self.vars = list(variables) + list(constants)
        for k, v in enumerate(self.vars):
            if v.color is None:
                v.color = "#4b5563" if v.const else PALETTE[k % len(PALETTE)]
        self.V = {v.key: v for v in self.vars}
        self.case, self.solve_fn = case, solve
        self.conversions = list(conversions)
        self.result_name, self.result_unit, self.source = result_name, result_unit, source
        self.inputs = inputs or [v.key for v in self.vars if v.role == "input" and not v.const]
        CARDS[id] = self

    # ---- LaTeX helpers
    def eq_tex(self, template=None):
        t = template or self.equation
        return re.sub(r"\{\{(\w+)\}\}", lambda m: self.V[m.group(1)].tex(), t)

    def defaults(self):
        return {v.key: v.value for v in self.vars if v.value is not None}

    def solve(self, **vals):
        d = self.defaults()
        d.update(vals)
        return self.solve_fn(self, d)

    # ---- static markdown
    def md_equation(self):
        return _katex_safe(self._md_equation())

    def _md_equation(self):
        rows = ["| Symbol | Meaning | Role in the calculation | Units | Source / value |", "|---|---|---|---|---|"]
        for v in self.vars:
            role = "constant" if v.const else v.role
            src = v.source
            if v.const and v.value is not None:
                src = "%s = %s %s; %s" % ("value", fmt(v.value), v.unit, v.source)
            rows.append("| $%s$ | %s | %s | %s | %s |" % (v.tex(), v.name, role, (v.unit or "—").replace("|", "/"), src))
        return ("**1. %s**\n\n$$%s$$\n\n" % (self.title, self.eq_tex())) + "\n".join(rows) + (
            "\n\n*Source of the relationship:* %s" % self.source if self.source else "")

    def md_case(self, vals=None):
        return _katex_safe(self._md_case(vals))

    def _md_case(self, vals=None):
        d = self.defaults()
        if vals:
            d.update(vals)
        txt = re.sub(r"\[\[(\w+)\]\]", lambda m: self.V[m.group(1)].hl(d.get(m.group(1))), self.case)
        return "**2. Practice problem (case)**\n\n" + txt

    def markdown(self, vals=None, collapse=True):
        return _katex_safe(self._markdown(vals, collapse))

    def _markdown(self, vals=None, collapse=True):
        sol = self.solve(**(vals or {}))
        body = sol.markdown()
        s3 = "**3. Step-by-step solution**\n\n" + body
        if collapse:
            s3 = "<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>\n\n" + body + "\n</details>"
        return self.md_equation() + "\n\n" + self.md_case(vals) + "\n\n" + s3


class Solution:
    def __init__(self, card, vals):
        self.card, self.vals, self.rows, self.result, self.unit, self.meaning = card, vals, [], None, "", ""
        self.extra_conversions = []

    # rows are (label, latex-or-markdown, kind)
    def general(self, template=None):
        self.rows.append(("General equation", "$%s$" % self.card.eq_tex(template), "tex"))
        return self

    def given(self):
        parts = []
        for k in self.card.inputs:
            v = self.card.V[k]
            parts.append("$%s = %s%s$" % (v.tex(), fmt(self.vals[k]), utex(v.unit)))
        consts = [v for v in self.card.vars if v.const]
        for v in consts:
            parts.append("$%s = %s%s$ (constant)" % (v.tex(), fmt(self.vals.get(v.key, v.value)), utex(v.unit)))
        self.rows.append(("Values from the case", "; ".join(parts), "md"))
        return self

    def unit_check(self, note=""):
        """Expand every prefix; say whether units cancel or carry through."""
        bits = []
        for v in self.card.vars:
            if v.key not in self.vals or not v.unit:
                continue
            ex = expand_prefix(self.vals[v.key], v.unit)
            if ex:
                bits.append("$%s$" % ex)
        units = sorted({v.unit for v in self.card.vars if v.key in self.vals and v.unit})
        txt = ("Prefixes written out: " + "; ".join(bits) + ". ") if bits else "No unit prefixes to expand. "
        txt += note or ("Units present: " + ", ".join(units) + ". " if units else "")
        self.rows.append(("Unit check", txt, "md"))
        return self

    def sub(self, latex, label="Substitute"):
        self.rows.append((label, "$%s$" % latex, "tex"))
        return self

    def step(self, label, latex):
        self.rows.append((label, "$%s$" % latex, "tex"))
        return self

    def answer(self, value, unit="", meaning="", label=None):
        self.result, self.unit, self.meaning = value, unit, meaning
        self.rows.append((label or "Answer", r"$\boxed{%s = %s%s}$" % (
            self.card.result_name or "x", fmt(value), utex(unit)), "tex"))
        if meaning:
            self.rows.append(("What it means", meaning, "md"))
        return self

    def convert(self, text):
        self.extra_conversions.append(text)
        return self

    def markdown(self):
        return _katex_safe(self._markdown())

    def _markdown(self):
        out = ["| Step | Work |", "|---|---|"]
        for k, (lab, s, kind) in enumerate(self.rows, 1):
            out.append("| %d. %s | %s |" % (k, lab, s.replace("|", r"\vert ")))
        conv = self.card.conversions + self.extra_conversions
        if conv:
            out.append("\n**Common conversions for this calculation:** " + "; ".join(conv))
        return "\n".join(out)

    def plain_lines(self):
        """[(label, latex_or_text)] for the report renderer."""
        return [(lab, _katex_safe(s)) for lab, s, _ in self.rows]


# ---- interactive solver (Colab) ------------------------------------------
def solver_widget(card, report=None, calculator=None):
    """Input boxes for the card's inputs + 'Solve step by step' + optional calculator."""
    try:
        import ipywidgets as W
        from IPython.display import display, Markdown, clear_output
    except Exception:                                     # plain Python (validator)
        sol = card.solve()
        print(sol.markdown())
        if report is not None:
            report.add_calc(card, sol)
        return sol
    boxes = {}
    for k in card.inputs:
        v = card.V[k]
        boxes[k] = W.FloatText(value=float(v.value if v.value is not None else 0.0),
                               description="%s (%s)" % (v.name[:28], v.unit or "–"),
                               style={"description_width": "230px"}, layout=W.Layout(width="420px"))
    btn = W.Button(description="Solve step by step", button_style="success", icon="list-ol",
                   layout=W.Layout(width="200px"))
    cbtn = W.Button(description="Open calculator", icon="calculator", layout=W.Layout(width="200px"))
    out, cout = W.Output(), W.Output()

    def go(_=None):
        with out:
            clear_output()
            try:
                sol = card.solve(**{k: b.value for k, b in boxes.items()})
            except (ValueError, ZeroDivisionError) as e:
                print("Check the inputs:", e)
                return
            display(Markdown(card.md_case({k: b.value for k, b in boxes.items()}) + "\n\n" + sol.markdown()))
            if report is not None:
                report.add_calc(card, sol)

    def calc(_=None):
        with cout:
            clear_output()
            if calculator is not None:
                calculator()
    btn.on_click(go)
    cbtn.on_click(calc)
    display(W.VBox(list(boxes.values()) + [W.HBox([btn, cbtn]), out, cout]))
    go()


# ---------------------------------------------------------------------------
# Lab report collector: every figure shown, every step-by-step calculation
# and every self-check answer is logged in this browser session; at the end
# the student downloads a formatted report (.html and .docx). Nothing is
# sent anywhere; files are created only when the student asks.
# ---------------------------------------------------------------------------
import io, re, base64, zipfile, datetime, html as _html

_LATEX_FIX = [(r"\\textcolor\{[^}]*\}", ""), (r"\\color\{[^}]*\}", ""),
              (r"\\text\{", r"\\mathrm{"), (r"\\boxed\{", "{"), (r"\\quad", r"\ \ "), (r"\\%", r"\%"),
              (r"\\vert", "|"), (r"\\mathbf\{", r"\\mathbf{")]


def latex_png(tex, fontsize=12, dpi=170):
    """Render one LaTeX line to PNG bytes with matplotlib mathtext (colours stripped)."""
    import matplotlib
    import matplotlib.pyplot as plt
    t = tex
    for a, b in _LATEX_FIX:
        t = re.sub(a, b, t)
    t = t.replace(r"\,", r"\ ")
    fig = plt.figure(figsize=(0.01, 0.01))
    try:
        fig.text(0, 0, "$%s$" % t, fontsize=fontsize)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", pad_inches=0.04, transparent=False,
                    facecolor="white")
        return buf.getvalue()
    except Exception:
        return None
    finally:
        plt.close(fig)


def _split_math(s):
    """Split '... $a$ ... $b$' into [('text', ..), ('math', ..)]."""
    out, pos = [], 0
    for m in re.finditer(r"\$([^$]+)\$", s):
        if m.start() > pos:
            out.append(("text", s[pos:m.start()]))
        out.append(("math", m.group(1)))
        pos = m.end()
    if pos < len(s):
        out.append(("text", s[pos:]))
    return out


class LabReport:
    def __init__(self, course, session, title):
        self.course, self.session, self.title = course, session, title
        self.entries = []          # dicts: kind, section, title, ...
        self.section = ""
        self.student = ""
        self.started = datetime.datetime.now()

    # ---- collection
    def set_section(self, label):
        self.section = label

    def add_figure(self, fig, title=None):
        buf = io.BytesIO()
        try:
            fig.savefig(buf, format="png", dpi=110, bbox_inches="tight", facecolor="white")
        except Exception:
            return
        t = title
        if not t:
            st = getattr(fig, "_suptitle", None)
            t = st.get_text() if st is not None else ""
            if not t:
                for ax in fig.axes:
                    tt = ax.get_title() or ax.get_title(loc="left") or ax.get_title(loc="right")
                    if tt:
                        t = tt; break
            if not t and fig.texts:
                t = fig.texts[0].get_text()
        self.entries.append(dict(kind="figure", section=self.section, title=t or "Figure", png=buf.getvalue()))

    def add_calc(self, card, sol):
        self.entries.append(dict(kind="calc", section=self.section, title=card.title, card_id=card.id,
                                 case=card.md_case(sol.vals), rows=sol.plain_lines(),
                                 result=sol.result, unit=sol.unit, meaning=sol.meaning))

    def add_answer(self, qid, stem, choice, correct, feedback=""):
        self.entries.append(dict(kind="answer", section=self.section, qid=qid, stem=stem, choice=choice,
                                 correct=correct, feedback=feedback))

    def add_note(self, title, text):
        self.entries.append(dict(kind="note", section=self.section, title=title, text=text))

    def counts(self):
        c = {}
        for e in self.entries:
            c[e["kind"]] = c.get(e["kind"], 0) + 1
        return c

    # ---- HTML
    def to_html(self, summary_md=""):
        css = ("body{font-family:Georgia,serif;max-width:860px;margin:2em auto;color:#1f2430;line-height:1.5}"
               "h1{font-size:1.6em}h2{border-bottom:2px solid #1d6fb8;padding-bottom:4px;margin-top:1.6em}"
               "table{border-collapse:collapse;width:100%;margin:.6em 0}td,th{border:1px solid #cbd5e1;padding:6px;"
               "vertical-align:top;font-size:.95em}th{background:#eef4fb}.fig{text-align:center;margin:1em 0}"
               ".fig img{max-width:100%}.cap{font-size:.9em;color:#4b5563}.ok{color:#15803d}.no{color:#b91c1c}"
               ".meta{color:#4b5563}")
        h = ['<!doctype html><html><head><meta charset="utf-8"><title>%s</title><style>%s</style>'
             '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.9/MathJax.js?config=TeX-AMS_HTML"></script>'
             '<script type="text/x-mathjax-config">MathJax.Hub.Config({tex2jax:{inlineMath:[["$","$"]]}});</script>'
             '</head><body>' % (_html.escape(self.title), css)]
        h.append("<h1>%s · %s</h1><p class='meta'>%s<br>Student: %s<br>Generated %s</p>" % (
            _html.escape(self.course), _html.escape(self.title), _html.escape(self.session),
            _html.escape(self.student or "________________"), datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
        sec = None
        nfig = ncalc = 0
        for e in self.entries:
            if e["section"] != sec:
                sec = e["section"]
                h.append("<h2>%s</h2>" % _html.escape(sec or "Work"))
            if e["kind"] == "figure":
                nfig += 1
                h.append("<div class='fig'><img src='data:image/png;base64,%s'><div class='cap'>Figure %d. %s</div></div>"
                         % (base64.b64encode(e["png"]).decode(), nfig, _html.escape(e["title"])))
            elif e["kind"] == "calc":
                ncalc += 1
                h.append("<h3>Calculation %d. %s</h3><p>%s</p><table><tr><th>Step</th><th>Work</th></tr>" % (
                    ncalc, _html.escape(e["title"]), e["case"].replace("**2. Practice problem (case)**", "").strip()))
                for k, (lab, s) in enumerate(e["rows"], 1):
                    h.append("<tr><td>%d. %s</td><td>%s</td></tr>" % (k, _html.escape(lab), s))
                h.append("</table>")
            elif e["kind"] == "answer":
                h.append("<p><b>%s.</b> %s<br>Answer: %s <span class='%s'>(%s)</span></p>" % (
                    _html.escape(e["qid"]), _html.escape(e["stem"]), _html.escape(e["choice"]),
                    "ok" if e["correct"] else "no", "correct" if e["correct"] else "review"))
            elif e["kind"] == "note":
                h.append("<h3>%s</h3><p>%s</p>" % (_html.escape(e["title"]), _html.escape(e["text"])))
        if summary_md:
            h.append("<h2>Summary</h2><pre>%s</pre>" % _html.escape(summary_md))
        h.append("</body></html>")
        return "\n".join(h)

    # ---- DOCX (minimal WordprocessingML writer; equations as images)
    def to_docx(self):
        D = _Docx()
        D.heading("%s · %s" % (self.course, self.title), 1)
        D.para("%s    Student: %s    Generated %s" % (self.session, self.student or "________________",
                                                      datetime.datetime.now().strftime("%Y-%m-%d %H:%M")), size=18)
        sec = None
        nfig = ncalc = 0
        for e in self.entries:
            if e["section"] != sec:
                sec = e["section"]
                D.heading(sec or "Work", 2)
            if e["kind"] == "figure":
                nfig += 1
                D.image(e["png"], width_in=6.0)
                D.para("Figure %d. %s" % (nfig, e["title"]), size=18, italic=True)
            elif e["kind"] == "calc":
                ncalc += 1
                D.heading("Calculation %d. %s" % (ncalc, e["title"]), 3)
                D.rich(e["case"].replace("**2. Practice problem (case)**", "").strip())
                rows = []
                for k, (lab, s) in enumerate(e["rows"], 1):
                    rows.append(("%d. %s" % (k, lab), s))
                D.table(rows)
            elif e["kind"] == "answer":
                D.para("%s. %s — answer %s (%s)" % (e["qid"], e["stem"], e["choice"],
                                                     "correct" if e["correct"] else "review"), size=20)
            elif e["kind"] == "note":
                D.heading(e["title"], 3)
                D.para(e["text"])
        return D.build()

    # ---- delivery
    def download(self, basename=None):
        base = basename or ("%s_%s_lab_report" % (self.course.replace(" ", ""), self.session.split()[0]))
        html_bytes = self.to_html().encode("utf-8")
        docx_bytes = self.to_docx()
        with open(base + ".html", "wb") as f:
            f.write(html_bytes)
        with open(base + ".docx", "wb") as f:
            f.write(docx_bytes)
        try:
            from google.colab import files
            files.download(base + ".docx")
            files.download(base + ".html")
            return "Downloading %s.docx and %s.html" % (base, base)
        except Exception:
            return "Saved %s.docx and %s.html in the notebook's folder" % (base, base)


# ---------------------------------------------------------------------------
_W = 'xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main" ' \
     'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" ' \
     'xmlns:wp="http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing" ' \
     'xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" ' \
     'xmlns:pic="http://schemas.openxmlformats.org/drawingml/2006/picture"'


def _x(s):
    return _html.escape(s, quote=False)


class _Docx:
    def __init__(self):
        self.body, self.media = [], []

    def _run(self, text, bold=False, italic=False, size=None, color=None):
        rpr = ""
        if bold: rpr += "<w:b/>"
        if italic: rpr += "<w:i/>"
        if color: rpr += '<w:color w:val="%s"/>' % color.lstrip("#")
        if size: rpr += '<w:sz w:val="%d"/>' % size
        return '<w:r>%s<w:t xml:space="preserve">%s</w:t></w:r>' % ("<w:rPr>%s</w:rPr>" % rpr if rpr else "", _x(text))

    def heading(self, text, level=1):
        size = {1: 36, 2: 28, 3: 24}.get(level, 22)
        self.body.append('<w:p><w:pPr><w:spacing w:before="240" w:after="80"/></w:pPr>%s</w:p>'
                         % self._run(text, bold=True, size=size, color="1d3557" if level < 3 else None))

    def para(self, text, size=22, italic=False):
        self.body.append("<w:p>%s</w:p>" % self._run(text, size=size, italic=italic))

    def _img_run(self, png, width_in):
        from PIL import Image
        im = Image.open(io.BytesIO(png))
        w, h = im.size
        cx = int(width_in * 914400)
        cy = int(cx * h / w)
        self.media.append(png)
        rid = "rIdImg%d" % len(self.media)
        n = len(self.media)
        return ('<w:r><w:drawing><wp:inline><wp:extent cx="%d" cy="%d"/><wp:docPr id="%d" name="img%d"/>'
                '<a:graphic><a:graphicData uri="http://schemas.openxmlformats.org/drawingml/2006/picture">'
                '<pic:pic><pic:nvPicPr><pic:cNvPr id="%d" name="img%d.png"/><pic:cNvPicPr/></pic:nvPicPr>'
                '<pic:blipFill><a:blip r:embed="%s"/><a:stretch><a:fillRect/></a:stretch></pic:blipFill>'
                '<pic:spPr><a:xfrm><a:off x="0" y="0"/><a:ext cx="%d" cy="%d"/></a:xfrm>'
                '<a:prstGeom prst="rect"><a:avLst/></a:prstGeom></pic:spPr></pic:pic></a:graphicData></a:graphic>'
                '</wp:inline></w:drawing></w:r>' % (cx, cy, n, n, n, n, rid, cx, cy))

    def image(self, png, width_in=6.0):
        self.body.append('<w:p><w:pPr><w:jc w:val="center"/></w:pPr>%s</w:p>' % self._img_run(png, width_in))

    def _rich_runs(self, s, size=22):
        runs = []
        for kind, part in _split_math(s):
            if kind == "text":
                txt = re.sub(r"\*\*(.*?)\*\*", r"\1", part)
                runs.append(self._run(txt, size=size))
            else:
                png = latex_png(part, fontsize=11)
                if png:
                    from PIL import Image
                    w, h = Image.open(io.BytesIO(png)).size
                    runs.append(self._img_run(png, min(6.0, w / 170.0)))
                else:
                    runs.append(self._run(part, size=size, italic=True))
        return "".join(runs)

    def rich(self, s):
        self.body.append("<w:p>%s</w:p>" % self._rich_runs(s))

    def table(self, rows):
        trs = []
        for a, b in rows:
            trs.append('<w:tr><w:tc><w:tcPr><w:tcW w:w="2400" w:type="dxa"/></w:tcPr><w:p>%s</w:p></w:tc>'
                       '<w:tc><w:tcPr><w:tcW w:w="7000" w:type="dxa"/></w:tcPr><w:p>%s</w:p></w:tc></w:tr>'
                       % (self._run(a, bold=True, size=20), self._rich_runs(b, size=20)))
        borders = "".join('<w:%s w:val="single" w:sz="4" w:space="0" w:color="CBD5E1"/>' % e
                          for e in ("top", "left", "bottom", "right", "insideH", "insideV"))
        self.body.append('<w:tbl><w:tblPr><w:tblW w:w="9400" w:type="dxa"/><w:tblBorders>%s</w:tblBorders>'
                         '</w:tblPr><w:tblGrid><w:gridCol w:w="2400"/><w:gridCol w:w="7000"/></w:tblGrid>%s</w:tbl>'
                         '<w:p/>' % (borders, "".join(trs)))

    def build(self):
        doc = ('<?xml version="1.0" encoding="UTF-8" standalone="yes"?><w:document %s><w:body>%s'
               '<w:sectPr><w:pgSz w:w="12240" w:h="15840"/><w:pgMar w:top="1080" w:right="1080" w:bottom="1080" '
               'w:left="1080" w:header="720" w:footer="720" w:gutter="0"/></w:sectPr></w:body></w:document>'
               % (_W, "".join(self.body)))
        rels = ['<Relationship Id="rIdImg%d" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/image" '
                'Target="media/image%d.png"/>' % (k, k) for k in range(1, len(self.media) + 1)]
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
            z.writestr("[Content_Types].xml",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
                       '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
                       '<Default Extension="xml" ContentType="application/xml"/>'
                       '<Default Extension="png" ContentType="image/png"/>'
                       '<Override PartName="/word/document.xml" '
                       'ContentType="application/vnd.openxmlformats-officedocument.wordprocessingml.document.main+xml"/>'
                       '</Types>')
            z.writestr("_rels/.rels",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
                       '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" '
                       'Target="word/document.xml"/></Relationships>')
            z.writestr("word/_rels/document.xml.rels",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">%s'
                       '</Relationships>' % "".join(rels))
            z.writestr("word/document.xml", doc)
            for k, png in enumerate(self.media, 1):
                z.writestr("word/media/image%d.png" % k, png)
        return buf.getvalue()


# ---------------------------------------------------------------------------
# Pathway figure kit (BioRender-inspired, matplotlib only).
# Muted palette, rounded compartments, lipid-bilayer membranes, short labels,
# arrows = conversion/activation, T-bars = inhibition, dashed = indirect,
# hexagon badges = drugs. Coordinates are in a 0-100 x 0-60 canvas.
# ---------------------------------------------------------------------------
import math
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, RegularPolygon, FancyArrowPatch, Ellipse, Rectangle
from matplotlib.lines import Line2D

PW = dict(teal="#2a9d8f", blue="#3a6ea5", coral="#e76f51", gold="#e9b44c", green="#5a9e6f",
          purple="#7b6fb0", grey="#6b7280", ink="#1f2937", paper="#ffffff",
          cyto="#f3f8fb", mito="#fdf3ea", nucleus="#f1eef8", lumen="#eef7f1", er="#fbf1f4",
          blood="#fdeeee", extracell="#f7f7f2")
KIND = {  # node fill, edge, text colour, shape
    "metabolite": (PW["paper"], PW["blue"], PW["ink"], "round"),
    "enzyme": ("#e3f1ef", PW["teal"], PW["ink"], "pill"),
    "transporter": ("#fbe9e3", PW["coral"], PW["ink"], "pill"),
    "receptor": ("#efeaf7", PW["purple"], PW["ink"], "pill"),
    "gene": ("#fdf5e1", PW["gold"], PW["ink"], "square"),
    "cofactor": (PW["paper"], PW["grey"], PW["grey"], "round"),
    "hormone": ("#e8f3ea", PW["green"], PW["ink"], "round"),
}


class Pathway:
    def __init__(self, title="", w=10.5, h=6.0, xlim=(0, 100), ylim=(0, 60)):
        self.fig, self.ax = plt.subplots(figsize=(w, h))
        self.ax.set_xlim(*xlim); self.ax.set_ylim(*ylim); self.ax.set_aspect("equal"); self.ax.axis("off")
        self.fig.patch.set_facecolor("white")
        if title:
            self.ax.set_title(title, fontsize=13, color=PW["ink"], loc="left", fontweight="bold", pad=10)
        self.nodes = {}
        self._legend = set()

    # ---- containers
    def compartment(self, x, y, w, h, label="", kind="cyto", membrane=True, label_pos="tl"):
        fill = PW.get(kind, kind)
        self.ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=%.1f" % min(4, h / 4),
                                         fc=fill, ec="#c9d3dd" if not membrane else "#d9a441", lw=0.8 if not membrane else 3.2,
                                         zorder=0))
        if membrane:   # inner line gives a bilayer look
            self.ax.add_patch(FancyBboxPatch((x + 0.6, y + 0.6), w - 1.2, h - 1.2,
                                             boxstyle="round,pad=0,rounding_size=%.1f" % max(0.5, min(4, h / 4) - 0.6),
                                             fc="none", ec="#f2d48f", lw=1.4, zorder=0))
        if label:
            tx, ha = (x + 1.6, "left") if "l" in label_pos else (x + w - 1.6, "right")
            ty, va = (y + h - 1.4, "top") if "t" in label_pos else (y + 1.4, "bottom")
            self.ax.text(tx, ty, label, ha=ha, va=va, fontsize=9, color=PW["grey"], style="italic", zorder=1)

    def membrane(self, x0, y0, x1, y1, label=""):
        """Straight lipid bilayer (two rows of heads) between two points."""
        n = max(4, int(math.hypot(x1 - x0, y1 - y0) / 1.3))
        dx, dy = (x1 - x0), (y1 - y0)
        L = math.hypot(dx, dy); nx, ny = -dy / L, dx / L
        for k in range(n + 1):
            px, py = x0 + dx * k / n, y0 + dy * k / n
            for s in (-1, 1):
                self.ax.add_patch(Circle((px + s * 0.9 * nx, py + s * 0.9 * ny), 0.42, fc="#e9b44c", ec="#c9902a", lw=0.4, zorder=1))
            self.ax.plot([px - 0.6 * nx, px + 0.6 * nx], [py - 0.6 * ny, py + 0.6 * ny], color="#d8c08a", lw=0.6, zorder=0.9)
        if label:
            self.ax.text(x1, y1 + 1.2, label, fontsize=8, color=PW["grey"], ha="right", style="italic")

    # ---- nodes
    def node(self, key, x, y, label, kind="metabolite", w=None, fs=9.5):
        fc, ec, tc, shape = KIND[kind]
        w = w or max(7.0, 1.15 * len(max(label.split("\n"), key=len)) + 2.6)
        h = 3.6 + 2.2 * label.count("\n")
        rs = {"round": h / 2, "pill": h / 2, "square": 0.6}[shape]
        self.ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0,rounding_size=%.2f" % rs,
                                         fc=fc, ec=ec, lw=1.6 if kind != "metabolite" else 1.2, zorder=3))
        self.ax.text(x, y, label, ha="center", va="center", fontsize=fs, color=tc, zorder=4,
                     fontweight="bold" if kind in ("enzyme", "transporter", "receptor") else "normal",
                     style="italic" if kind == "gene" else "normal")
        self.nodes[key] = (x, y, w, h)
        self._legend.add(kind)
        return key

    def drug(self, key, x, y, label, r=2.6):
        self.ax.add_patch(RegularPolygon((x, y), 6, radius=r, orientation=math.pi / 6, fc=PW["coral"], ec="#a8432b",
                                         lw=1.2, zorder=5))
        self.ax.text(x, y - r - 0.9, label, ha="center", va="top", fontsize=8.6, color="#a8432b", fontweight="bold", zorder=5)
        self.ax.text(x, y, "Rx", ha="center", va="center", fontsize=7.5, color="white", fontweight="bold", zorder=6)
        self.nodes[key] = (x, y, 2 * r, 2 * r)
        self._legend.add("drug")
        return key

    def _edge_pt(self, key, toward):
        x, y, w, h = self.nodes[key]
        tx, ty = toward
        dx, dy = tx - x, ty - y
        if dx == 0 and dy == 0:
            return x, y
        sx = (w / 2 + 0.5) / abs(dx) if dx else 1e9
        sy = (h / 2 + 0.5) / abs(dy) if dy else 1e9
        s = min(sx, sy)
        return x + dx * s, y + dy * s

    # ---- edges
    def arrow(self, a, b, kind="convert", label="", rad=0.0, color=None, lw=1.7, label_side=1):
        pa = self.nodes[a][:2] if isinstance(a, str) else a
        pb = self.nodes[b][:2] if isinstance(b, str) else b
        p0 = self._edge_pt(a, pb) if isinstance(a, str) else pa
        p1 = self._edge_pt(b, pa) if isinstance(b, str) else pb
        col = color or {"convert": PW["ink"], "activate": PW["green"], "inhibit": PW["coral"],
                        "indirect": PW["grey"], "transport": PW["blue"]}[kind]
        ls = "--" if kind == "indirect" else "-"
        style = "-[" if kind == "inhibit" else "-|>"
        self.ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle=style if kind != "inhibit" else "-",
                                          mutation_scale=13, color=col, lw=lw, ls=ls,
                                          connectionstyle="arc3,rad=%.2f" % rad, zorder=2, shrinkA=0, shrinkB=0))
        if kind == "inhibit":   # T-bar
            dx, dy = p1[0] - p0[0], p1[1] - p0[1]
            L = math.hypot(dx, dy) or 1
            nx, ny = -dy / L * 1.5, dx / L * 1.5
            self.ax.plot([p1[0] - nx, p1[0] + nx], [p1[1] - ny, p1[1] + ny], color=col, lw=2.6, zorder=2,
                         solid_capstyle="round")
        if label:
            mx, my = (p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2
            dx, dy = p1[0] - p0[0], p1[1] - p0[1]
            L = math.hypot(dx, dy) or 1
            ox, oy = -dy / L * 1.6 * label_side, dx / L * 1.6 * label_side
            if rad:
                ox += -dy * rad * 0.5; oy += dx * rad * 0.5
            self.ax.text(mx + ox, my + oy, label, fontsize=8.2, color=col, ha="center", va="center", zorder=6,
                         bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
        self._legend.add(kind)

    def catalyze(self, enzyme, a, b, label_side=1):
        """Metabolite a -> b with enzyme drawn beside the arrow and a short dashed tie to its midpoint."""
        self.arrow(a, b, "convert")
        xa, ya = self.nodes[a][:2]; xb, yb = self.nodes[b][:2]
        xe, ye = self.nodes[enzyme][:2]
        self.ax.add_patch(FancyArrowPatch((xe, ye), ((xa + xb) / 2, (ya + yb) / 2), arrowstyle="-", color=PW["teal"],
                                          lw=1.0, ls=(0, (2, 2)), zorder=1, shrinkA=10, shrinkB=2))

    def note(self, x, y, text, fs=8.4, color=None, ha="left"):
        import textwrap
        xmax = self.ax.get_xlim()[1]
        room = (xmax - x) if ha == "left" else (x if ha == "right" else 2 * min(x, xmax - x))
        width = max(12, int(room * 1.55 * 8.4 / fs))
        text = "\n".join(textwrap.fill(line, width) for line in text.split("\n"))
        self.ax.text(x, y, text, fontsize=fs, color=color or PW["grey"], ha=ha, va="top", zorder=6)

    def legend(self, loc="below"):
        items = []
        names = {"convert": ("conversion", PW["ink"], "-"), "activate": ("activates", PW["green"], "-"),
                 "inhibit": ("inhibits (T-bar)", PW["coral"], "-"), "indirect": ("indirect / multistep", PW["grey"], "--"),
                 "transport": ("transport", PW["blue"], "-")}
        for k in ("convert", "activate", "inhibit", "indirect", "transport"):
            if k in self._legend:
                n, c, ls = names[k]
                items.append(Line2D([0], [0], color=c, lw=2, ls=ls, label=n))
        for k in ("enzyme", "transporter", "receptor", "gene", "hormone"):
            if k in self._legend:
                fc, ec, _, _ = KIND[k]
                items.append(Rectangle((0, 0), 1, 1, fc=fc, ec=ec, lw=1.4, label=k))
        if "drug" in self._legend:
            items.append(Line2D([0], [0], marker="h", color="w", markerfacecolor=PW["coral"], markersize=11, label="drug"))
        if items and loc == "below":
            self.ax.legend(handles=items, loc="upper center", bbox_to_anchor=(0.5, -0.01), fontsize=7.8,
                           frameon=False, ncol=min(6, len(items)))
        elif items:
            self.ax.legend(handles=items, loc=loc, fontsize=7.8, frameon=True, framealpha=0.95, edgecolor="#d1d5db",
                           ncol=2 if len(items) > 4 else 1)

    def show(self, report=None):
        self.fig.tight_layout()
        plt.show()
        return self.fig


def demo_orlistat():
    """Example: dietary TAG digestion, absorption and the orlistat block."""
    P = Pathway("Fat digestion in the small-intestine lumen and the orlistat block", 10.5, 6.2)
    P.compartment(2, 30, 62, 27, "intestinal lumen", "lumen", membrane=False)
    P.compartment(66, 4, 32, 53, "enterocyte", "cyto")
    P.compartment(2, 3, 62, 24, "pancreas → duodenum", "extracell", membrane=False, label_pos="bl")
    P.node("tag", 10, 48, "TAG", "metabolite")
    P.node("bile", 10, 38, "bile salts", "cofactor")
    P.node("mic", 32, 48, "emulsion\ndroplet", "metabolite")
    P.node("mag", 48, 48, "2-MAG +\n2 FFA", "metabolite")
    P.node("pl", 32, 14, "pancreatic\nlipase", "enzyme")
    P.node("col", 50, 14, "colipase", "enzyme")
    P.node("cd36", 66, 36, "FATP4 /\nCD36", "transporter")
    P.node("rtag", 82, 34, "TAG", "metabolite")
    P.node("cm", 82, 16, "chylomicron", "metabolite")
    P.drug("orl", 18, 22, "orlistat")
    P.arrow("tag", "mic", "convert")
    P.arrow("bile", "mic", "activate", "emulsifies", rad=-0.2, label_side=-1)
    P.arrow("mic", "mag", "convert", "lipolysis")
    P.arrow("pl", "mic", "activate", rad=0.15)
    P.arrow("col", "pl", "activate", "anchors", label_side=-1)
    P.arrow("orl", "pl", "inhibit", "covalent Ser", label_side=1)
    P.arrow("mag", "cd36", "transport")
    P.arrow("cd36", "rtag", "indirect", "re-esterified", label_side=-1)
    P.arrow("rtag", "cm", "convert", "+ apoB-48")
    P.note(3, 2.2, "Unabsorbed TAG passes to stool (steatorrhea): measured as fecal fat.", fs=8)
    P.legend()
    return P.show()


import json as _json
CALC_LAYOUTS = _json.loads('{"HP-15C": {"sources": ["https://finseth.com/hpdata/hp15c.php", "https://www.manualslib.com/manual/558055/Hp-Hp-15c.html", "https://manualshive.com/en/manual/hp/15c/156479?page=10"], "shift_keys": {"f": {"color": "#E3A63A", "label_position": "above the key, printed on the faceplate (gold)"}, "g": {"color": "#5AA7D6", "label_position": "on the slanted front face of the key, below the main legend (blue)"}}, "grid": {"rows": 4, "cols": 10}, "keys": [{"row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "√x", "shift1": "A", "shift2": "x²", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "eˣ", "shift1": "B", "shift2": "LN", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "10ˣ", "shift1": "C", "shift2": "LOG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "yˣ", "shift1": "D", "shift2": "%", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "1/x", "shift1": "E", "shift2": "Δ%", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue legend is Δ% (percent change); Finseth renders it \'%CHG\'."}, {"row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "CHS", "shift1": "MATRIX", "shift2": "ABS", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 6, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "FIX", "shift2": "DEG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 7, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "SCI", "shift2": "RAD", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 8, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "ENG", "shift2": "GRD", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 9, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "SOLVE", "shift2": "x≤y", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "SST", "shift1": "LBL", "shift2": "BST", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "GTO", "shift1": "HYP", "shift2": "HYP⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue legend is HYP⁻¹ (inverse hyperbolic); Finseth\'s text shows it as a second \'HYP\' because the superscript is lost."}, {"row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "SIN", "shift1": "DIM", "shift2": "SIN⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "COS", "shift1": "(i)", "shift2": "COS⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "TAN", "shift1": "I", "shift2": "TAN⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "EEX", "shift1": "RESULT", "shift2": "π", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 6, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "x≷", "shift2": "SF", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend is x≷ (exchange X with a register), often written x<>."}, {"row": 1, "col": 7, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "DSE", "shift2": "CF", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 8, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "ISG", "shift2": "F?", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 9, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "∫ᵧˣ", "shift2": "x=0", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend is the integral sign ∫ with limits y and x (∫ᵧˣ)."}, {"row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "R/S", "shift1": "PSE", "shift2": "P/R", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "GSB", "shift1": "Σ", "shift2": "RTN", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR Σ."}, {"row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "R↓", "shift1": "PRGM", "shift2": "R↑", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR PRGM."}, {"row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "x≷y", "shift1": "REG", "shift2": "RND", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR REG."}, {"row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "←", "shift1": "PREFIX", "shift2": "CLx", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR PREFIX."}, {"row": 2, "col": 5, "rowspan": 2, "colspan": 1, "main": "ENTER", "shift1": "RAN #", "shift2": "LSTx", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Tall ENTER: occupies column 5 of rows 2 and 3."}, {"row": 2, "col": 6, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "→R", "shift2": "→P", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 7, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "→H.MS", "shift2": "→H", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 8, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "→RAD", "shift2": "→DEG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 9, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "Re≷Im", "shift2": "TEST", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "ON", "shift1": "", "shift2": "", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "No shifted functions. Held with other keys at power-on for self-tests / decimal-point swap."}, {"row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "f", "shift1": "", "shift2": "", "alpha": "", "key_color": "#E3A63A", "text_color": "#1E1E1E", "uncertain": false, "note": "Gold prefix key; activates the gold legends printed above the keys."}, {"row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "g", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5AA7D6", "text_color": "#1E1E1E", "uncertain": false, "note": "Blue prefix key; activates the blue legends printed on the slanted front face of the keys."}, {"row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "STO", "shift1": "FRAC", "shift2": "INT", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "RCL", "shift1": "USER", "shift2": "MEM", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 6, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "x!", "shift2": "x̄", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 7, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "ŷ,r", "shift2": "s", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 8, "rowspan": 1, "colspan": 1, "main": "Σ+", "shift1": "L.R.", "shift2": "Σ−", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 9, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "Py,x", "shift2": "Cy,x", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}], "notes": "Landscape (Voyager) layout: 4 rows x 10 columns, 39 physical keys. ENTER is the tall key in column 5 spanning rows 2-3, so row 3 has no separate key at column 5. Bottom-left of row 3 holds ON, f (gold key) and g (blue key). Main legends are white on black keys; f-legends are gold on the faceplate above each key; g-legends are blue on the key\'s front face. A gold \'CLEAR\' bracket spans above GSB, R↓, x≷y and ← (CLEAR Σ, CLEAR PRGM, CLEAR REG, CLEAR PREFIX). The HP 15C Collector\'s Edition (2023) keeps the identical layout and legends. Hex colours are approximations of the printed/moulded colours, not official values."}, "HP 35s": {"sources": ["https://finseth.com/hpdata/hp35s.php", "https://stefanv.com/calculators/hp35s.html", "https://hpmuseum.org/forum/post-7043.html", "https://www.hpmuseum.org/forum/post-39775.html", "https://literature.hpcalc.org/official/hp35s-qsg-en.pdf", "https://www.manualslib.com/manual/257003/Hp-35s.html?page=18"], "shift_keys": {"left_shift": {"color": "#F2B33D", "label_position": "above the key, printed on the faceplate (yellow)"}, "right_shift": {"color": "#3D8BD4", "label_position": "on the bottom/front face of the key (blue)"}, "alpha": {"color": "#E0457B", "label_position": "small red/pink letter at the bottom-right corner of the key (used after STO/RCL/XEQ/GTO, no ALPHA key)"}}, "sections": [{"name": "top", "rows": 2, "cols": 4, "placement": "upper-left block"}, {"name": "cursor", "rows": 3, "cols": 3, "placement": "right of \'top\', 4 keys in a diamond (only (0,1),(1,0),(1,2),(2,1) used)"}, {"name": "upper", "rows": 2, "cols": 6, "placement": "full width below top/cursor"}, {"name": "lower", "rows": 4, "cols": 5, "placement": "full width, bottom"}], "grid": {"rows": 10, "cols": 6, "note": "overall; use \'sections\' for exact positions"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "R/S", "shift1": "FN=", "shift2": "PRGM", "alpha": "A", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "GTO", "shift1": "ISG", "shift2": "DSE", "alpha": "B", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "XEQ", "shift1": "RTN", "shift2": "LBL", "alpha": "C", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "MODE", "shift1": "x?y", "shift2": "x?0", "alpha": "D", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source). MODE is a menu key."}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "RCL", "shift1": "x≷", "shift2": "STO", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "STO is the blue (right-shifted) function of RCL - confirmed by Stefan Vorkoetter\'s review."}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "R↓", "shift1": "VIEW", "shift2": "R↑", "alpha": "E", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "x≷y", "shift1": "INPUT", "shift2": "PSE", "alpha": "F", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "i", "shift1": "ARG", "shift2": "θ", "alpha": "G", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "cursor", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "FLAGS", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "DISPLAY", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "CONST", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "MEM", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "upper", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "SIN", "shift1": "HYP", "shift2": "ASIN", "alpha": "H", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "COS", "shift1": "π", "shift2": "ACOS", "alpha": "I", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "TAN", "shift1": "INTG", "shift2": "ATAN", "alpha": "J", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "√x", "shift1": "ˣ√y", "shift2": "x²", "alpha": "K", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "yˣ", "shift1": "LOG", "shift2": "LN", "alpha": "L", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "1/x", "shift1": "10ˣ", "shift2": "eˣ", "alpha": "M", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 1, "col": 0, "rowspan": 1, "colspan": 2, "main": "ENTER", "shift1": "SHOW", "shift2": "LASTx", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Double-width ENTER spanning columns 0-1."}, {"section": "upper", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "+/−", "shift1": "=", "shift2": "ABS", "alpha": "N", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "E", "shift1": "←ENG", "shift2": "RND", "alpha": "O", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "E = EEX (enter exponent)."}, {"section": "upper", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "( )", "shift1": "ENG→", "shift2": "[ ]", "alpha": "P", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Parentheses key; blue shift gives vector brackets [ ] (Dieter, hpmuseum post-7043; Vorkoetter review)."}, {"section": "upper", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "←", "shift1": "UNDO", "shift2": "CLEAR", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Backspace. CLEAR is a menu."}, {"section": "lower", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "EQN", "shift1": "∫FN", "shift2": "SOLVE", "alpha": "Q", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Finseth encodes the yellow legend as \'\\\\GI\'; read as the integrate function ∫FN (pairs with FN= on R/S). Verify against a photo."}, {"section": "lower", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "→°F", "shift2": "→°C", "alpha": "R", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "HMS→", "shift2": "→HMS", "alpha": "S", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "→RAD", "shift2": "→DEG", "alpha": "T", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "%CHG", "shift2": "%", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀ (left shift)", "shift1": "", "shift2": "", "alpha": "", "key_color": "#F2B33D", "text_color": "#1E1E1E", "uncertain": false, "note": "Yellow/gold left-shift key (key cap printed with a bent arrow)."}, {"section": "lower", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "→lb", "shift2": "→kg", "alpha": "U", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "→MILE", "shift2": "→KM", "alpha": "V", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "→in", "shift2": "→cm", "alpha": "W", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "nCr", "shift2": "nPr", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "▶ (right shift)", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3D8BD4", "text_color": "#1E1E1E", "uncertain": false, "note": "Blue right-shift key (key cap printed with a bent arrow)."}, {"section": "lower", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "LOGIC", "shift2": "BASE", "alpha": "X", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "→gal", "shift2": "→l", "alpha": "Y", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "SEED", "shift2": "RAND", "alpha": "Z", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "L.R.", "shift2": "SUMS", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "C", "shift1": "OFF", "shift2": "ON", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Clear/cancel key; turns calculator on. \'ON\' is printed in blue on the bottom (front) edge; OFF in yellow above."}, {"section": "lower", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": ",", "shift2": "SPACE", "alpha": "(I)", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Yellow \',\' confirmed (hpmuseum post-7043: comma on yellow-shifted zero). Red \'(I)\' read from Finseth alpha line \'(i)\'; manual says indirect registers (I)/(J) are on specific keys but key glyphs are lost in text."}, {"section": "lower", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "/c", "shift2": "FDISP", "alpha": "(J)", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Yellow \'/c\' (max fraction denominator) confirmed by hpmuseum post-39775 (\'left shift /c\'). \'a b/c\' reminder printed in red below the key. Red \'(J)\' read from Finseth alpha line; exact red legend arrangement uncertain."}, {"section": "lower", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "Σ+", "shift1": "Σ−", "shift2": "!", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue \'!\' is factorial (x!)."}, {"section": "lower", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "x̄,ȳ", "shift2": "s,σ", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Both shifted legends are menus (x̄,ȳ means; s,σ standard deviations)."}], "notes": "43 keys counted here (8 top + 4 cursor + 11 upper + 20 lower). shift1 = yellow left-shift, shift2 = blue right-shift, alpha = red letter. The yellow shift key is in the first column of the 2nd row of the lower block (left of 4) and the blue shift key below it (left of 1); the C/ON key is bottom-left. R/S is top-left (not bottom-right as on classic HPs). ENTER is double width. 16 keys open menus. Finseth\'s text map is the primary positional source; HP\'s own QSG/manual confirm the colour scheme (yellow above, blue on key bottom, letters bottom-right, cursor keys yellow-only) but their key glyphs do not survive text extraction. Colours are approximations."}, "TI-30XS MultiView": {"sources": ["https://education.ti.com/html/eguides/scientifics/PDFs/EN/TI30XS-MultiView-Guide_EN.pdf", "https://education.ti.com/html/eguides/scientifics/TI-30XS-MultiView/EN/Content/EG_SmartView/M_KeyBoard/M_KeyBoard.HTML", "https://cdn.geogebra.org/resource/dckja6sx/Hxe3aChHaf42lER1/material-dckja6sx.pdf", "https://tech.aph.org/Orion%20TI-30XS%20Documents/xs_doc.htm", "https://www.rskey.org/~mwsebastian/reviews/ti30xsmv.htm", "https://education.ti.com/en/customer-support/knowledge-base/scientific-elem-calculators/product-usage/20309"], "shift_keys": {"2nd": {"color": "#3BAA4B", "label_position": "above the key (green text)"}}, "sections": [{"name": "top", "rows": 2, "cols": 3, "placement": "upper-left, under the display"}, {"name": "arrows", "rows": 3, "cols": 3, "placement": "4-way rocker in the top-right corner beside \'top\' (occupies the space of columns 3-4 of a 5-column grid)"}, {"name": "main", "rows": 7, "cols": 5, "placement": "full width below"}], "grid": {"rows": 9, "cols": 5, "note": "overall; \'top\' + \'arrows\' form the first two rows"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "2nd", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3BAA4B", "text_color": "#FFFFFF", "uncertain": false, "note": "Green 2nd key; 2nd legends are printed in green above keys."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "mode", "shift1": "quit", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "delete", "shift1": "insert", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "log", "shift1": "10ˣ", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "prb", "shift1": "angle", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Key position derived (see notes). 2nd legend (angle) from memory; guidebook confirms 2nd[angle] and 2nd[stat] exist but text extraction loses which key."}, {"section": "top", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "data", "shift1": "stat", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Key position derived (see notes). 2nd legend (stat) from memory; guidebook confirms 2nd[angle] and 2nd[stat] exist but text extraction loses which key."}, {"section": "arrows", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker."}, {"section": "arrows", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker. 2nd ◀ jumps to start of entry (no printed legend)"}, {"section": "arrows", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker. 2nd ▶ jumps to end of entry (no printed legend)"}, {"section": "arrows", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker."}, {"section": "main", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "eˣ", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "n/d", "shift1": "U n/d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "×10ⁿ", "shift1": "n/d◀▶U n/d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": "2nd n/d◀▶U n/d confirmed by GED reference sheet (2nd + this key toggles fraction/mixed)."}, {"section": "main", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "table", "shift1": "f◀▶d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": "2nd f◀▶d confirmed by GED reference sheet."}, {"section": "main", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "clear", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "π", "shift1": "hyp", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "K", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend K (constant) placed from memory; guidebook confirms 2nd[K] exists."}, {"section": "main", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "^", "shift1": "ˣ√", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Position inferred from TI-SmartView key-letter sequence (only remaining key for slot). 2nd legend unknown."}, {"section": "main", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "%", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend (%) placed from memory; TI KB 20309 confirms 2nd[%] and 2nd[▶%] exist but not which key."}, {"section": "main", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": "▶%", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend (▶%) placed from memory; TI KB 20309 confirms 2nd[%] and 2nd[▶%] exist but not which key."}, {"section": "main", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "√", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 0, "rowspan": 1, "colspan": 1, "main": "x y z t a b c", "shift1": "clear var", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 0, "rowspan": 1, "colspan": 1, "main": "sto→", "shift1": "recall", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 4, "rowspan": 1, "colspan": 1, "main": "◀▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Answer-toggle key (switches fraction/decimal/π display). Position directly above enter derived from SmartView key-letter sequence; 2nd legend unknown."}, {"section": "main", "row": 6, "col": 0, "rowspan": 1, "colspan": 1, "main": "on", "shift1": "off", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "reset", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": false, "note": "2nd reset confirmed by GED sheet (\'2nd 0 2\' = reset all)."}, {"section": "main", "row": 6, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 6, "col": 3, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "Ans", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd Ans from memory; Ans is a confirmed 2nd function."}, {"section": "main", "row": 6, "col": 4, "rowspan": 1, "colspan": 1, "main": "enter", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}], "notes": "45 keys including the 4 rocker positions (rskey.org review; APH guide: 41 keys in a 9x5 arrangement plus a 4-key directional pad top-right; ON bottom-left, 2nd top-left). No ALPHA key; variables come from the \'x y z t a b c\' key. POSITION METHOD: TI\'s own guidebooks render keys in a symbol font that does not survive text extraction, and the keypad figures are images. Positions were reconstructed from the TI-SmartView computer-key shortcuts (mode=[W] stated by TI; a GED reference sheet using the same shortcut letters gives q=2nd, w=mode, e=delete, r/u=log/ln, t=prb, y=data, i=n/d, p=×10ⁿ, a=table, s=clear, d=π, f/g/h=sin/cos/tan, j=^, l=x², c=toggle, n=enter). These letters run in keyboard order (q w e r t y u i [o skipped] p a s d f g h j k l z x c v b n) across the non-digit/non-operator keys row by row, which pins every key into a consistent 9x5 layout. Treat as strong inference, not a direct reading of a keypad image. Colours approximate (dark function keys, light number keys, green 2nd key; varies by case colour)."}, "TI-84 Plus CE": {"sources": ["https://education.ti.com/html/eguides/graphing/84CE-Online-Calculator/EN/Content/EG_GSGuide/M_Map/KB_Key_Map.HTML", "https://learn.cemetech.net/index.php/TI-BASIC:Key_Codes", "https://learn.cemetech.net/index.php/TI-BASIC:Getkey", "https://education.ti.com/en/product-resources/guidebooks/ti-84-plus-ce-python"], "shift_keys": {"2nd": {"color": "#3FA4DC", "label_position": "above the key, left side (blue)"}, "alpha": {"color": "#62B446", "label_position": "above the key, right side (green)"}}, "sections": [{"name": "top_row", "rows": 1, "cols": 5, "placement": "directly under the screen"}, {"name": "upper_left", "rows": 2, "cols": 3, "placement": "left 3 columns under top_row"}, {"name": "arrows", "rows": 3, "cols": 3, "placement": "right of upper_left (diamond; only (0,1),(1,0),(1,2),(2,1) used)"}, {"name": "main", "rows": 7, "cols": 5, "placement": "full width below"}], "grid": {"rows": 10, "cols": 5, "note": "getKey rows 1-10; use \'sections\' for exact positions"}, "keys": [{"section": "top_row", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "y=", "shift1": "stat plot", "shift2": "", "alpha": "f1", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": "Graph keys; 2nd legends blue, alpha (f1-f5 menu softkeys) green."}, {"section": "top_row", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "window", "shift1": "tblset", "shift2": "", "alpha": "f2", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "zoom", "shift1": "format", "shift2": "", "alpha": "f3", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "trace", "shift1": "calc", "shift2": "", "alpha": "f4", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "graph", "shift1": "table", "shift2": "", "alpha": "f5", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "2nd", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3FA4DC", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue 2nd key (getKey 21)."}, {"section": "upper_left", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "mode", "shift1": "quit", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "del", "shift1": "ins", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "alpha", "shift1": "A-lock", "shift2": "", "alpha": "", "key_color": "#62B446", "text_color": "#FFFFFF", "uncertain": false, "note": "Green alpha key (getKey 31)."}, {"section": "upper_left", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "X,T,θ,n", "shift1": "link", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "stat", "shift1": "list", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "arrows", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 25). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 24). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 26). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 34). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "main", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "math", "shift1": "test", "shift2": "", "alpha": "A", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "apps", "shift1": "angle", "shift2": "", "alpha": "B", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "prgm", "shift1": "draw", "shift2": "", "alpha": "C", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "vars", "shift1": "distr", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "clear", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "matrix", "shift2": "", "alpha": "D", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "E", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "F", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "G", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "^", "shift1": "π", "shift2": "", "alpha": "H", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "√", "shift2": "", "alpha": "I", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": ",", "shift1": "EE", "shift2": "", "alpha": "J", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "{", "shift2": "", "alpha": "K", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": "}", "shift2": "", "alpha": "L", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "e", "shift2": "", "alpha": "M", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "log", "shift1": "10ˣ", "shift2": "", "alpha": "N", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "u", "shift2": "", "alpha": "O", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "v", "shift2": "", "alpha": "P", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "w", "shift2": "", "alpha": "Q", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "[", "shift2": "", "alpha": "R", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 0, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "eˣ", "shift2": "", "alpha": "S", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "L4", "shift2": "", "alpha": "T", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "L5", "shift2": "", "alpha": "U", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "L6", "shift2": "", "alpha": "V", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "]", "shift2": "", "alpha": "W", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 0, "rowspan": 1, "colspan": 1, "main": "sto→", "shift1": "rcl", "shift2": "", "alpha": "X", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "L1", "shift2": "", "alpha": "Y", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "L2", "shift2": "", "alpha": "Z", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "L3", "shift2": "", "alpha": "θ", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "mem", "shift2": "", "alpha": "\\"", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 0, "rowspan": 1, "colspan": 1, "main": "on", "shift1": "off", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": "ON has no getKey code."}, {"section": "main", "row": 6, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "catalog", "shift2": "", "alpha": "␣", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": "Alpha legend is a space character (␣)."}, {"section": "main", "row": 6, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "i", "shift2": "", "alpha": ":", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 3, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "ans", "shift2": "", "alpha": "?", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 4, "rowspan": 1, "colspan": 1, "main": "enter", "shift1": "entry", "shift2": "", "alpha": "solve", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}], "notes": "50 keys (5 + 6 + 4 arrows + 35). Positions cross-checked two ways: TI\'s online-calculator key map lists keys in row order, and the getKey code grid (row*10+col: 11-15, 21-23 + arrows 24/25/26/34, 31-33, 41-45 ... 101-105) fixes rows/columns. Legend text from TI\'s standard keyboard (identical to TI-84 Plus except CE uses f1-f5 alpha legends on the top row). Key/body colours vary by case colour; hex values are approximate (dark function keys, light number keys, blue 2nd, green alpha, grey arrows)."}, "Casio fx-991EX": {"sources": ["https://www.casio.com/content/dam/casio/global/support/manuals/calculators/pdf/004-en/f/fx-570_991EX_EN.pdf", "https://de.ifixit.com/Document/DynfKy4jK4lucArw/QUICK-START-GUIDE-fx-991EX.pdf", "https://www.manualslib.com/manual/1260605/Casio-Fx-991ex.html", "https://pundit.pratt.duke.edu/wiki/Calculator_Tips/Classwiz", "https://hpmuseum.org/forum/thread-4442-page-3.html", "https://www.casio.com/content/dam/casio/global/support/manuals/calculators/pdf/004-en/f/fx-115ESPLUS_991ESPLUSC_EN.pdf", "https://support.casio.com/global/en/calc/manual/fx-570ESPLUS_991ESPLUS_en/before_using_calculator/getting_started.html"], "shift_keys": {"SHIFT": {"color": "#F2C230", "label_position": "above the key, left (yellow)"}, "ALPHA": {"color": "#E0453A", "label_position": "above the key, right (red)"}, "complex_mode": {"color": "#9B59B6", "label_position": "purple legends/brackets (Complex mode)"}, "base_n": {"color": "#3D7CC9", "label_position": "blue legends/brackets (Base-N: DEC HEX BIN OCT)"}}, "sections": [{"name": "top", "rows": 2, "cols": 6, "placement": "under the display; columns 2-3 are occupied by the cursor pad"}, {"name": "cursor", "rows": 3, "cols": 3, "placement": "round pad centred in \'top\' columns 2-3"}, {"name": "function", "rows": 3, "cols": 6, "placement": "6-column scientific block"}, {"name": "numeric", "rows": 4, "cols": 5, "placement": "5-column number block (larger keys)"}], "grid": {"rows": 9, "cols": 6, "note": "overall; use \'sections\'"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "SHIFT", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#F2C230", "uncertain": false, "note": "SHIFT key: dark key with yellow \'SHIFT\' text."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "ALPHA", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#E0453A", "uncertain": false, "note": "ALPHA key: dark key with red \'ALPHA\' text."}, {"section": "top", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "MENU", "shift1": "SETUP", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT MENU = SETUP (confirmed in manual and Duke ClassWiz notes)."}, {"section": "top", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "ON", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "ON is top-right on ClassWiz."}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "OPTN", "shift1": "QR", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT OPTN = QR (QR-code) per quick-start-guide keyboard OCR."}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "CALC", "shift1": "SOLVE", "shift2": "", "alpha": "=", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "ALPHA CALC gives the \'=\' used in equations."}, {"section": "top", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "∫▫□", "shift1": "d/dx▫", "shift2": "", "alpha": ":", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "ALPHA \':\' (multi-statement) assumed from ES PLUS, not confirmed for EX."}, {"section": "top", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "x", "shift1": "∑▫□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "Dedicated variable-x key. SHIFT legend Σ placed from memory; manual confirms Σ( exists but not its key."}, {"section": "cursor", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "function", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "■/□", "shift1": "■■/□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT = mixed-fraction template."}, {"section": "function", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "√■", "shift1": "∛■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "No ALPHA legend (hpmuseum forum)."}, {"section": "function", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "x³", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'DEC\' printed beside x³ (OCR of QSG keyboard)."}, {"section": "function", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "x^■", "shift1": "■√□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'HEX\'."}, {"section": "function", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "log▫□", "shift1": "10^■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'BIN\'."}, {"section": "function", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "e^■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'OCT\'."}, {"section": "function", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "log", "shift2": "", "alpha": "A", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT (−) = log( (log with base argument), from QSG keyboard OCR (\'log A\') and manual \'(log)\'. Differs from ES PLUS, where SHIFT (−) = ∠."}, {"section": "function", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "°\'\\"", "shift1": "FACT", "shift2": "", "alpha": "B", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "x!", "shift2": "", "alpha": "C", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "D", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "E", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "F", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "STO", "shift1": "RECALL", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "ENG", "shift1": "∠", "shift2": "", "alpha": "i", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ENG = ∠ per Duke ClassWiz notes (\'angle symbol (SHIFT ENG)\'). Imaginary unit i is printed in purple (Complex mode) on/above this key - exact placement (ALPHA vs purple bracket) uncertain."}, {"section": "function", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "Abs", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ( = Abs read from QSG keyboard OCR order \'RECALL Abs ...\'; single source."}, {"section": "function", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": ",", "shift2": "", "alpha": "x", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ) = \',\' confirmed (Casio UK QSG \'SHIFT ) (,)\'). ALPHA legend x assumed from ES PLUS."}, {"section": "function", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "S⇔D", "shift1": "a b/c⇔d/c", "shift2": "", "alpha": "y", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT legend assumed from ES PLUS; ALPHA y assumed."}, {"section": "function", "row": 2, "col": 5, "rowspan": 1, "colspan": 1, "main": "M+", "shift1": "M−", "shift2": "", "alpha": "M", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT M+ = M−, ALPHA M (QSG OCR \'M- M\')."}, {"section": "numeric", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "CONST", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "CONV", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "RESET", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "DEL", "shift1": "INS", "shift2": "", "alpha": "UNDO", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT DEL = INS; ALPHA DEL = UNDO read from OCR \'INS UNDO OFF\'."}, {"section": "numeric", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "AC", "shift1": "OFF", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "nPr", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "nPr/nCr exist (manual) and are SHIFT ×/÷ on ES PLUS, but the EX quick-start OCR shows no legend line above this row."}, {"section": "numeric", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "nCr", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "See ×."}, {"section": "numeric", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "Pol", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "Rec", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "Rnd", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "Ran#", "shift2": "", "alpha": "RanInt", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "×10ˣ", "shift1": "π", "shift2": "", "alpha": "e", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT ×10ˣ = π confirmed (Casio UK QSG). ALPHA = e."}, {"section": "numeric", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "Ans", "shift1": "%", "shift2": "", "alpha": "PreAns", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT Ans = % read from QSG OCR (π % ≈ sequence). ALPHA legend PreAns from memory; not found in manual."}, {"section": "numeric", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "=", "shift1": "≈", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT = gives decimal result (≈)."}], "notes": "50 keys counted (8 top + 4 cursor directions + 18 function + 20 numeric). Casio manuals and quick-start guides draw keys as font glyphs/images, so legends were assembled from: the EN user\'s guide list of alternate-function names, an OCR of the quick-start-guide keyboard figure (iFixit copy), Casio UK QSG text, and Duke\'s ClassWiz notes; remaining items carried over from the ES PLUS layout are flagged uncertain. Digit keys 1-6 have no shifted legends on the EX (Matrix/Vector/Stat etc. moved to MENU apps and OPTN). Colours approximate.", "fx-115ES_PLUS_differences": {"uncertain": true, "summary": "The US fx-115ES PLUS is the previous (ES PLUS) generation and its keyboard differs materially: MODE (SHIFT=SETUP) instead of MENU; no OPTN key and no dedicated x key; row under SHIFT/ALPHA is CALC (SHIFT SOLVE, ALPHA =), ∫ (SHIFT d/dx, ALPHA :) on the left and x⁻¹ (SHIFT x!), log▫□ (SHIFT Σ) on the right of the cursor pad; next row ■/□, √■, x², x^■, log, ln; then (−) (SHIFT ∠, ALPHA A), °\'\\" (SHIFT ←, ALPHA B), hyp (SHIFT Abs, ALPHA C), sin/cos/tan (ALPHA D/E/F); then RCL (SHIFT STO), ENG (SHIFT ←, purple i), ( (SHIFT %), ) (SHIFT \',\', ALPHA X), S⇔D (ALPHA Y), M+ (SHIFT M−, ALPHA M). Number block: SHIFT 1 STAT, 2 CMPLX, 3 DISTR, 4 MATRIX, 5 VECTOR, 7 CONST, 8 CONV, 9 CLR, × nPr, ÷ nCr, + Pol, − Rec, 0 Rnd, . Ran# (ALPHA RanInt#), ×10ˣ π (ALPHA e), Ans DRG▸, DEL INS, AC OFF. Manual confirms (STO), (CLR), (DRG▸), (←), (Abs), (FACT), (∠) as shifted functions; exact key placement is from memory of the ES PLUS keyboard and should be checked against a photo before use."}}}')

# ---------------------------------------------------------------------------
# Calculator faceplates. Each model's keys sit at the same row/column as on
# the physical device, with the shifted legends printed where the device
# prints them. Pressing a key sends the legend that is active (main, first
# shift, second shift or alpha) to the simulated engine; legends that the
# engine does not cover (programming, graphing menus, cursor editing) show
# "not simulated" instead of a wrong result.
# ---------------------------------------------------------------------------
import re as _re_ui

# legend (normalised) -> engine key.  Normalisation: strip spaces, unify minus
# signs and superscripts.  RPN and algebraic legends share most names.
_LEG = {
    # digits / entry
    **{d: d for d in "0123456789"}, ".": ".", "·": ".",
    "enter": "ENTER", "=": "EQ", "exe": "EXE", "enter↑": "ENTER", "e": "E",
    "chs": "CHS", "+/-": "CHS", "(-)": "NEG", "eex": "EEX", "×10ˣ": "EEX", "x10ˣ": "EEX", "ee": "EEX",
    "×10^x": "EEX", "x10^x": "EEX", "×10x": "EEX",
    "←": "BSP", "bsp": "BSP", "del": "DEL", "clx": "CLX", "clear": "CLEAR", "ac": "AC", "c": "CLEAR",
    "clstk": "CLSTK", "clrstk": "CLSTK", "clear stk": "CLSTK",
    # arithmetic
    "+": "ADD", "-": "SUB", "×": "MUL", "*": "MUL", "÷": "DIV", "/": "DIV",
    "(": "LP", ")": "RP", ",": "COMMA",
    # powers and roots
    "yˣ": "YX", "y^x": "YX", "^": "YX", "xʸ": "YX", "x^y": "YX", "x■": "YX", "x^□": "YX", "□^□": "YX",
    "√x": "SQRT", "√": "SQRT", "√(": "SQRT", "√■": "SQRT", "√□": "SQRT",
    "x²": "SQ", "x^2": "SQ", "x■²": "SQ", "□²": "SQ",
    "1/x": "INV", "x⁻¹": "INV", "x^-1": "INV", "□⁻¹": "INV",
    "ˣ√y": "XROOTY", "x√y": "XROOTY", "ˣ√": "XROOTY", "x√": "XROOTY", "ˣ√■": "XROOTY", "■√□": "XROOTY",
    "□√□": "XROOTY", "ˣ√(": "XROOTY", "∛": "CBRT", "∛■": "CBRT", "∛□": "CBRT", "3√": "CBRT",
    # exp / log
    "eˣ": "EXP", "e^x": "EXP", "e■": "EXP", "e^□": "EXP", "ln": "LN", "10ˣ": "10X", "10^x": "10X",
    "10■": "10X", "10^□": "10X", "log": "LOG",
    # trig
    "sin": "SIN", "cos": "COS", "tan": "TAN", "sin⁻¹": "ASIN", "cos⁻¹": "ACOS", "tan⁻¹": "ATAN",
    "sin^-1": "ASIN", "cos^-1": "ACOS", "tan^-1": "ATAN", "asin": "ASIN", "acos": "ACOS", "atan": "ATAN",
    "hyp": "HYP", "hyp⁻¹": "AHYP", "hyp^-1": "AHYP",
    "deg": "DEG", "rad": "RAD", "grd": "GRD", "→rad": "TORAD", "→deg": "TODEG", "→r": "TOR", "→p": "TOP",
    "→h.ms": "HMS", "→h": "HR", "→hms": "HMS", "→hr": "HR",
    # misc functions
    "%": "PCT", "δ%": "DPCT", "∆%": "DPCT", "abs": "ABS", "|x|": "ABS", "abs(": "ABS", "int": "INT", "ip": "INT",
    "frac": "FRAC", "fp": "FRAC", "x!": "FACT", "!": "FACT", "rnd": "RND",
    "py,x": "PERM", "cy,x": "COMB", "npr": "PERM", "ncr": "COMB",
    "π": "PI", "pi": "PI", "ans": "ANS",
    # stack
    "x≷y": "SWAP", "x<>y": "SWAP", "x⇄y": "SWAP", "r↓": "RDN", "r↑": "RUP", "lstx": "LASTX", "lastx": "LASTX",
    # statistics
    "σ+": "SIGMAP", "σ-": "SIGMAM", "x̄": "MEAN", "s": "SDEV", "l.r.": "LR", "ŷ,r": "YHATR", "clσ": "CLSIGMA",
    "clear σ": "CLSIGMA",
    # display / memory
    "fix": "FIX", "sci": "SCI", "eng": "ENG", "all": "ALL", "sto": "STO", "rcl": "RCL", "sto→": "STO",
    "sto▸": "STO", "sto►": "STO",
    # model-specific spellings (from calc_layouts.json)
    "E": "EEX", "×10ⁿ": "EEX", "x10ⁿ": "EEX", "%chg": "DPCT", "x̄,ȳ": "MEAN", "s,σ": "SDEV", "hms→": "HR", "→hms": "HMS",
    "→°f": "TOF", "→°c": "TOC", "→lb": "TOLB", "→kg": "TOKG", "→mile": "TOMILE", "→km": "TOKM", "→in": "TOIN",
    "→cm": "TOCM", "→gal": "TOGAL", "→l": "TOL", "x^■": "YX", "10^■": "10X", "e^■": "EXP", "x³": "CUBE",
    "■/□": "DIV", "entry": "ENTRY", "delete": "DEL", "ans": "ANS", "√": "SQRT", "ip": "INT", "fp": "FRAC",
}
_SHIFT_NAMES = {"◀(leftshift)": "shift1", "▶(rightshift)": "shift2","f": "shift1", "g": "shift2", "2nd": "shift1", "shift": "shift1", "alpha": "alpha",
                "◢": "shift1", "◣": "shift2", "left-shift": "shift1", "right-shift": "shift2",
                "leftshift": "shift1", "rightshift": "shift2", "⬏": "shift1", "⬎": "shift2"}
_ALG_VARS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZθxyzt")


def _norm(s):
    s = (s or "").strip()
    s = s.replace("−", "-").replace("–", "-").replace("Σ", "σ").replace("x̅", "x̄")
    s = _re_ui.sub(r"\s+", "", s)
    return s.lower() if len(s) > 1 else s


def legend_to_key(legend, rpn):
    if not legend:
        return None
    n = _norm(legend)
    if n in _LEG:
        k = _LEG[n]
        if rpn and k == "NEG":
            return "CHS"
        if not rpn and k == "CHS":
            return "NEG"
        if not rpn and k == "CLX":
            return "CLEAR"
        if rpn and k in ("CLEAR", "AC"):
            return "CLX"
        if rpn and k in ("CUBE", "ENTRY", "DEL", "ANS"):
            return {"DEL": "BSP"}.get(k)
        if not rpn and k.startswith("TO") and k not in ("TOP", "TOR"):
            return None
        if rpn and k == "EQ":
            return None
        if rpn and k == "LP":
            return None
        return k
    if not rpn and len(legend.strip()) == 1 and legend.strip() in _ALG_VARS:
        return "VAR_" + legend.strip()
    return None


class Faceplate:
    """UI-independent model of one calculator: key grid + shift state + engine."""

    RPN_MODELS = ("HP-15C", "HP 35s")

    def __init__(self, model, layout):
        self.model, self.layout = model, layout
        self.rpn = model in self.RPN_MODELS
        self.engine = RPNEngine(model) if self.rpn else AlgEngine(model, implicit_mult=True)
        if model.startswith("Casio"):
            self.engine.disp_mode = "ALL"
        self.state = None            # None | 'shift1' | 'shift2' | 'alpha'
        self.state_used = None
        self.log = []
        self.keys = layout["keys"]

    def shift_of(self, k):
        for lab in (k.get("main"), k.get("note")):
            n = _norm(lab)
            if n in _SHIFT_NAMES:
                return _SHIFT_NAMES[n]
        return None

    def active_legend(self, k):
        if self.state and k.get(self.state):
            return k[self.state]
        return k.get("main", "")

    def press(self, k):
        """Press key dict k. Returns the legend that acted."""
        sh = self.shift_of(k)
        if sh:
            self.state = None if self.state == sh else sh
            self.log.append(k.get("main"))
            return k.get("main")
        legend = self.active_legend(k)
        self.state_used = self.state if (self.state and k.get(self.state)) else None
        self.log.append(("%s " % {"shift1": self.layout.get("_s1", "2nd"), "shift2": self.layout.get("_s2", "g"),
                                  "alpha": "ALPHA"}[self.state] if self.state else "") + (k.get("main") or ""))
        self.state = None
        n = _norm(legend)
        if self.state_used == "alpha" and not self.rpn and len(legend.strip()) == 1 and legend.strip().isalpha():
            self.engine.press("VAR_" + legend.strip()); return legend
        if n in ("on", "on/c", "on/off", "c/on"):
            self.engine.__init__(self.model) if self.rpn else self.engine.__init__(self.model, True)
            return legend
        key = legend_to_key(legend, self.rpn)
        if key is None:
            self.engine.message = "%s: not simulated" % legend
            return legend
        self.engine.press(key)
        return legend

    def press_legend(self, legend, state=None):
        """Find the key whose legend (in the given state) matches and press it (used by tests)."""
        for k in self.keys:
            if (k.get(state) if state else k.get("main")) == legend:
                if state:
                    sk = next(x for x in self.keys if self.shift_of(x) == state)
                    self.press(sk)
                return self.press(k)
        raise KeyError("%s has no key %r (%s)" % (self.model, legend, state or "main"))

    def screen(self):
        e = self.engine
        if self.rpn:
            st = e.stack()
            return dict(main=e.display(), stack=[("T", st["T"]), ("Z", st["Z"]), ("Y", st["Y"])],
                        ann=" ".join(x for x in (self.state and {"shift1": "f", "shift2": "g"}[self.state],
                                                 e.angle if e.angle != "DEG" else "") if x))
        return dict(main=e.display(), entry=e.entry_text(), ann=" ".join(
            x for x in ({"shift1": "2nd", "shift2": "", "alpha": "ALPHA"}.get(self.state or "", ""), e.angle) if x))

    def coverage(self):
        """(mapped, total, unmapped legends) over every legend on the faceplate."""
        tot, ok, miss = 0, 0, []
        for k in self.keys:
            if self.shift_of(k):
                continue
            for st in ("main", "shift1", "shift2", "alpha"):
                lab = k.get(st)
                if not lab or (st == "alpha" and not self.rpn and False):
                    continue
                tot += 1
                if legend_to_key(lab, self.rpn) or _norm(lab) in ("on", "on/c", "on/off", "c/on"):
                    ok += 1
                else:
                    miss.append(lab)
        return ok, tot, miss


# ---------------------------------------------------------------------------
# How the JSON sections sit relative to each other on the physical device:
# each block is one grid; a section is placed at (row offset, col offset, col remap).
ARRANGE = {
    "HP 35s": [[("top", 0, 0), ("cursor", 0, 4)], [("upper", 0, 0)], [("lower", 0, 0)]],
    "TI-30XS MultiView": [[("top", 0, 0), ("arrows", 0, 3)], [("main", 0, 0)]],
    "TI-84 Plus CE": [[("top_row", 0, 0)], [("upper_left", 0, 0), ("arrows", 0, 3)], [("main", 0, 0)]],
    "Casio fx-991EX": [[("top", 0, 0, {4: 5, 5: 6}), ("cursor", 0, 2)], [("function", 0, 0)], [("numeric", 0, 0)]],
}


def blocks(model, layout):
    """[(rows, cols, [(key, row, col)])] — one grid per block, keys at device positions."""
    secs = {s["name"]: s for s in layout.get("sections", [])}
    if not secs:
        g = layout["grid"]
        return [(g["rows"], g["cols"], [(k, k["row"], k["col"]) for k in layout["keys"]])]
    arr = ARRANGE.get(model) or [[(n, 0, 0)] for n in secs]
    out = []
    for blk in arr:
        items, R, Cn = [], 0, 0
        for ent in blk:
            name, ro, co = ent[:3]
            remap = ent[3] if len(ent) > 3 else {}
            for k in layout["keys"]:
                if k.get("section") == name:
                    c = remap.get(k["col"], k["col"]) + co
                    r = k["row"] + ro
                    items.append((k, r, c))
                    R = max(R, r + k.get("rowspan", 1)); Cn = max(Cn, c + k.get("colspan", 1))
        out.append((R, Cn, items))
    return out


CALC_LAYOUTS = globals().get("CALC_LAYOUTS", {})
_PLATES = {}


def _key_html(k, model, layout):
    sk = layout.get("shift_keys", {})
    pick = lambda names, d: next((sk[n]["color"] for n in names if n in sk), d)
    s1c = pick(("f", "left_shift", "2nd", "SHIFT"), "#d4a017")
    s2c = pick(("g", "right_shift"), "#3b82c4")
    ac = pick(("alpha", "ALPHA"), "#16a34a")
    top = []
    if k.get("shift1"):
        top.append('<span style="color:%s">%s</span>' % (s1c, k["shift1"]))
    if k.get("alpha"):
        top.append('<span style="color:%s;float:right">%s</span>' % (ac, k["alpha"]))
    bottom = ('<span style="color:%s">%s</span>' % (s2c, k["shift2"])) if k.get("shift2") else ""
    return ("<div style='font:9px/11px sans-serif;height:11px;white-space:nowrap;overflow:hidden'>%s</div>" % " ".join(top),
            "<div style='font:9px/11px sans-serif;height:11px;white-space:nowrap;overflow:hidden;text-align:center'>%s</div>" % bottom)


def calculator_widget(model):
    import ipywidgets as W
    layout = CALC_LAYOUTS[model]
    fp = _PLATES.get(model) or Faceplate(model, layout)
    _PLATES[model] = fp
    lcd = W.HTML()
    keylog = W.HTML()

    def refresh():
        s = fp.screen()
        if fp.rpn:
            stack = "".join("<div style='color:#6b7280;font-size:11px'>%s: %s</div>" % (n, fmt_number(v, "ALL", 10))
                            for n, v in s["stack"])
            body = "%s<div style='font-size:22px;text-align:right'>%s</div>" % (stack, s["main"])
        else:
            body = ("<div style='font-size:15px;min-height:20px;word-break:break-all'>%s</div>"
                    "<div style='font-size:22px;text-align:right'>%s</div>" % (s["entry"] or "&nbsp;", s["main"] or "&nbsp;"))
        lcd.value = ("<div style='background:#c9d6b8;color:#111;font-family:monospace;padding:6px 10px;border-radius:6px;"
                     "width:300px;border:2px solid #555'><div style='font-size:10px;height:12px'>%s</div>%s</div>"
                     % (s["ann"], body))
        keylog.value = "<span style='font:11px monospace;color:#555'>Keys: %s</span>" % " ".join(
            str(x) for x in fp.log[-24:])

    boxes = []
    for R, Cn, items in blocks(model, layout):
        children = []
        for k, r, c in items:
            top, bot = _key_html(k, model, layout)
            b = W.Button(description=k.get("main", ""), layout=W.Layout(width="auto", height="30px", padding="0"),
                         tooltip="; ".join("%s: %s" % (a, k[a]) for a in ("shift1", "shift2", "alpha") if k.get(a)))
            b.style.button_color = k.get("key_color", "#2b2b2b")
            b.style.text_color = k.get("text_color", "#ffffff")
            b.style.font_weight = "bold"
            if k.get("uncertain"):
                b.layout.border = "2px dashed #f59e0b"
                b.tooltip = (b.tooltip + "; " if b.tooltip else "") + "position/legend not yet confirmed: " + (k.get("note") or "")

            def on(_b, k=k):
                fp.press(k)
                refresh()
            b.on_click(on)
            cell = W.VBox([W.HTML(top), b, W.HTML(bot)], layout=W.Layout(
                grid_row="%d / span %d" % (r + 1, k.get("rowspan", 1)),
                grid_column="%d / span %d" % (c + 1, k.get("colspan", 1)), margin="0"))
            children.append(cell)
        boxes.append(W.GridBox(children, layout=W.Layout(
            grid_template_columns="repeat(%d, 58px)" % Cn, grid_gap="2px 4px", width="max-content", margin="0 0 8px 0")))
    refresh()
    body = layout.get("body_color", "#3a3a3a")
    return W.VBox([lcd] + boxes + [keylog], layout=W.Layout(
        padding="10px", border="1px solid #999", width="max-content")), fp


def open_calculator(model=None):
    """Model selector + faceplate. Engines keep their state between questions."""
    import ipywidgets as W
    from IPython.display import display
    models = [m for m in ("HP-15C", "HP 35s", "TI-30XS MultiView", "TI-84 Plus CE", "Casio fx-991EX") if m in CALC_LAYOUTS]
    pick = W.Dropdown(options=models, value=model or globals().get("_LAST_CALC", models[0]), description="Calculator")
    holder = W.Output()
    help_ = W.HTML("<small>Shifted functions: press the coloured shift key, then the key under the legend. "
                   "Grey legends that are not simulated (programming, graphing, cursor editing) say so instead of "
                   "giving a wrong answer.</small>")

    def show(_=None):
        globals()["_LAST_CALC"] = pick.value
        holder.clear_output()
        with holder:
            w, _ = calculator_widget(pick.value)
            display(w)
    pick.observe(show, names="value")
    display(W.VBox([pick, help_, holder]))
    show()


# ---------------------------------------------------------------------------
# Notebook glue: lab-report hooks, per-cell section labels, calculation cards
# and the end-of-notebook report panel.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as _plt_glue

REPORT = None


def _start_report(course, session, title):
    global REPORT
    REPORT = LabReport(course, session, title)
    _orig_show = _plt_glue.show

    def _show(*a, **k):                      # every figure shown is copied into the report
        try:
            for num in _plt_glue.get_fignums():
                f = _plt_glue.figure(num)
                if not getattr(f, "_in_report", False):
                    REPORT.add_figure(f)
                    f._in_report = True
        except Exception:
            pass
        return _orig_show(*a, **k)
    _plt_glue.show = _show
    return REPORT


def _cell(label):
    """Called at the top of each hidden code cell: sets the report section."""
    if REPORT is not None:
        REPORT.set_section(label)


def card(card_id, calculator=True):
    """Interactive solver for one calculation card (+ the calculator panel)."""
    c = CARDS[card_id]
    opener = None
    if calculator:
        try:
            opener = lambda: open_calculator()      # defined by calc_ui
        except NameError:
            opener = None
    return solver_widget(c, REPORT, opener)


def report_panel():
    """Student name, reflection notes and download buttons for the lab report."""
    try:
        import ipywidgets as W
        from IPython.display import display, Markdown, clear_output
    except Exception:
        print(REPORT.counts() if REPORT else "no report")
        return
    name = W.Text(description="Your name", placeholder="as in Canvas", layout=W.Layout(width="420px"))
    notes = W.Textarea(description="Discussion", placeholder="What do your calculations show? What would you check next?",
                       layout=W.Layout(width="95%", height="120px"))
    btn = W.Button(description="Build and download report (.docx + .html)", button_style="primary", icon="download",
                   layout=W.Layout(width="360px"))
    out = W.Output()

    def go(_):
        with out:
            clear_output()
            REPORT.student = name.value.strip()
            REPORT.entries = [e for e in REPORT.entries if not (e["kind"] == "note" and e["title"] == "Discussion")]
            if notes.value.strip():
                REPORT.section = "Discussion"
                REPORT.add_note("Discussion", notes.value.strip())
            c = REPORT.counts()
            display(Markdown("Collected **%d** figures, **%d** calculations and **%d** self-check answers." % (
                c.get("figure", 0), c.get("calc", 0), c.get("answer", 0))))
            print(REPORT.download())
    btn.on_click(go)
    display(W.VBox([name, notes, btn, out]))


def _log_answers(gb):
    """Wrap the gradebook's answer() so first answers (not the readiness check) go into the report."""
    orig = gb.answer

    def _resp():
        r = getattr(gb, "resp", None)
        return r if r is not None else getattr(gb, "responses", {})

    def answer(qid, choice, *a, **k):
        first = qid not in _resp()
        r = orig(qid, choice, *a, **k)
        try:
            if first and qid in _resp() and REPORT is not None:
                it = gb.items[qid]
                if getattr(it, "section", "") != "precheck":
                    ch = _resp()[qid]
                    REPORT.add_answer(qid, getattr(it, "stem", qid), ch, ch == it.key)
        except Exception:
            pass
        return r
    gb.answer = answer



# ---------------------------------------------------------------------------
# PHS 312 L27 calculation cards: de novo design (ligand efficiency, LLE,
# fragment growth bookkeeping, fragment linking, rule of three, fold -> ΔΔG).
# ---------------------------------------------------------------------------

# ---- shared card helpers (same block in every PHS 312 L22-L27 card file) --------------
def _cc_f(x):
    """fmt() with a fix for 10^4 <= |x| < 10^5 (fmt prints these as 3.6e+04): 36{,}000."""
    if not isinstance(x, str) and 1e4 <= abs(x) < 1e5:
        return "{:,.0f}".format(x).replace(",", "{,}")
    return fmt(x)


def _cc_sci(x):
    """Power-of-ten form for very large/small numbers, plain otherwise."""
    if isinstance(x, str) or x == 0 or 1e-3 <= abs(x) < 1e5:
        return _cc_f(x)
    return sci(float(x))


def _cc_a(x):
    """Value for S.answer(): a preformatted string when fmt() would misprint it."""
    if not isinstance(x, str) and 1e4 <= abs(x) < 1e5:
        return _cc_f(x)
    return float(x)


def _cc_m(x):
    """A number for markdown text (inline math so powers of ten render)."""
    return "$%s$" % _cc_sci(x)


def _cc_int(n):
    """Integer with LaTeX thousands separators (20{,}736), or a power of ten when huge."""
    n = int(round(n))
    if abs(n) >= 1e9:
        return sci(float(n))
    return "{:,}".format(n).replace(",", "{,}")


def _cc_units(S, note=""):
    """Unit-check row: every prefix written as a power of ten, then the cancellation note."""
    bits, seen = [], set()
    for v in S.card.vars:
        if v.key not in S.vals or not v.unit:
            continue
        e, base = unit_factor(v.unit)
        if e == 0:
            continue
        val = S.vals[v.key]
        b = r"$%s%s = %s \times 10^{%d}%s = %s%s$" % (_cc_f(val), utex(v.unit), _cc_f(val), e, utex(base), _cc_sci(val * 10 ** e), utex(base))
        if b not in seen:
            seen.add(b); bits.append(b)
    txt = ("Prefixes written out: " + "; ".join(bits) + ". ") if bits else "No unit prefixes to expand. "
    S.rows.append(("Unit check", txt + note, "md"))
    return S


def _cc_R():
    return Var("R", r"R", "gas constant", "J/mol/K", "constant", "physical constant (8.314 J·mol⁻¹·K⁻¹ = 1.987 cal·mol⁻¹·K⁻¹)", 8.314, const=True)


def _cc_Rcal():
    return Var("R", r"R", "gas constant (calorie units)", "cal/mol/K", "constant", "physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹)", 1.987, const=True)


def _cc_T(T, src="case"):
    return Var("T", r"T", "absolute temperature", "K", "input", src, T)


def _cc_temp_step(S, V, key="T"):
    T = S.vals[key]
    if T <= 0:
        raise ValueError("temperature must be > 0 K")
    C = T - 273.15
    S.step("Temperature scales", r"%s = %s\,\mathrm{K} = %s\,^{\circ}\mathrm{C} = %s\,^{\circ}\mathrm{F}" % (
        V[key].tex(), _cc_f(T), _cc_f(round(C, 2)), _cc_f(round(1.8 * C + 32, 2))))
    return T


def _cc_RT_step(S, V, key="T", label="Thermal energy RT"):
    """RT in kJ/mol (R in J) or kcal/mol (R in cal); returns RT in the kilo unit."""
    T = S.vals[key]
    R = S.vals["R"]
    cal = S.card.V["R"].unit.startswith("cal")
    u = "cal" if cal else "J"
    RT = R * T
    S.step(label, r"%s %s = %s\,\mathrm{%s/(mol\cdot K)} \times %s\,\mathrm{K} = %s\,\mathrm{%s/mol} = %s\,\mathrm{k%s/mol}" % (
        V["R"].tex(), V[key].tex(), _cc_f(R), u, _cc_f(T), _cc_f(RT), u, _cc_f(RT / 1000), u))
    return RT / 1000


_CC_THERMO = ["T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32",
              "1 kcal = 4.184 kJ",
              "R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K",
              "RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K",
              "RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio"]
_CC_KJ_NOTE = ("R is in J/(mol·K) and the energies are in kJ/mol (kilo = 10³), so RT is converted from J/mol to kJ/mol "
               "before dividing; K cancels in R × T and kJ/mol cancels in ΔE/RT, leaving a pure number in the exponent.")
_CC_KCAL_NOTE = ("R is in cal/(mol·K) and the energies are in kcal/mol (kilo = 10³), so RT is converted from cal/mol to kcal/mol "
                 "before dividing; K cancels in R × T and kcal/mol cancels in ΔE/RT, leaving a pure number in the exponent.")
# ---------------------------------------------------------------------------------------------

def _cc_ln10RT(S, V, key="T"):
    """2.303RT in kcal/mol, with the temperature and RT steps shown."""
    _cc_temp_step(S, V, key)
    RT = _cc_RT_step(S, V, key)
    f = math.log(10) * RT
    S.step("Free energy per log unit", r"\ln 10 \times %s %s = %s \times %s = %s\,\mathrm{kcal/mol}" % (V["R"].tex(), V[key].tex(), _cc_f(math.log(10)), _cc_f(RT), _cc_f(f)))
    return f


# ---- ligand efficiency and lipophilic ligand efficiency ------------------------------------------
def _solve_le(card, d):
    S = Solution(card, d); V = card.V
    p, HA, lp = d["p"], d["HA"], d["logP"]
    if HA <= 0:
        raise ValueError("heavy atoms must be > 0")
    _cc_units(S.general().given(), _CC_KCAL_NOTE.replace("ΔE/RT", "ΔG/RT") + " pKd and log P are logarithms (no units), so LLE is unitless; "
              "LE is in kcal/mol per heavy atom.")
    f = _cc_ln10RT(S, V)
    dG = -f * p
    LE = -dG / HA
    lle = p - lp
    S.step("Kd from pKd", r"K_d = 10^{-%s} = %s\,\mathrm{M}" % (_cc_f(p), _cc_sci(10 ** -p)))
    S.step("Binding free energy", r"%s = -%s \times %s = %s\,\mathrm{kcal/mol} = %s\,\mathrm{kJ/mol}" % (V["dG"].tex(), _cc_f(f), _cc_f(p), _cc_f(dG), _cc_f(dG * 4.184)))
    S.step("Ligand efficiency", r"%s = \frac{-%s}{%s} = \frac{%s}{%s} = %s\,\mathrm{kcal/mol\ per\ heavy\ atom}" % (V["LE"].tex(), V["dG"].tex(), V["HA"].tex(), _cc_f(-dG), _cc_f(HA), _cc_f(LE)))
    S.step("Lipophilic ligand efficiency", r"%s = %s - %s = %s - %s = %s" % (V["LLE"].tex(), V["p"].tex(), V["logP"].tex(), _cc_f(p), _cc_f(lp), _cc_f(lle)))
    S.answer(LE, "kcal/mol", "LE = %s kcal/mol per heavy atom (%s 0.3) and LLE = %s (%s the ≥ 5 usually sought for a candidate). %s" % (
        _cc_f(LE), "≥" if LE >= 0.3 else "<", _cc_f(lle), "meets" if lle >= 5 else "below",
        "An efficient but weakly lipophilic-efficient compound gains potency mainly through greasiness; fix that before adding more atoms." if (LE >= 0.3 and lle < 3.5) else
        "Optimization should add potency without letting either number collapse."), label="Answer (LE)")
    return S


def _le_vars(p, HA, lp, src, Tsrc="25 °C convention"):
    return [Var("p", r"pK_d", "−log₁₀ of Kd (or pIC₅₀)", "", "input", src, p),
            Var("HA", r"N_{heavy}", "heavy (non-hydrogen) atoms", "", "input", "counted from the structure (case)", HA),
            Var("logP", r"\log P", "lipophilicity (clog P)", "", "input", "calculated property (case)", lp),
            _cc_T(298, Tsrc),
            Var("dG", r"\Delta G", "binding free energy", "kcal/mol", "intermediate", "calculated"),
            Var("LE", r"LE", "ligand efficiency", "kcal/mol", "result", "calculated"),
            Var("LLE", r"LLE", "lipophilic ligand efficiency", "", "intermediate", "calculated")]


_LE_EQ = r"{{dG}} = -2.303\,{{R}}{{T}}\,{{p}};\quad {{LE}} = \frac{-{{dG}}}{{{HA}}};\quad {{LLE}} = {{p}} - {{logP}}"
_LE_CONV = _CC_THERMO + ["pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM", "2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C"]

L27_LE = Card("l27_le_lle", "Ligand efficiency and lipophilic ligand efficiency", _LE_EQ,
    _le_vars(6.0, 25, 2.5, "binding assay (case, illustrative)"),
    "A de novo design for the thymidylate synthase cofactor site is made and binds with [[p]]; it has [[HA]] and [[logP]]. "
    "At [[T]], calculate ΔG, LE and LLE. Is it an efficient starting point?",
    _solve_le, constants=[_cc_Rcal()], conversions=_LE_CONV, result_name=r"LE", source="ligand efficiency (Hopkins); LLE (Leeson)")

L27_LE_F = Card("l27_le_fragment", "Ligand efficiency of a fragment versus a lead", _LE_EQ,
    _le_vars(3.5, 11, 1.2, "fragment screen (case)"),
    "A fragment ([[HA]], [[logP]]) binds with [[p]] at [[T]]. A lead from another project has 38 heavy atoms and pKd 7.0 "
    "(LE ≈ 0.25). Calculate the fragment's LE and LLE. Which is the better starting point for growing?",
    _solve_le, constants=[_cc_Rcal()], conversions=_LE_CONV, result_name=r"LE", source="ligand efficiency (Hopkins); LLE (Leeson)")

L27_LLE_P = Card("l27_lle_practice", "Practice: is potency being bought with lipophilicity?", _LE_EQ,
    _le_vars(8.3, 36, 5.4, "kinase assay pIC₅₀ (case, illustrative)"),
    "An oncology team's kinase lead has pIC₅₀ [[p]] (used as pKd), [[HA]] and clog P [[logP]]. At [[T]], calculate LE and "
    "LLE = pIC₅₀ − log P. What should the next round of design change?",
    _solve_le, constants=[_cc_Rcal()], conversions=_LE_CONV, result_name=r"LE", source="ligand efficiency (Hopkins); LLE (Leeson)")


# ---- fragment growth bookkeeping --------------------------------------------------------------------
def _solve_grow(card, d):
    S = Solution(card, d); V = card.V
    LE, h0, h1 = d["LE"], d["h0"], d["h1"]
    if h1 <= 0 or h0 <= 0:
        raise ValueError("heavy atoms must be > 0")
    _cc_units(S.general().given(), "LE (kcal/mol per heavy atom) × heavy atoms = kcal/mol; dividing by 2.303RT (kcal/mol per log unit) leaves a pure pKd.")
    f = _cc_ln10RT(S, V)
    g1 = LE * h1
    p1 = g1 / f
    p0 = LE * h0 / f
    per = LE / f
    S.step("Binding energy needed at the target size", r"-\Delta G = %s \times %s = %s \times %s = %s\,\mathrm{kcal/mol}" % (V["LE"].tex(), V["h1"].tex(), _cc_f(LE), _cc_f(h1), _cc_f(g1)))
    S.step("Target pKd", r"%s = \frac{%s}{%s} = %s" % (V["p1"].tex(), _cc_f(g1), _cc_f(f), _cc_f(p1)))
    S.step("Target Kd", r"K_d = 10^{-%s} = %s\,\mathrm{M} = %s\,\mathrm{nM}" % (_cc_f(p1), _cc_sci(10 ** -p1), _cc_f(1e9 * 10 ** -p1)))
    S.step("Starting pKd at the same LE", r"\frac{%s \times %s}{%s} = %s" % (_cc_f(LE), _cc_f(h0), _cc_f(f), _cc_f(p0)))
    S.step("Potency bought per added heavy atom", r"\frac{%s}{%s} = %s\ \mathrm{pK_d\ per\ atom};\quad %s \times %s = %s\ \mathrm{log\ units}" % (
        _cc_f(LE), _cc_f(f), _cc_f(per), _cc_f(h1 - h0), _cc_f(per), _cc_f(per * (h1 - h0))))
    S.answer(p1, "", "Growing from %s to %s heavy atoms at LE %s must reach pKd %s (Kd ≈ %s nM): %s log units for %s added atoms. If a grown analogue "
             "falls short, its LE has dropped and the added atoms are not earning their place." % (
                 _cc_f(h0), _cc_f(h1), _cc_f(LE), _cc_f(p1), _cc_f(1e9 * 10 ** -p1), _cc_f(per * (h1 - h0)), _cc_f(h1 - h0)), label="Answer (target pKd)")
    return S


def _grow_vars(LE, h0, h1, src):
    return [Var("LE", r"LE", "ligand efficiency to be maintained", "kcal/mol", "input", src, LE),
            Var("h0", r"N_{0}", "heavy atoms in the starting fragment", "", "input", "fragment (case)", h0),
            Var("h1", r"N_{1}", "heavy atoms after growing", "", "input", "design plan (case)", h1),
            _cc_T(298, "25 °C convention"),
            Var("p1", r"pK_{d,1}", "pKd the grown compound must reach", "", "result", "calculated")]


_GROW_EQ = r"{{p1}} = \frac{{{LE}}\,{{h1}}}{2.303\,{{R}}{{T}}};\quad \Delta pK_d = \frac{{{LE}}\left({{h1}} - {{h0}}\right)}{2.303\,{{R}}{{T}}}"

L27_GROW = Card("l27_grow", "Fragment growth bookkeeping: potency needed to keep LE", _GROW_EQ,
    _grow_vars(0.40, 12, 20, "fragment LE (case)"),
    "A [[h0]] fragment is to be grown to [[h1]] while keeping LE at [[LE]] ([[T]]). What pKd should the grown compound reach?",
    _solve_grow, constants=[_cc_Rcal()], conversions=_LE_CONV, result_name=r"pK_{d,1}", source="definition of ligand efficiency")

L27_GROW_P = Card("l27_grow_practice", "Practice: growing a kinase-hinge fragment", _GROW_EQ,
    _grow_vars(0.35, 9, 28, "project target LE (case, illustrative)"),
    "A 7-azaindole fragment ([[h0]]) binds the hinge of a mutant kinase. The team plans to grow it to [[h1]] and accepts LE "
    "falling to [[LE]] ([[T]]). What potency must the final compound reach?",
    _solve_grow, constants=[_cc_Rcal()], conversions=_LE_CONV, result_name=r"pK_{d,1}", source="definition of ligand efficiency")


# ---- fragment linking (additivity model) ----------------------------------------------------------------
def _solve_link(card, d):
    S = Solution(card, d); V = card.V
    KA, KB, hA, hB, hL, c = d["KA"], d["KB"], d["hA"], d["hB"], d["hL"], d["c"]
    if KA <= 0 or KB <= 0:
        raise ValueError("Kd values must be > 0")
    _cc_units(S.general().given(), "Kd in µM is written in mol/L (micro = 10⁻⁶) before taking −log₁₀; free energies are in kcal/mol and add directly.")
    f = _cc_ln10RT(S, V)
    pA, pB = -math.log10(KA * 1e-6), -math.log10(KB * 1e-6)
    gA, gB = -f * pA, -f * pB
    gI = gA + gB
    gS = gI + c
    pI, pS = -gI / f, -gS / f
    HA = hA + hB + hL
    S.step("Fragment A", r"pK_{A} = -\log_{10}\left(%s \times 10^{-6}\right) = %s;\quad \Delta G_{A} = -%s \times %s = %s\,\mathrm{kcal/mol}" % (_cc_f(KA), _cc_f(pA), _cc_f(f), _cc_f(pA), _cc_f(gA)))
    S.step("Fragment B", r"pK_{B} = -\log_{10}\left(%s \times 10^{-6}\right) = %s;\quad \Delta G_{B} = -%s \times %s = %s\,\mathrm{kcal/mol}" % (_cc_f(KB), _cc_f(pB), _cc_f(f), _cc_f(pB), _cc_f(gB)))
    S.step("Perfect additivity", r"\Delta G_{ideal} = %s + \left(%s\right) = %s\,\mathrm{kcal/mol};\quad pK_d = %s;\quad K_d = %s\,\mathrm{nM}" % (_cc_f(gA), _cc_f(gB), _cc_f(gI), _cc_f(pI), _cc_f(1e9 * 10 ** -pI)))
    S.step("With the linking penalty", r"%s = %s + %s = %s\,\mathrm{kcal/mol};\quad pK_d = %s" % (V["gL"].tex(), _cc_f(gI), V["c"].tex(), _cc_f(gS), _cc_f(pS)))
    S.step("Kd of the linked compound", r"K_d = 10^{-%s} = %s\,\mathrm{M} = %s\,\mathrm{\mu M}" % (_cc_f(pS), _cc_sci(10 ** -pS), _cc_f(1e6 * 10 ** -pS)))
    S.step("Heavy atoms and LE", r"%s + %s + %s = %s;\quad LE_{ideal} = \frac{%s}{%s} = %s;\quad LE = \frac{%s}{%s} = %s" % (
        V["hA"].tex(), V["hB"].tex(), V["hL"].tex(), _cc_f(HA), _cc_f(-gI), _cc_f(HA), _cc_f(-gI / HA), _cc_f(-gS), _cc_f(HA), _cc_f(-gS / HA)))
    S.answer(1e6 * 10 ** -pS, "µM", "Two weak fragments (%s and %s µM) give about %s nM if their energies add perfectly, but %s µM with a %s kcal/mol "
             "linking penalty (LE %s): %s kcal/mol costs a factor of %s. The linker must hold both fragments in place without strain." % (
                 _cc_f(KA), _cc_f(KB), _cc_f(1e9 * 10 ** -pI), _cc_f(1e6 * 10 ** -pS), _cc_f(c), _cc_f(-gS / HA), _cc_f(c), _cc_f(10 ** (c / f))))
    return S


def _link_vars(KA, hA, KB, hB, hL, c, src):
    return [Var("KA", r"K_{d,A}", "Kd of fragment A", "µM", "input", src, KA),
            Var("hA", r"N_{A}", "heavy atoms of fragment A", "", "input", src, hA),
            Var("KB", r"K_{d,B}", "Kd of fragment B", "µM", "input", src, KB),
            Var("hB", r"N_{B}", "heavy atoms of fragment B", "", "input", src, hB),
            Var("hL", r"N_{link}", "heavy atoms in the linker", "", "input", "design (case)", hL),
            Var("c", r"\Delta G_{link}", "linking penalty (strain, poor geometry); positive = worse", "kcal/mol", "input", "modelling estimate (case, illustrative)", c),
            _cc_T(298, "25 °C convention"),
            Var("gL", r"\Delta G_{linked}", "binding free energy of the linked compound", "kcal/mol", "result", "calculated")]


_LINK_EQ = r"{{gL}} = \Delta G_{A} + \Delta G_{B} + {{c}};\quad \Delta G_{X} = 2.303\,{{R}}{{T}}\log_{10} K_{d,X}"

L27_LINK = Card("l27_link", "Linking two fragments (additivity model)", _LINK_EQ,
    _link_vars(1000, 10, 500, 9, 3, 1.5, "fragment screen (case, illustrative)"),
    "Fragment A ([[hA]]) binds with [[KA]]; fragment B ([[hB]]) with [[KB]] in the neighbouring pocket. They are joined by a "
    "[[hL]] linker. Estimate the Kd of the linked compound if the energies add perfectly, and with a strain penalty of [[c]] ([[T]]).",
    _solve_link, constants=[_cc_Rcal()], conversions=_LE_CONV + ["1 mM = 1000 µM = 10⁻³ M"], result_name=r"K_{d,linked}",
    source="additivity of fragment binding free energies (illustrative model)")

L27_LINK_P = Card("l27_link_practice", "Practice: linking fragments found by NMR", _LINK_EQ,
    _link_vars(2000, 8, 300, 11, 2, 1.0, "NMR fragment screen (case, illustrative)"),
    "An NMR screen against an anti-apoptotic protein finds fragment A ([[hA]], [[KA]]) and fragment B ([[hB]], [[KB]]) in "
    "adjacent pockets. A [[hL]] linker is modelled with an estimated penalty of [[c]] ([[T]]). What Kd and LE should the "
    "linked compound have?",
    _solve_link, constants=[_cc_Rcal()], conversions=_LE_CONV + ["1 mM = 1000 µM = 10⁻³ M"], result_name=r"K_{d,linked}",
    source="additivity of fragment binding free energies (illustrative model)")


# ---- rule of three ---------------------------------------------------------------------------------------
def _solve_ro3(card, d):
    S = Solution(card, d); V = card.V
    tests = [("MW", 300), ("logP", 3), ("HBD", 3), ("HBA", 3), ("rot", 3)]
    _cc_units(S.general().given(), "MW in g/mol (Da); clog P is a logarithm; the other three are counts.")
    v = 0
    for key, lim in tests:
        x = d[key]; bad = x > lim; v += bad
        S.step("Test: %s" % V[key].name, r"%s = %s %s %s \Rightarrow \mathrm{%s}" % (V[key].tex(), _cc_f(x), ">" if bad else r"\leq", _cc_f(lim), "fail" if bad else "pass"))
    S.step("Count the failures", r"%s = %d" % (V["v"].tex(), v))
    S.answer(v, "", "%d failure(s): %s" % (v, "a good fragment-library member with room to grow while staying drug-like." if v == 0 else
             "already lead-sized or too lipophilic for a fragment library; little room to grow."))
    return S


L27_RO3 = Card("l27_ro3", "Practice: the fragment 'rule of three'",
    r"{{v}} = \left[{{MW}} > 300\right] + \left[{{logP}} > 3\right] + \left[{{HBD}} > 3\right] + \left[{{HBA}} > 3\right] + \left[{{rot}} > 3\right]",
    [Var("MW", r"MW", "molecular weight", "g/mol", "input", "vendor data (case, illustrative)", 188),
     Var("logP", r"c\log P", "calculated log P", "", "input", "vendor data (case, illustrative)", 1.4),
     Var("HBD", r"HBD", "H-bond donors", "", "input", "counted (case)", 1),
     Var("HBA", r"HBA", "H-bond acceptors", "", "input", "counted (case)", 3),
     Var("rot", r"n_{rot}", "rotatable bonds", "", "input", "counted (case)", 1),
     Var("v", r"n_{fail}", "number of rule-of-three failures", "", "result", "calculated")],
    "A hospital research pharmacy's fragment library is being expanded. A candidate has MW [[MW]], clog P [[logP]], "
    "[[HBD]] donor, [[HBA]] acceptors and [[rot]] rotatable bond. Does it meet the rule of three?",
    _solve_ro3, conversions=["1 Da = 1 g/mol", "rule of five (drugs): MW ≤ 500, log P ≤ 5, HBD ≤ 5, HBA ≤ 10"],
    result_name=r"n_{fail}", source="rule of three for fragment libraries (Congreve et al.)")


# ---- fold improvement -> ΔΔG ------------------------------------------------------------------------------
def _solve_fold(card, d):
    S = Solution(card, d); V = card.V
    F = d["F"]
    if F <= 0:
        raise ValueError("fold change must be > 0")
    _cc_units(S.general().given(), "The fold change is a ratio of two Kd values, so their units (and any nM/µM prefix) cancel and ln F is a pure number. R is in cal/(mol·K), so RT comes out in cal/mol and is divided by 10³ to give kcal/mol; K cancels in R × T.")
    _cc_temp_step(S, V)
    RT = _cc_RT_step(S, V)
    lnF = math.log(F)
    dd = RT * lnF
    S.step("Natural log of the fold change", r"\ln %s = \ln %s = %s" % (V["F"].tex(), _cc_f(F), _cc_f(lnF)))
    S.step("Free-energy gain", r"%s = %s \times %s = %s\,\mathrm{kcal/mol}" % (V["ddG"].tex(), _cc_f(RT), _cc_f(lnF), _cc_f(dd)))
    S.step("In kJ/mol and log units", r"%s \times 4.184 = %s\,\mathrm{kJ/mol};\quad \log_{10} %s = %s" % (_cc_f(dd), _cc_f(dd * 4.184), _cc_f(F), _cc_f(math.log10(F))))
    S.answer(dd, "kcal/mol", "A %s-fold gain in affinity is worth %s kcal/mol (%s kJ/mol), about the energy of one or two good hydrogen bonds or "
             "an ionic contact made in the right place, here the amidine replacing the clashing lactam C=O." % (_cc_f(F), _cc_f(dd), _cc_f(dd * 4.184)))
    return S


L27_FOLD = Card("l27_fold_ddg", "What a fold improvement is worth: ΔΔG = RT ln(fold)",
    r"{{ddG}} = {{R}}{{T}}\ln {{F}};\quad {{F}} = \frac{K_{d,lead}}{K_{d,optimized}}",
    [Var("F", r"F", "fold improvement in affinity", "", "input", "TS case study (notes)", 500),
     _cc_T(298, "assay temperature (case)"),
     Var("ddG", r"-\Delta\Delta G", "free-energy gain from optimization", "kcal/mol", "result", "calculated")],
    "Structure-based optimization of the naphthostyril thymidylate synthase lead gave a compound about [[F]] more active. "
    "Treating activity as affinity at [[T]], how much binding free energy did the optimization add?",
    _solve_fold, constants=[_cc_Rcal()], conversions=_CC_THERMO, result_name=r"-\Delta\Delta G", source="ΔG° = RT ln Kd applied to two compounds")


# ---------------------------------------------------------------------------
# L27 lecture-notes figures and calculators: de novo design, fragment growing
# and linking, ligand efficiency, the thymidylate synthase case study.
# Pyrimidines are drawn with N1 at the bottom (sugar attachment); fused ring
# systems on the left with side chains running to the right.
# Every number printed is computed here from the inputs or from the drawing.
# ---------------------------------------------------------------------------
import math as _m
import re as _re
import numpy as _np
from matplotlib.patches import FancyBboxPatch as _FBox, Circle as _Circ, FancyArrowPatch as _FArr, Ellipse as _Ell

_AW = {"C": 12.011, "H": 1.008, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "P": 30.974}
_VAL = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1}
_SUBD = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
_SUBU = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
RT_LN10 = 0.0019872 * 298.15 * _m.log(10)   # kcal/mol per pKd unit at 25 °C (≈ 1.364)


def _parse(lab):
    t = lab.translate(_SUBD).replace("⁺", "").replace("⁻", "")
    out = {}
    for el, n in _re.findall(r"([A-Z][a-z]?)(\d*)", t):
        out[el] = out.get(el, 0) + int(n or 1)
    return out


def _graph(m):
    nb = {i: [] for i in range(len(m.atoms))}
    for b in m.bonds:
        nb[b["i"]].append((b["j"], b["order"])); nb[b["j"]].append((b["i"], b["order"]))
    return nb


def _in_ring(m, i, j):
    nb = _graph(m)
    seen, stack = {i}, [i]
    while stack:
        a = stack.pop()
        for k, _ in nb[a]:
            if (a, k) in ((i, j), (j, i)) or k in seen:
                continue
            seen.add(k); stack.append(k)
    return j in seen


def mol_props(m):
    """Formula, MW, donors (N–H + O–H), acceptors (N + O), heavy atoms, rotatable bonds, counted from a drawing."""
    nb = _graph(m)
    tot, hbd, hba, deg = {}, 0, 0, {}
    for i, a in enumerate(m.atoms):
        lab = a["label"]
        order = sum(o for _, o in nb[i])
        deg[i] = len(nb[i])
        if lab in ("R", "R'"):
            continue
        if not lab:
            tot["C"] = tot.get("C", 0) + 1
            tot["H"] = tot.get("H", 0) + max(0, 4 - order)
            continue
        p = _parse(lab)
        heavy = [e for e in p if e != "H"]
        h = p.get("H", 0)
        if not heavy:                       # an explicit "H" drawn on a carbon
            tot["H"] = tot.get("H", 0) + h
            continue
        el = heavy[0]
        if len(p) == 1 and p[el] == 1 and el in _VAL:
            h += max(0, _VAL[el] - order)
        for e, n in p.items():
            if e != "H":
                tot[e] = tot.get(e, 0) + n
        tot["H"] = tot.get("H", 0) + h
        if el in ("N", "O"):
            hba += 1; hbd += h
    rot = 0
    for b in m.bonds:
        i, j = b["i"], b["j"]
        if b["order"] == 1 and deg[i] > 1 and deg[j] > 1 and not _in_ring(m, i, j):
            amide = any(m.atoms[c]["label"] == "" and m.atoms[n]["label"].startswith("N") and
                        any(m.atoms[k]["label"] == "O" and o == 2 for k, o in _graph(m)[c])
                        for c, n in ((i, j), (j, i)))
            rot += 0 if amide else 1
    order_ = ["C", "H"] + sorted(e for e in tot if e not in ("C", "H"))
    formula = "".join(e + (str(tot[e]) if tot[e] > 1 else "") for e in order_ if tot.get(e))
    mw = sum(_AW[e] * n for e, n in tot.items())
    return dict(formula=formula, MW=mw, HBD=hbd, HBA=hba, heavy=sum(n for e, n in tot.items() if e != "H"), nrot=rot)


def _f(s):
    return s.translate(_SUBU)


# ---------------------------------------------------------------- de novo workflow
def fig_denovo_workflow():
    """The de novo design cycle drawn with the Pathway kit: bench steps (green) around in-silico steps (teal)."""
    P = Pathway("The de novo design cycle: from a crystal structure to a tested compound", 11.0, 6.0)
    P.compartment(25, 30, 73, 28, "in silico (molecular modelling)", "nucleus", membrane=False, label_pos="tl")
    P.node("xr", 12, 41, "1. X-ray structure\nprotein + ligand", "hormone")
    P.node("site", 38, 41, "2. Find the site\nand its contacts", "enzyme")
    P.node("map", 62, 41, "3. Map regions;\nremove ligand", "enzyme")
    P.node("des", 85, 41, "4. Design ligands;\nscore the fit", "enzyme")
    P.node("syn", 85, 14, "5. Synthesize, test;\nco-crystallize best", "hormone")
    P.note(38, 51, "H-bond, ionic and\nhydrophobic regions", fs=7.6, ha="center")
    P.note(62, 51, "probes or grids on\nthe empty pocket", fs=7.6, ha="center")
    P.note(85, 51, "grow, link or build;\nleave room to optimize", fs=7.6, ha="center")
    P.arrow("xr", "site", "convert")
    P.arrow("site", "map", "convert")
    P.arrow("map", "des", "convert")
    P.arrow("des", "syn", "convert", "best designs", label_side=-1)
    P.arrow("syn", "xr", "indirect", rad=-0.2)
    P.note(36, 13.5, "new complex structure\nstarts the next cycle", fs=8, ha="center")
    P.note(3, 7, "at the bench: green;  in silico: teal", fs=8)
    return P.show()


# ---------------------------------------------------------------- growing / linking schematic
def _site(ax, title):
    ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.set_aspect("equal"); ax.axis("off")
    pocket = [(0.6, 6.2), (0.6, 2.0), (2.0, 0.6), (8.0, 0.6), (9.4, 2.0), (9.4, 6.2)]
    xs, ys = zip(*pocket)
    ax.plot(xs, ys, color=COL["muted"], lw=6, alpha=0.35, solid_capstyle="round")
    ax.add_patch(_Ell((2.4, 2.5), 3.0, 2.3, fc=COL["hl"], ec=COL["nonpolar"], lw=1.2, alpha=0.8))
    ax.add_patch(_Ell((7.6, 2.5), 3.0, 2.3, fc=COL["hl"], ec=COL["nonpolar"], lw=1.2, alpha=0.8))
    ax.add_patch(_Circ((5.0, 1.15), 0.42, fc="#cffafe", ec=COL["polar"], lw=1.4))
    ax.text(2.4, 1.78, "hydrophobic pocket 1", ha="center", va="bottom", fontsize=7, color=COL["nonpolar"])
    ax.text(7.6, 1.78, "hydrophobic pocket 2", ha="center", va="bottom", fontsize=7, color=COL["nonpolar"])
    ax.text(5.0, 1.15, "HBD", ha="center", va="center", fontsize=7, color=COL["polar"], fontweight="bold")
    ax.text(5.0, 0.15, "H-bond donor in the site", ha="center", fontsize=7.5, color=COL["polar"])
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")


def _hexagon(ax, cx, cy, r=0.55, color=None, ls="-"):
    pts = [(cx + r * _m.cos(_m.radians(30 + 60 * k)), cy + r * _m.sin(_m.radians(30 + 60 * k))) for k in range(7)]
    ax.plot(*zip(*pts), color=color or COL["ink"], lw=1.8, ls=ls)


def fig_grow_link():
    fig, axs = plt.subplots(2, 2, figsize=(10.5, 7.6))
    a, b, c, d = axs.ravel()
    _site(a, "A. Site map: one polar region, two hydrophobic pockets")
    _site(b, "B. Growing: extend one bound fragment")
    _hexagon(b, 2.4, 2.6)
    b.text(2.4, 4.0, "fragment hit", ha="center", fontsize=8)
    for (x0, y0), (x1, y1), lab in (((2.95, 2.6), (4.6, 2.0), "add C=O\nacceptor"), ((5.0, 2.3), (7.0, 2.6), "add aryl")):
        b.add_patch(_FArr((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=12, lw=1.3, color=COL["drug"], ls="--"))
        b.text((x0 + x1) / 2, max(y0, y1) + 0.55, lab, ha="center", fontsize=7.5, color=COL["drug"])
    _hexagon(b, 7.6, 2.6, color=COL["drug"], ls="--")
    _site(c, "C. Linking: join two fragments that bind separately")
    _hexagon(c, 2.4, 2.6); _hexagon(c, 7.6, 2.6)
    c.text(2.4, 3.6, "fragment A", ha="center", fontsize=8); c.text(7.6, 3.6, "fragment B", ha="center", fontsize=8)
    zz = [(2.95, 2.6), (3.85, 2.2), (4.6, 2.65), (5.4, 2.65), (6.15, 2.2), (7.05, 2.6)]
    c.plot(*zip(*zz), color=COL["drug"], lw=2)
    c.text(5.0, 3.1, "linker (C=O accepts\nthe H-bond)", ha="center", fontsize=7.5, color=COL["drug"])
    c.plot([5.0, 5.0], [1.6, 2.6], color=COL["polar"], lw=1.2, ls=":")
    _site(d, "D. Atom-by-atom building (e.g. LEGEND)")
    rng = _np.random.RandomState(27)
    pts = [(2.4, 2.6)]
    for k in range(13):
        x, y = pts[-1]
        ang = rng.uniform(-50, 50)
        pts.append((x + 0.42 * _m.cos(_m.radians(ang)), y + 0.42 * _m.sin(_m.radians(ang)) * 0.6))
    d.plot(*zip(*pts), color=COL["ink"], lw=1.2)
    d.scatter(*zip(*pts), s=22, color=COL["ester"], zorder=4)
    d.text(5.0, 4.3, "one atom at a time; each choice\nscored, poor branches pruned", ha="center", fontsize=8)
    fig.tight_layout(); plt.show()


# ---------------------------------------------------------------- structures
def pyrimidine_base(c5="H", n1="R"):
    """Uracil-type ring, N1 at the bottom. c5: 'H', 'CH3' or 'F'; n1: 'R' (sugar-phosphate) or 'H'."""
    m = Mol("base")
    r = ring(m, 0, 0, 6, rot=270, labels=["NH" if n1 == "H" else "N", "", "NH", "", "", ""])
    set_double(m, [(r[4], r[5])])
    sub(m, r[1], "O", order=2)
    sub(m, r[3], "O", order=2)
    if c5 == "CH3":
        sub(m, r[4], "H₃C", anchor=2, hl=COL["phos"])
    elif c5 == "F":
        sub(m, r[4], "F", hl=COL["drug"])
    else:
        sub(m, r[4], "H", color=COL["drug"])
    if n1 != "H":
        sub(m, r[0], "R")
    return m


def fig_ts_reaction():
    scheme([pyrimidine_base("H"), ("thymidylate synthase", "N⁵,N¹⁰-methylene-THF → DHF"), pyrimidine_base("CH3")],
           title="Thymidylate synthase methylates C5 of dUMP (left) to give dTMP (right)", per_row=2, gap=5.2,
           caption="R = 2′-deoxyribose 5′-phosphate. The cofactor gives the one-carbon unit and a hydride and leaves as dihydrofolate.")
    grid([pyrimidine_base("F", "H"), pyrimidine_base("F")],
         ["A. 5-Fluorouracil (C₄H₃FN₂O₂), the prodrug", "B. FdUMP base (R = 2′-deoxyribose 5′-phosphate)"], ncols=2)
    _md(["FdUMP enters the **substrate (dUMP) site**. The enzyme starts the normal reaction, but the C5–F bond cannot be "
         "broken in the step where C5–H would be removed, so enzyme, FdUMP and cofactor are locked in a **covalent ternary complex**."])


def _naph_core(m):
    """Naphthalene drawn horizontally with the shared edge vertical; returns (L, Rr) ring lists.
    L: [UR(shared top), top, UL, LL, bottom, LR(shared bottom)]; Rr from fuse: [shared top, shared bottom, bottom, LR, UR, top]."""
    L = ring(m, -C30, 0, 6, rot=30)
    Rr = fuse(m, L[0], L[5], 6)
    return L, Rr


def naphthalene():
    m = Mol("naphthalene")
    L, Rr = _naph_core(m)
    kekule(m, L, 0); set_double(m, [(Rr[2], Rr[3]), (Rr[4], Rr[5])])
    return m


def naphthostyril():
    """Benz[cd]indol-2(1H)-one (1,8-naphtholactam), C11H7NO: lactam bridges the two peri carbons at the top."""
    m = Mol("naphthostyril")
    L, Rr = _naph_core(m)
    kekule(m, L, 0); set_double(m, [(Rr[2], Rr[3]), (Rr[4], Rr[5])])
    c8, c1 = L[1], Rr[5]                        # the two top (peri) carbons
    x8, y8 = m.pos(c8); x1, y1 = m.pos(c1)
    n = m.add(x8 + 0.32, y8 + 0.95, "NH"); m.bond(c8, n)
    co = m.add(x1 - 0.32, y1 + 0.95); m.bond(c1, co); m.bond(n, co)
    sub(m, co, "O", angle=60, order=2)
    return m


def azaindole():
    """7-Azaindole (1H-pyrrolo[2,3-b]pyridine), C7H6N2: pyridine left (N7 at the bottom right), pyrrole right."""
    m = Mol("7-azaindole")
    V = ring(m, 0, 0, 6, rot=30, labels=["", "", "", "", "N", ""])   # V0 C3a, V1 C4, V2 C5, V3 C6, V4 N7, V5 C7a
    F = fuse(m, V[0], V[5], 5, labels=["NH", "", ""])                # [C3a, C7a, N1, C2, C3]
    set_double(m, [(V[5], V[4]), (V[3], V[2]), (V[1], V[0]), (F[3], F[4])])
    return m


def raltitrexed():
    """Raltitrexed, C21H22N4O6S: 2-methylquinazolin-4(3H)-one (left), N-methyl aminomethyl, thiophene, L-glutamate (right)."""
    m = Mol("raltitrexed")
    P = ring(m, 0, 0, 6, rot=30, labels=["", "", "NH", "", "N", ""])   # P0 C4a, P1 C4, P2 N3, P3 C2, P4 N1, P5 C8a
    Q = fuse(m, P[0], P[5], 6)                                        # [C4a, C8a, C8, C7, C6, C5]
    set_double(m, [(P[3], P[4]), (Q[0], Q[5]), (Q[4], Q[3]), (Q[2], Q[1])])
    sub(m, P[1], "O", angle=90, order=2)
    sub(m, P[3], "H₃C", angle=210, anchor=2)
    c6 = Q[4]
    x, y = m.pos(c6)
    ch2 = m.add(x + C30, y + 0.5); m.bond(c6, ch2)
    x, y = m.pos(ch2)
    n = m.add(x + C30, y - 0.5, "N"); m.bond(ch2, n)
    sub(m, n, "CH₃", angle=270)
    x, y = m.pos(n)
    # thiophene: C5 attached to N, S at the bottom, C2 bears the amide
    Rp = 1 / (2 * _m.sin(_m.pi / 5))
    c5x, c5y = x + C30, y + 0.5
    cx, cy = c5x - Rp * _m.cos(_m.radians(198)), c5y - Rp * _m.sin(_m.radians(198))
    T = ring(m, cx, cy, 5, rot=270, labels=["S", "", "", "", ""])     # T0 S, T1 C2, T2 C3, T3 C4, T4 C5
    m.bond(n, T[4])
    set_double(m, [(T[1], T[2]), (T[3], T[4])])
    x, y = m.pos(T[1])
    ca = m.add(x + _m.cos(_m.radians(-18)), y + _m.sin(_m.radians(-18))); m.bond(T[1], ca)
    x, y = m.pos(ca)
    m.bond(ca, m.add(x + _m.cos(_m.radians(282)), y + _m.sin(_m.radians(282)), "O"), 2)
    nh = m.add(x + _m.cos(_m.radians(42)), y + _m.sin(_m.radians(42)), "NH"); m.bond(ca, nh)
    x, y = m.pos(nh)
    alpha = m.add(x + _m.cos(_m.radians(-18)), y + _m.sin(_m.radians(-18))); m.bond(nh, alpha)
    x, y = m.pos(alpha)
    cooh = m.add(x + _m.cos(_m.radians(282)), y + _m.sin(_m.radians(282))); m.bond(alpha, cooh)
    xc, yc = m.pos(cooh)
    m.bond(cooh, m.add(xc + _m.cos(_m.radians(222)), yc + _m.sin(_m.radians(222)), "O"), 2)
    m.bond(cooh, m.add(xc + _m.cos(_m.radians(342)), yc + _m.sin(_m.radians(342)), "OH"))
    b_ = m.add(x + _m.cos(_m.radians(42)), y + _m.sin(_m.radians(42))); m.bond(alpha, b_)
    x, y = m.pos(b_)
    g_ = m.add(x + _m.cos(_m.radians(-18)), y + _m.sin(_m.radians(-18))); m.bond(b_, g_)
    x, y = m.pos(g_)
    d_ = m.add(x + _m.cos(_m.radians(42)), y + _m.sin(_m.radians(42))); m.bond(g_, d_)
    x, y = m.pos(d_)
    m.bond(d_, m.add(x + _m.cos(_m.radians(102)), y + _m.sin(_m.radians(102)), "O"), 2)
    m.bond(d_, m.add(x + _m.cos(_m.radians(-18)), y + _m.sin(_m.radians(-18)), "OH"))
    xa, ya = m.pos(alpha)
    m.note(xa + 0.05, ya + 0.45, "(S)", color=COL["muted"], size=8)
    return m


def fig_ts_scaffold():
    grid([naphthalene(), naphthostyril()],
         ["A. Step 1: naphthalene fills the hydrophobic pteridine pocket (C₁₀H₈)",
          "B. Step 2: a lactam adds H-bonding groups; benz[cd]indol-2(1H)-one (C₁₁H₇NO)"], ncols=2, align="center")
    _md(["The de novo team started from an empty, **hydrophobic** cofactor pocket, chose a scaffold **smaller** than the pocket "
         "(naphthalene), then added polar groups where the site offered hydrogen-bonding partners. Later steps added an amino "
         "group to reach a vacant region, a benzyl group mimicking the cofactor's benzene ring, and a solvent-exposed "
         "solubilizing group."])


def fig_raltitrexed():
    m = raltitrexed()
    p = mol_props(m)
    show(m, "Raltitrexed: a quinazolinone antifolate designed for the cofactor (folate) site of thymidylate synthase",
         caption="%s, MW %.1f (from the drawing). Quinazolinone replaces the folate pteridine; thiophene replaces the benzene ring; "
                 "L-glutamate (S) as in folate." % (_f(p["formula"]), p["MW"]))
    return p


# ---------------------------------------------------------------- calculators
def ligand_efficiency(pKd=6.0, heavy_atoms=25, logP=None, show_md=True):
    dG = -RT_LN10 * pKd
    le = -dG / heavy_atoms
    lines = ["1. ΔG = −2.303RT × pKd = −%.3f × %.2f = **%.2f kcal/mol** (25 °C)" % (RT_LN10, pKd, dG),
             "2. LE = −ΔG / heavy atoms = %.2f / %d = **%.2f kcal/mol per heavy atom**" % (-dG, heavy_atoms, le),
             "3. %s" % ("LE ≥ 0.3: efficient binder, a good starting point" if le >= 0.3 else
                        "LE < 0.3: affinity is being bought with size; look for a more efficient core")]
    out = dict(dG=dG, LE=le)
    if logP is not None:
        lle = pKd - logP
        out["LLE"] = lle
        lines.append("4. Lipophilic ligand efficiency LLE = pKd − log P = %.2f − %.2f = **%.2f** (≥ 5 is usually sought for a drug candidate)" % (pKd, logP, lle))
    if show_md:
        _md(lines)
    return out


def link_fragments(Kd_A_uM=1000.0, HA_A=10, Kd_B_uM=500.0, HA_B=9, linker_HA=3, link_correction=0.0):
    """Ideal additive linking (illustrative): ΔG_linked = ΔG_A + ΔG_B + correction (positive = strain/poor geometry)."""
    pA, pB = 6 - _m.log10(Kd_A_uM), 6 - _m.log10(Kd_B_uM)
    gA, gB = -RT_LN10 * pA, -RT_LN10 * pB
    gL = gA + gB + link_correction
    pL = -gL / RT_LN10
    HA_L = HA_A + HA_B + linker_HA
    KdL_nM = 10 ** (-pL) * 1e9
    leA, leB, leL = -gA / HA_A, -gB / HA_B, -gL / HA_L
    _md(["1. Fragment A: pKd = 6 − log₁₀(%.4g µM) = %.2f → ΔG = **%.2f** kcal/mol, LE = %.2f" % (Kd_A_uM, pA, gA, leA),
         "2. Fragment B: pKd = %.2f → ΔG = **%.2f** kcal/mol, LE = %.2f" % (pB, gB, leB),
         "3. Linked: ΔG = (%.2f) + (%.2f) + (%+.2f correction) = **%.2f kcal/mol** → pKd %.2f, Kd ≈ **%s**" % (gA, gB, link_correction, gL, pL,
             ("%.3g nM" % KdL_nM) if KdL_nM < 1000 else ("%.3g µM" % (KdL_nM / 1000))),
         "4. Linked heavy atoms = %d + %d + %d (linker) = %d → LE = **%.2f**" % (HA_A, HA_B, linker_HA, HA_L, leL),
         "",
         "*Illustrative model.* Real linkers rarely give perfect additivity: strain or a poor angle adds a positive correction, "
         "while the entropy saved by joining two pieces into one can make linking better than additive."])
    fig, ax = plt.subplots(figsize=(6.6, 4.0))
    ha = _np.arange(0, 41)
    for le_, ls in ((0.3, "--"), (0.45, ":")):
        ax.plot(ha, le_ * ha / RT_LN10, color=COL["muted"], ls=ls, lw=1)
        ytop = max(12, pL + 1)
        xe = min(40.0, 0.93 * ytop * RT_LN10 / le_)
        ax.text(xe, le_ * xe / RT_LN10, " LE %.2f" % le_, fontsize=8, color=COL["muted"], va="top", ha="left")
    for (h, p, lab, c) in ((HA_A, pA, "A", COL["nonpolar"]), (HA_B, pB, "B", COL["nonpolar"]), (HA_L, pL, "A–linker–B", COL["drug"])):
        ax.scatter([h], [p], s=60, color=c, zorder=5)
        ax.annotate(lab, (h, p), textcoords="offset points", xytext=(6, 4), fontsize=8.5)
    ax.set_xlim(0, 46); ax.set_ylim(0, max(12, pL + 1))
    ax.set_xlabel("heavy atoms"); ax.set_ylabel("pKd")
    ax.set_title("Fragment linking on a ligand-efficiency map (illustrative)", fontsize=10.5, fontweight="bold", loc="left")
    fig.tight_layout(); plt.show()
    return dict(pKd_A=pA, pKd_B=pB, dG_linked=gL, pKd_linked=pL, Kd_linked_nM=KdL_nM, LE_A=leA, LE_B=leB, LE_linked=leL)


RO3 = [("MW ≤ 300", "MW", 300), ("H-bond donors ≤ 3", "HBD", 3), ("H-bond acceptors (N + O) ≤ 3", "HBA", 3),
       ("Rotatable bonds ≤ 3", "nrot", 3)]


def fig_fragments_ro3():
    mols = [("7-Azaindole", azaindole()), ("Benz[cd]indol-2(1H)-one", naphthostyril())]
    props = [(n, mol_props(m)) for n, m in mols]
    grid([m for _, m in mols], ["%s. %s (%s)" % ("AB"[k], n, _f(p["formula"])) for k, (n, p) in enumerate(props)],
         ncols=2, align="center")
    rows = ["| Rule-of-three test | " + " | ".join(n for n, _ in props) + " |", "|---|" + "---|" * len(props)]
    out = {}
    for lab, key, lim in RO3:
        cells = []
        for n, p in props:
            v = p[key]
            cells.append(("%.1f" % v if key == "MW" else "%d" % v) + (" ✓" if v <= lim else " ✗"))
        rows.append("| %s | %s |" % (lab, " | ".join(cells)))
    rows.append("| Heavy atoms | " + " | ".join("%d" % p["heavy"] for _, p in props) + " |")
    for n, p in props:
        out[n] = dict(p, passes=all(p[k] <= lim for _, k, lim in RO3))
    _md(["Counted from the drawings. log P (≤ 3) and polar surface area (≤ 60 Å²) need a property calculator and are not judged here.", ""] + rows)
    return out


# ---- auditing a proposed design for binding and synthesis risks (objective L27.3) ----
def design_risk_audit(fills_site_completely=True, rotatable_bonds=1, buried_polar_without_partner=True,
                      solvent_exposed_position=False, stereocentres=4, fused_rings=4,
                      heavy_atoms=38, pKd=8.0, clogP=5.5):
    """Flag binding risks and synthetic-feasibility / property risks of a proposed de novo structure.
    Thresholds are rules of thumb used in these notes (LE >= 0.3 kcal/mol per heavy atom; LLE >= 5;
    clog P <= 5); the stereocentre and ring cut-offs are illustrative."""
    if heavy_atoms <= 0 or rotatable_bonds < 0 or stereocentres < 0 or fused_rings < 0:
        raise ValueError("heavy atoms must be > 0 and counts >= 0")
    LE = 1.364 * pKd / heavy_atoms          # kcal/mol per heavy atom at 25 °C
    LLE = pKd - clogP
    B, S = [], []                           # (flag, finding, why)
    B.append((fills_site_completely, "fills the site completely" if fills_site_completely else "leaves room in the site",
              "no tolerance for errors in the coordinates or for protein movement, and no room left to optimize"))
    B.append((rotatable_bonds <= 1 or rotatable_bonds > 10, "%d rotatable bond%s" % (rotatable_bonds, "" if rotatable_bonds == 1 else "s"),
              "too rigid to shift to an alternative binding mode if the predicted one is wrong" if rotatable_bonds <= 1 else
              "large entropy cost on binding; the designed shape may not be a low-energy conformation"))
    B.append((buried_polar_without_partner, "buried polar group with no partner" if buried_polar_without_partner else "buried polar groups all paired",
              "pays a desolvation penalty the score may not capture"))
    B.append((LE < 0.3, "LE = %.2f kcal/mol per heavy atom" % LE, "below 0.3: potency is being bought with size"))
    B.append((LLE < 5, "LLE = %.1f" % LLE, "below 5: potency is being bought with lipophilicity"))
    S.append((stereocentres >= 3, "%d stereocentre%s" % (stereocentres, "" if stereocentres == 1 else "s"),
              "each must be set correctly; many isomers to make, separate or test (illustrative cut-off: 3)"))
    S.append((fused_rings >= 4, "%d fused ring%s" % (fused_rings, "" if fused_rings == 1 else "s"),
              "polycyclic cores usually need long, low-yield routes (illustrative cut-off: 4)"))
    S.append((not solvent_exposed_position, "no position pointing to solvent" if not solvent_exposed_position else "one position points to solvent",
              "nowhere to add solubilizing groups; solubility will be hard to fix later"))
    S.append((clogP > 5, "clog P = %.1f" % clogP, "above 5: poor solubility and absorption"))
    def tab(rows, head):
        out = ["| %s | Finding | Why it matters |" % head, "|---|---|---|"]
        for flag, f_, w in rows:
            out.append("| %s | %s | %s |" % ("⚠ risk" if flag else "✓ ok", f_, w if flag else "—"))
        return out
    nb, ns = sum(r[0] for r in B), sum(r[0] for r in S)
    _md(["**Binding risks** (will it bind as designed?)", ""] + tab(B, "Binding") +
        ["", "**Synthetic-feasibility and property risks** (can it be made, and is it worth making?)", ""] + tab(S, "Synthesis / property") +
        ["", "**%d binding and %d synthesis/property risks flagged.** %s" % (nb, ns,
            "A good first design is smaller, slightly flexible, pairs every buried polar group and keeps one position open to solvent."
            if nb + ns else "Nothing flagged: check the synthetic route with a chemist (or a program such as CAESA) before making it.")])
    return dict(LE=LE, LLE=LLE, binding_risks=nb, synthesis_risks=ns)

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110})
_start_report('PHS 312', 'L27', 'De Novo Drug Design')
EXTRA_ITEMS = _json.loads('[{"qid": "L27-X1", "section": "concept", "bloom": "Apply", "silo": "5", "stem": "A site has one polar region (a backbone N-H donor) between two hydrophobic pockets. Screening finds a weak aromatic fragment in each pocket, and crystal structures show both poses. Which first design follows a sound fragment-linking strategy?", "options": {"A": "Join the two fragments with a linker that keeps both poses and carries an acceptor for the N-H", "B": "Join the two fragments directly, letting each shift until they meet, and add the acceptor later", "C": "Keep one fragment and add atoms until both pockets and the polar region are completely filled", "D": "Set the fragments aside and choose a catalogue compound that docks across all three regions"}, "key": "A", "why": {"B": "Linking pays off only if both fragments stay where they bind. Letting them shift to meet gives away the binding that made each fragment worth keeping.", "C": "This is growing, not linking, and filling every region leaves no tolerance for coordinate error and no room for later optimization.", "D": "This replaces design with virtual screening and discards the two validated binding poses that the strategy should build on.", "_correct": "Each fragment is kept in its measured pose, the linker spans the gap without strain, and a group on the linker pairs with the polar region, so all three regions are used. Affinities of well-linked fragments can roughly add."}}]')
gb.add_many([Item(**d) for d in EXTRA_ITEMS if d['qid'] not in gb.items])
_log_answers(gb)
# ===== END kit =====


## Learning objectives

| # | By the end of this activity you should be able to | Bloom | Syllabus level | SILO |
|---|---|---|---|---|
| 1 | Summarize the de novo workflow from a crystal structure to synthesis and testing. | Understand | V | 6 |
| 2 | Justify why a first de novo ligand should be loose-fitting and flexible. | Evaluate | II | 6 |
| 3 | Identify synthetic-feasibility risks and binding risks in a proposed structure. | Analyze | III | 5 |
| 4 | Design a fragment-linking strategy for a site with one polar and two hydrophobic regions. | Create | I | 5 |

*Syllabus levels follow the PHS 312 legend, in which Level I is Creating and Level VI is Remembering.*

## Part 1 - What do you already know?

Three questions before you begin. **You will not be told whether you are
right**, and that is deliberate: the point is to measure what you knew coming
in, so that at the end you can see how much the activity moved you. Answer
from what you remember from the reading. Guessing is fine.

In [ ]:
#@title P1   (run after choosing) { display-mode: "form" }
#@markdown **P1.**  What distinguishes de novo design from virtual screening?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Structures are restricted to compounds already on the market", "B. Structures are retrieved from a database rather than generated", "C. Structures are tested experimentally before any scoring is done", "D. Structures are generated to fit the site rather than retrieved"]
gb.answer("P1", response[:1])

In [ ]:
#@title P2   (run after choosing) { display-mode: "form" }
#@markdown **P2.**  Why should a first-generation de novo ligand not fill the binding site completely?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Because a large ligand cannot be scored by any docking function", "B. Because site coordinates carry error and optimization needs room", "C. Because filling the site would raise the molecular weight too far", "D. Because complete filling always produces an insoluble compound"]
gb.answer("P2", response[:1])

In [ ]:
#@title P3   (run after choosing) { display-mode: "form" }
#@markdown **P3.**  A designed structure scores superbly but a chemist says it cannot be made. What does that establish?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. That the scoring function must have been applied incorrectly", "B. That the binding site coordinates were of insufficient quality", "C. That a design score and synthetic feasibility are independent", "D. That the compound will bind but will not reach its target site"]
gb.answer("P3", response[:1])

## Part 2 - Key ideas: designing a ligand from the empty site

*Objectives addressed: L27.1 · L27.2 · L27.3*


**Summary.** Docking and virtual screening *look for* existing compounds that fit a target. **De novo design** *builds* new structures inside the binding site, guided by where the site can make hydrogen bonds, ionic contacts and hydrophobic contacts. It needs a reliable 3D structure of the target (usually an X-ray structure of the protein with a ligand bound), and its output is a **lead**, not a drug: the first designs are made, tested and co-crystallized, and the new structure drives the next round of structure-based optimization.

| | Manual design | Automated design (programs) |
|---|---|---|
| Who proposes structures | a chemist working on the displayed site | an algorithm placing fragments or atoms |
| Strength | uses chemical intuition, including what can be made | no operator bias; explores many skeletons quickly |
| Weakness | slow; limited by one person's imagination | can propose unstable or unmakeable structures; relies on an approximate scoring function |
| Typical use | one well-reasoned lead (the thymidylate synthase case in Part 4) | many candidate ideas to filter and inspect |

**Points to consider before making a design**

| Point | Why it matters |
|---|---|
| **Do not fill the site completely** | crystal coordinates carry error, the real binding mode may differ, and later optimization needs room to add groups |
| **Keep some flexibility** | a slightly flexible ligand can adopt an alternative binding mode if the predicted one is wrong |
| **Synthetic feasibility** | a superb score is worthless if the compound cannot be made in a reasonable number of steps |
| **Stable conformation** | the designed bound shape should be a low-energy conformation of the free ligand |
| **Desolvation** | polar groups buried without a partner pay a desolvation penalty; solubilizing groups should point out to solvent |
| **Species differences** | the crystal structure may be from a bacterial or animal enzyme; the human site may differ |

<details>
<summary><b>▸ Key terms (open)</b></summary>


| Term | Meaning |
|---|---|
| De novo design | Building new ligand structures to fit the binding site of a target of known 3D structure |
| Binding region / interaction site | Part of the site able to make a particular interaction (H-bond donor, acceptor, hydrophobic, ionic) |
| Fragment | Small molecule (roughly 5–30 heavy atoms, typically < 300 Da) that binds weakly but efficiently |
| Fragment growing | Adding groups to one bound fragment to reach neighbouring binding regions |
| Fragment linking | Joining two fragments that bind in neighbouring regions with a linker |
| Fragment merging | Combining overlapping features of two fragments into one new core |
| Scaffold / template | The core skeleton onto which functional groups are placed |
| Ligand efficiency (LE) | Binding free energy per heavy atom, −ΔG / N(heavy) |
| Synthetic accessibility | How easily a proposed structure can be made (estimated by programs such as CAESA) |
| Desolvation penalty | Free energy needed to strip water from polar groups before binding |

</details>

In [ ]:
#@title 2.1 Figure — the de novo design cycle (bench and in silico steps) { display-mode: "form" }
_cell('2.1 Figure — the de novo design cycle (bench and in silico steps)')
fig_denovo_workflow()

## Part 3 - Growing, linking and building: fragments and the programs that use them

*Objectives addressed: L27.4*


**Three ways to fill a site.** The site below has one polar region (an H-bond donor) and two hydrophobic pockets, the same problem shape as the pair activity.

| Strategy | What is done | Advantage | Risk |
|---|---|---|---|
| **Growing** | start from one fragment bound in one region; add groups step by step toward the next regions | each step keeps the validated binding of the first fragment | can drift into a large, inefficient molecule; early choices limit later ones |
| **Linking** | find fragments for separate regions; join them with a linker | affinities can add (or better), giving a large jump in potency | the linker must hold both fragments in their exact positions without strain |
| **Atom-by-atom building** | add single atoms, scoring each choice | the most diverse and novel skeletons | very many possibilities; slow; needs heavy pruning; more unmakeable output |
| **Template (scaffold) building** | join predefined templates that grow toward each other, then convert the skeleton to real atoms | controlled geometry; links regions systematically | limited by the template set |

In [ ]:
#@title 3.1 Figure — growing, linking and atom-by-atom building in a three-region site { display-mode: "form" }
_cell('3.1 Figure — growing, linking and atom-by-atom building in a three-region site')
fig_grow_link()

| Program | Approach | Key feature |
|---|---|---|
| **LUDI** | fragment fitting and linking | finds **interaction sites** (H-bond sites as vectors, because an H-bond needs both a distance and an angle; lipophilic sites as points), fits rigid library fragments of about 5–30 atoms to them, then **bridges** fragments with linker fragments; scored with Böhm's empirical function |
| **SPROUT** | template growth and linking | templates are graphs (vertices = atom hybridization types, edges = bond types) grown from target sites toward each other and joined; the skeleton is then converted into a real molecule |
| **LEGEND** | atom-by-atom | adds one atom at a time with a force field check; diverse skeletons but slow and needs pruning |
| **GROW** | residue-by-residue | builds **peptides** from a library of amino-acid conformations |
| **SYNOPSIS** | reaction-based | builds molecules by applying known reactions to available starting materials, so each design comes with a route |
| **CAESA** | synthesis check | estimates the **synthetic accessibility** of a designed structure (used with SPROUT) |

**Fragments and the "rule of three".** Fragment-based design screens small, simple molecules (often by X-ray crystallography or NMR, because their affinities are weak, typically 0.1–10 mM). A good fragment binds weakly but **efficiently**, leaving room to grow while staying drug-like. A widely used guide for fragment libraries is the rule of three:

| Property | Rule of three (fragments) | Rule of five (oral drugs, for comparison) |
|---|---|---|
| Molecular weight | ≤ 300 | ≤ 500 |
| clog P | ≤ 3 | ≤ 5 |
| H-bond donors | ≤ 3 | ≤ 5 |
| H-bond acceptors | ≤ 3 | ≤ 10 |
| Rotatable bonds | ≤ 3 | — |
| Polar surface area | ≤ 60 Å² | — |

In [ ]:
#@title 3.2 Figure — two fragment-sized starting points checked against the rule of three { display-mode: "form" }
_cell('3.2 Figure — two fragment-sized starting points checked against the rule of three')
_ = fig_fragments_ro3()

**1. Practice: the fragment 'rule of three'**

$${\color{#0e7490}{n_{fail}}} = \left[{\color{#1d6fb8}{MW}} > 300\right] + \left[{\color{#c2410c}{c\log P}} > 3\right] + \left[{\color{#15803d}{HBD}} > 3\right] + \left[{\color{#7c3aed}{HBA}} > 3\right] + \left[{\color{#b91c1c}{n_{rot}}} > 3\right]$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{MW}}$ | molecular weight | input | g/mol | vendor data (case, illustrative) |
| ${\color{#c2410c}{c\log P}}$ | calculated log P | input | — | vendor data (case, illustrative) |
| ${\color{#15803d}{HBD}}$ | H-bond donors | input | — | counted (case) |
| ${\color{#7c3aed}{HBA}}$ | H-bond acceptors | input | — | counted (case) |
| ${\color{#b91c1c}{n_{rot}}}$ | rotatable bonds | input | — | counted (case) |
| ${\color{#0e7490}{n_{fail}}}$ | number of rule-of-three failures | result | — | calculated |

*Source of the relationship:* rule of three for fragment libraries (Congreve et al.)

**2. Practice problem (case)**

A hospital research pharmacy's fragment library is being expanded. A candidate has MW $\mathbf{{\color{#1d6fb8}{188\,\mathrm{g/mol}}}}$, clog P $\mathbf{{\color{#c2410c}{1.4}}}$, $\mathbf{{\color{#15803d}{1}}}$ donor, $\mathbf{{\color{#7c3aed}{3}}}$ acceptors and $\mathbf{{\color{#b91c1c}{1}}}$ rotatable bond. Does it meet the rule of three?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#0e7490}{n_{fail}}} = \left[{\color{#1d6fb8}{MW}} > 300\right] + \left[{\color{#c2410c}{c\log P}} > 3\right] + \left[{\color{#15803d}{HBD}} > 3\right] + \left[{\color{#7c3aed}{HBA}} > 3\right] + \left[{\color{#b91c1c}{n_{rot}}} > 3\right]$ |
| 2. Values from the case | ${\color{#1d6fb8}{MW}} = 188\,\mathrm{g/mol}$; ${\color{#c2410c}{c\log P}} = 1.4$; ${\color{#15803d}{HBD}} = 1$; ${\color{#7c3aed}{HBA}} = 3$; ${\color{#b91c1c}{n_{rot}}} = 1$ |
| 3. Unit check | No unit prefixes to expand. MW in g/mol (Da); clog P is a logarithm; the other three are counts. |
| 4. Test: molecular weight | ${\color{#1d6fb8}{MW}} = 188 \leq 300 \Rightarrow \mathrm{pass}$ |
| 5. Test: calculated log P | ${\color{#c2410c}{c\log P}} = 1.4 \leq 3 \Rightarrow \mathrm{pass}$ |
| 6. Test: H-bond donors | ${\color{#15803d}{HBD}} = 1 \leq 3 \Rightarrow \mathrm{pass}$ |
| 7. Test: H-bond acceptors | ${\color{#7c3aed}{HBA}} = 3 \leq 3 \Rightarrow \mathrm{pass}$ |
| 8. Test: rotatable bonds | ${\color{#b91c1c}{n_{rot}}} = 1 \leq 3 \Rightarrow \mathrm{pass}$ |
| 9. Count the failures | ${\color{#0e7490}{n_{fail}}} = 0$ |
| 10. Answer | $\boxed{n_{fail} = 0}$ |
| 11. What it means | 0 failure(s): a good fragment-library member with room to grow while staying drug-like. |

**Common conversions for this calculation:** 1 Da = 1 g/mol; rule of five (drugs): MW ≤ 500, log P ≤ 5, HBD ≤ 5, HBA ≤ 10
</details>

In [ ]:
#@title 3.3 Solve it yourself: Practice: the fragment 'rule of three' { display-mode: "form" }
_cell("3.3 Solve it yourself: Practice: the fragment 'rule of three'")
card('l27_ro3')

**Ligand efficiency.** Bigger molecules usually bind more tightly simply because they make more contacts, so potency alone rewards size. Ligand efficiency divides the binding free energy by the number of heavy (non-hydrogen) atoms:

ΔG = −2.303 RT · pKd  (≈ −1.364 × pKd kcal/mol at 25 °C);  **LE = −ΔG / heavy atoms**

LE ≥ 0.3 kcal/mol per heavy atom is a common target. Fragments often have LE of 0.4 or more; the job of optimization is to add potency without letting LE collapse. **Lipophilic ligand efficiency** (LLE = pKd − log P) checks that potency is not just being bought with greasiness.

**1. Ligand efficiency and lipophilic ligand efficiency**

$${\color{#b91c1c}{\Delta G}} = -2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}\,{\color{#1d6fb8}{pK_d}};\quad {\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}};\quad {\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{pK_d}}$ | −log₁₀ of Kd (or pIC₅₀) | input | — | binding assay (case, illustrative) |
| ${\color{#c2410c}{N_{heavy}}}$ | heavy (non-hydrogen) atoms | input | — | counted from the structure (case) |
| ${\color{#15803d}{\log P}}$ | lipophilicity (clog P) | input | — | calculated property (case) |
| ${\color{#7c3aed}{T}}$ | absolute temperature | input | K | 25 °C convention |
| ${\color{#b91c1c}{\Delta G}}$ | binding free energy | intermediate | kcal/mol | calculated |
| ${\color{#0e7490}{LE}}$ | ligand efficiency | result | kcal/mol | calculated |
| ${\color{#a16207}{LLE}}$ | lipophilic ligand efficiency | intermediate | — | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* ligand efficiency (Hopkins); LLE (Leeson)

**2. Practice problem (case)**

A de novo design for the thymidylate synthase cofactor site is made and binds with $\mathbf{{\color{#1d6fb8}{6}}}$; it has $\mathbf{{\color{#c2410c}{25}}}$ and $\mathbf{{\color{#15803d}{2.5}}}$. At $\mathbf{{\color{#7c3aed}{298\,\mathrm{K}}}}$, calculate ΔG, LE and LLE. Is it an efficient starting point?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{\Delta G}} = -2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}\,{\color{#1d6fb8}{pK_d}};\quad {\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}};\quad {\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{pK_d}} = 6$; ${\color{#c2410c}{N_{heavy}}} = 25$; ${\color{#15803d}{\log P}} = 2.5$; ${\color{#7c3aed}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | No unit prefixes to expand. R is in cal/(mol·K) and the energies are in kcal/mol (kilo = 10³), so RT is converted from cal/mol to kcal/mol before dividing; K cancels in R × T and kcal/mol cancels in ΔG/RT, leaving a pure number in the exponent. pKd and log P are logarithms (no units), so LLE is unitless; LE is in kcal/mol per heavy atom. |
| 4. Temperature scales | ${\color{#7c3aed}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Free energy per log unit | $\ln 10 \times {\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 2.303 \times 0.5921 = 1.363\,\mathrm{kcal/mol}$ |
| 7. Kd from pKd | $K_d = 10^{-6} = 1 \times 10^{-6}\,\mathrm{M}$ |
| 8. Binding free energy | ${\color{#b91c1c}{\Delta G}} = -1.363 \times 6 = -8.181\,\mathrm{kcal/mol} = -34.23\,\mathrm{kJ/mol}$ |
| 9. Ligand efficiency | ${\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}} = \frac{8.181}{25} = 0.3272\,\mathrm{kcal/mol\ per\ heavy\ atom}$ |
| 10. Lipophilic ligand efficiency | ${\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}} = 6 - 2.5 = 3.5$ |
| 11. Answer (LE) | $\boxed{LE = 0.3272\,\mathrm{kcal/mol}}$ |
| 12. What it means | LE = 0.3272 kcal/mol per heavy atom (≥ 0.3) and LLE = 3.5 (below the ≥ 5 usually sought for a candidate). Optimization should add potency without letting either number collapse. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio; pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM; 2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C
</details>

In [ ]:
#@title 3.4 Solve it yourself: Ligand efficiency and lipophilic ligand efficiency { display-mode: "form" }
_cell('3.4 Solve it yourself: Ligand efficiency and lipophilic ligand efficiency')
card('l27_le_lle')

**1. Ligand efficiency of a fragment versus a lead**

$${\color{#b91c1c}{\Delta G}} = -2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}\,{\color{#1d6fb8}{pK_d}};\quad {\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}};\quad {\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{pK_d}}$ | −log₁₀ of Kd (or pIC₅₀) | input | — | fragment screen (case) |
| ${\color{#c2410c}{N_{heavy}}}$ | heavy (non-hydrogen) atoms | input | — | counted from the structure (case) |
| ${\color{#15803d}{\log P}}$ | lipophilicity (clog P) | input | — | calculated property (case) |
| ${\color{#7c3aed}{T}}$ | absolute temperature | input | K | 25 °C convention |
| ${\color{#b91c1c}{\Delta G}}$ | binding free energy | intermediate | kcal/mol | calculated |
| ${\color{#0e7490}{LE}}$ | ligand efficiency | result | kcal/mol | calculated |
| ${\color{#a16207}{LLE}}$ | lipophilic ligand efficiency | intermediate | — | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* ligand efficiency (Hopkins); LLE (Leeson)

**2. Practice problem (case)**

A fragment ($\mathbf{{\color{#c2410c}{11}}}$, $\mathbf{{\color{#15803d}{1.2}}}$) binds with $\mathbf{{\color{#1d6fb8}{3.5}}}$ at $\mathbf{{\color{#7c3aed}{298\,\mathrm{K}}}}$. A lead from another project has 38 heavy atoms and pKd 7.0 (LE ≈ 0.25). Calculate the fragment's LE and LLE. Which is the better starting point for growing?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{\Delta G}} = -2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}\,{\color{#1d6fb8}{pK_d}};\quad {\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}};\quad {\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{pK_d}} = 3.5$; ${\color{#c2410c}{N_{heavy}}} = 11$; ${\color{#15803d}{\log P}} = 1.2$; ${\color{#7c3aed}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | No unit prefixes to expand. R is in cal/(mol·K) and the energies are in kcal/mol (kilo = 10³), so RT is converted from cal/mol to kcal/mol before dividing; K cancels in R × T and kcal/mol cancels in ΔG/RT, leaving a pure number in the exponent. pKd and log P are logarithms (no units), so LLE is unitless; LE is in kcal/mol per heavy atom. |
| 4. Temperature scales | ${\color{#7c3aed}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Free energy per log unit | $\ln 10 \times {\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 2.303 \times 0.5921 = 1.363\,\mathrm{kcal/mol}$ |
| 7. Kd from pKd | $K_d = 10^{-3.5} = 3.162 \times 10^{-4}\,\mathrm{M}$ |
| 8. Binding free energy | ${\color{#b91c1c}{\Delta G}} = -1.363 \times 3.5 = -4.772\,\mathrm{kcal/mol} = -19.97\,\mathrm{kJ/mol}$ |
| 9. Ligand efficiency | ${\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}} = \frac{4.772}{11} = 0.4338\,\mathrm{kcal/mol\ per\ heavy\ atom}$ |
| 10. Lipophilic ligand efficiency | ${\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}} = 3.5 - 1.2 = 2.3$ |
| 11. Answer (LE) | $\boxed{LE = 0.4338\,\mathrm{kcal/mol}}$ |
| 12. What it means | LE = 0.4338 kcal/mol per heavy atom (≥ 0.3) and LLE = 2.3 (below the ≥ 5 usually sought for a candidate). An efficient but weakly lipophilic-efficient compound gains potency mainly through greasiness; fix that before adding more atoms. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio; pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM; 2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C
</details>

In [ ]:
#@title 3.5 Solve it yourself: Ligand efficiency of a fragment versus a lead { display-mode: "form" }
_cell('3.5 Solve it yourself: Ligand efficiency of a fragment versus a lead')
card('l27_le_fragment')

**1. Practice: is potency being bought with lipophilicity?**

$${\color{#b91c1c}{\Delta G}} = -2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}\,{\color{#1d6fb8}{pK_d}};\quad {\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}};\quad {\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{pK_d}}$ | −log₁₀ of Kd (or pIC₅₀) | input | — | kinase assay pIC₅₀ (case, illustrative) |
| ${\color{#c2410c}{N_{heavy}}}$ | heavy (non-hydrogen) atoms | input | — | counted from the structure (case) |
| ${\color{#15803d}{\log P}}$ | lipophilicity (clog P) | input | — | calculated property (case) |
| ${\color{#7c3aed}{T}}$ | absolute temperature | input | K | 25 °C convention |
| ${\color{#b91c1c}{\Delta G}}$ | binding free energy | intermediate | kcal/mol | calculated |
| ${\color{#0e7490}{LE}}$ | ligand efficiency | result | kcal/mol | calculated |
| ${\color{#a16207}{LLE}}$ | lipophilic ligand efficiency | intermediate | — | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* ligand efficiency (Hopkins); LLE (Leeson)

**2. Practice problem (case)**

An oncology team's kinase lead has pIC₅₀ $\mathbf{{\color{#1d6fb8}{8.3}}}$ (used as pKd), $\mathbf{{\color{#c2410c}{36}}}$ and clog P $\mathbf{{\color{#15803d}{5.4}}}$. At $\mathbf{{\color{#7c3aed}{298\,\mathrm{K}}}}$, calculate LE and LLE = pIC₅₀ − log P. What should the next round of design change?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{\Delta G}} = -2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}\,{\color{#1d6fb8}{pK_d}};\quad {\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}};\quad {\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{pK_d}} = 8.3$; ${\color{#c2410c}{N_{heavy}}} = 36$; ${\color{#15803d}{\log P}} = 5.4$; ${\color{#7c3aed}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | No unit prefixes to expand. R is in cal/(mol·K) and the energies are in kcal/mol (kilo = 10³), so RT is converted from cal/mol to kcal/mol before dividing; K cancels in R × T and kcal/mol cancels in ΔG/RT, leaving a pure number in the exponent. pKd and log P are logarithms (no units), so LLE is unitless; LE is in kcal/mol per heavy atom. |
| 4. Temperature scales | ${\color{#7c3aed}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Free energy per log unit | $\ln 10 \times {\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 2.303 \times 0.5921 = 1.363\,\mathrm{kcal/mol}$ |
| 7. Kd from pKd | $K_d = 10^{-8.3} = 5.012 \times 10^{-9}\,\mathrm{M}$ |
| 8. Binding free energy | ${\color{#b91c1c}{\Delta G}} = -1.363 \times 8.3 = -11.32\,\mathrm{kcal/mol} = -47.35\,\mathrm{kJ/mol}$ |
| 9. Ligand efficiency | ${\color{#0e7490}{LE}} = \frac{-{\color{#b91c1c}{\Delta G}}}{{\color{#c2410c}{N_{heavy}}}} = \frac{11.32}{36} = 0.3143\,\mathrm{kcal/mol\ per\ heavy\ atom}$ |
| 10. Lipophilic ligand efficiency | ${\color{#a16207}{LLE}} = {\color{#1d6fb8}{pK_d}} - {\color{#15803d}{\log P}} = 8.3 - 5.4 = 2.9$ |
| 11. Answer (LE) | $\boxed{LE = 0.3143\,\mathrm{kcal/mol}}$ |
| 12. What it means | LE = 0.3143 kcal/mol per heavy atom (≥ 0.3) and LLE = 2.9 (below the ≥ 5 usually sought for a candidate). An efficient but weakly lipophilic-efficient compound gains potency mainly through greasiness; fix that before adding more atoms. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio; pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM; 2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C
</details>

In [ ]:
#@title 3.6 Solve it yourself: Practice: is potency being bought with lipophilicity? { display-mode: "form" }
_cell('3.6 Solve it yourself: Practice: is potency being bought with lipophilicity?')
card('l27_lle_practice')

**1. Fragment growth bookkeeping: potency needed to keep LE**

$${\color{#b91c1c}{pK_{d,1}}} = \frac{{\color{#1d6fb8}{LE}}\,{\color{#15803d}{N_{1}}}}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}};\quad \Delta pK_d = \frac{{\color{#1d6fb8}{LE}}\left({\color{#15803d}{N_{1}}} - {\color{#c2410c}{N_{0}}}\right)}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{LE}}$ | ligand efficiency to be maintained | input | kcal/mol | fragment LE (case) |
| ${\color{#c2410c}{N_{0}}}$ | heavy atoms in the starting fragment | input | — | fragment (case) |
| ${\color{#15803d}{N_{1}}}$ | heavy atoms after growing | input | — | design plan (case) |
| ${\color{#7c3aed}{T}}$ | absolute temperature | input | K | 25 °C convention |
| ${\color{#b91c1c}{pK_{d,1}}}$ | pKd the grown compound must reach | result | — | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* definition of ligand efficiency

**2. Practice problem (case)**

A $\mathbf{{\color{#c2410c}{12}}}$ fragment is to be grown to $\mathbf{{\color{#15803d}{20}}}$ while keeping LE at $\mathbf{{\color{#1d6fb8}{0.4\,\mathrm{kcal/mol}}}}$ ($\mathbf{{\color{#7c3aed}{298\,\mathrm{K}}}}$). What pKd should the grown compound reach?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{pK_{d,1}}} = \frac{{\color{#1d6fb8}{LE}}\,{\color{#15803d}{N_{1}}}}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}};\quad \Delta pK_d = \frac{{\color{#1d6fb8}{LE}}\left({\color{#15803d}{N_{1}}} - {\color{#c2410c}{N_{0}}}\right)}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{LE}} = 0.4\,\mathrm{kcal/mol}$; ${\color{#c2410c}{N_{0}}} = 12$; ${\color{#15803d}{N_{1}}} = 20$; ${\color{#7c3aed}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | Prefixes written out: $0.4\,\mathrm{kcal/mol} = 0.4 \times 10^{3}\,\mathrm{cal/mol} = 400\,\mathrm{cal/mol}$. LE (kcal/mol per heavy atom) × heavy atoms = kcal/mol; dividing by 2.303RT (kcal/mol per log unit) leaves a pure pKd. |
| 4. Temperature scales | ${\color{#7c3aed}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Free energy per log unit | $\ln 10 \times {\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 2.303 \times 0.5921 = 1.363\,\mathrm{kcal/mol}$ |
| 7. Binding energy needed at the target size | $-\Delta G = {\color{#1d6fb8}{LE}} \times {\color{#15803d}{N_{1}}} = 0.4 \times 20 = 8\,\mathrm{kcal/mol}$ |
| 8. Target pKd | ${\color{#b91c1c}{pK_{d,1}}} = \frac{8}{1.363} = 5.868$ |
| 9. Target Kd | $K_d = 10^{-5.868} = 1.356 \times 10^{-6}\,\mathrm{M} = 1356\,\mathrm{nM}$ |
| 10. Starting pKd at the same LE | $\frac{0.4 \times 12}{1.363} = 3.521$ |
| 11. Potency bought per added heavy atom | $\frac{0.4}{1.363} = 0.2934\ \mathrm{pK_d\ per\ atom};\quad 8 \times 0.2934 = 2.347\ \mathrm{log\ units}$ |
| 12. Answer (target pKd) | $\boxed{pK_{d,1} = 5.868}$ |
| 13. What it means | Growing from 12 to 20 heavy atoms at LE 0.4 must reach pKd 5.868 (Kd ≈ 1356 nM): 2.347 log units for 8 added atoms. If a grown analogue falls short, its LE has dropped and the added atoms are not earning their place. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio; pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM; 2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C
</details>

In [ ]:
#@title 3.7 Solve it yourself: Fragment growth bookkeeping: potency needed to keep LE { display-mode: "form" }
_cell('3.7 Solve it yourself: Fragment growth bookkeeping: potency needed to keep LE')
card('l27_grow')

**1. Practice: growing a kinase-hinge fragment**

$${\color{#b91c1c}{pK_{d,1}}} = \frac{{\color{#1d6fb8}{LE}}\,{\color{#15803d}{N_{1}}}}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}};\quad \Delta pK_d = \frac{{\color{#1d6fb8}{LE}}\left({\color{#15803d}{N_{1}}} - {\color{#c2410c}{N_{0}}}\right)}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{LE}}$ | ligand efficiency to be maintained | input | kcal/mol | project target LE (case, illustrative) |
| ${\color{#c2410c}{N_{0}}}$ | heavy atoms in the starting fragment | input | — | fragment (case) |
| ${\color{#15803d}{N_{1}}}$ | heavy atoms after growing | input | — | design plan (case) |
| ${\color{#7c3aed}{T}}$ | absolute temperature | input | K | 25 °C convention |
| ${\color{#b91c1c}{pK_{d,1}}}$ | pKd the grown compound must reach | result | — | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* definition of ligand efficiency

**2. Practice problem (case)**

A 7-azaindole fragment ($\mathbf{{\color{#c2410c}{9}}}$) binds the hinge of a mutant kinase. The team plans to grow it to $\mathbf{{\color{#15803d}{28}}}$ and accepts LE falling to $\mathbf{{\color{#1d6fb8}{0.35\,\mathrm{kcal/mol}}}}$ ($\mathbf{{\color{#7c3aed}{298\,\mathrm{K}}}}$). What potency must the final compound reach?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{pK_{d,1}}} = \frac{{\color{#1d6fb8}{LE}}\,{\color{#15803d}{N_{1}}}}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}};\quad \Delta pK_d = \frac{{\color{#1d6fb8}{LE}}\left({\color{#15803d}{N_{1}}} - {\color{#c2410c}{N_{0}}}\right)}{2.303\,{\color{#4b5563}{R}}{\color{#7c3aed}{T}}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{LE}} = 0.35\,\mathrm{kcal/mol}$; ${\color{#c2410c}{N_{0}}} = 9$; ${\color{#15803d}{N_{1}}} = 28$; ${\color{#7c3aed}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | Prefixes written out: $0.35\,\mathrm{kcal/mol} = 0.35 \times 10^{3}\,\mathrm{cal/mol} = 350\,\mathrm{cal/mol}$. LE (kcal/mol per heavy atom) × heavy atoms = kcal/mol; dividing by 2.303RT (kcal/mol per log unit) leaves a pure pKd. |
| 4. Temperature scales | ${\color{#7c3aed}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Free energy per log unit | $\ln 10 \times {\color{#4b5563}{R}} {\color{#7c3aed}{T}} = 2.303 \times 0.5921 = 1.363\,\mathrm{kcal/mol}$ |
| 7. Binding energy needed at the target size | $-\Delta G = {\color{#1d6fb8}{LE}} \times {\color{#15803d}{N_{1}}} = 0.35 \times 28 = 9.8\,\mathrm{kcal/mol}$ |
| 8. Target pKd | ${\color{#b91c1c}{pK_{d,1}}} = \frac{9.8}{1.363} = 7.188$ |
| 9. Target Kd | $K_d = 10^{-7.188} = 6.489 \times 10^{-8}\,\mathrm{M} = 64.89\,\mathrm{nM}$ |
| 10. Starting pKd at the same LE | $\frac{0.35 \times 9}{1.363} = 2.31$ |
| 11. Potency bought per added heavy atom | $\frac{0.35}{1.363} = 0.2567\ \mathrm{pK_d\ per\ atom};\quad 19 \times 0.2567 = 4.877\ \mathrm{log\ units}$ |
| 12. Answer (target pKd) | $\boxed{pK_{d,1} = 7.188}$ |
| 13. What it means | Growing from 9 to 28 heavy atoms at LE 0.35 must reach pKd 7.188 (Kd ≈ 64.89 nM): 4.877 log units for 19 added atoms. If a grown analogue falls short, its LE has dropped and the added atoms are not earning their place. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio; pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM; 2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C
</details>

In [ ]:
#@title 3.8 Solve it yourself: Practice: growing a kinase-hinge fragment { display-mode: "form" }
_cell('3.8 Solve it yourself: Practice: growing a kinase-hinge fragment')
card('l27_grow_practice')

**Why linking can give a large jump.** If two fragments bind independently in neighbouring regions, joining them perfectly would roughly **add their binding free energies** (multiply their affinities), and the linked molecule also loses less rigid-body entropy on binding than two separate pieces. In practice a linker that strains either fragment or shifts it out of position gives back much of that gain.

In [ ]:
#@title 3.9 Calculator — linking two fragments (illustrative additivity model) { display-mode: "form" }
_cell('3.9 Calculator — linking two fragments (illustrative additivity model)')
Kd_fragment_A_uM = 1000 #@param {type:"number"}
heavy_atoms_A = 10 #@param {type:"integer"}
Kd_fragment_B_uM = 500 #@param {type:"number"}
heavy_atoms_B = 9 #@param {type:"integer"}
linker_heavy_atoms = 3 #@param {type:"integer"}
linking_correction_kcal = 0.0 #@param {type:"slider", min:-3.0, max:4.0, step:0.5}
_ = link_fragments(Kd_fragment_A_uM, heavy_atoms_A, Kd_fragment_B_uM, heavy_atoms_B, linker_heavy_atoms, linking_correction_kcal)

**1. Linking two fragments (additivity model)**

$${\color{#be185d}{\Delta G_{linked}}} = \Delta G_{A} + \Delta G_{B} + {\color{#0e7490}{\Delta G_{link}}};\quad \Delta G_{X} = 2.303\,{\color{#4b5563}{R}}{\color{#a16207}{T}}\log_{10} K_{d,X}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{K_{d,A}}}$ | Kd of fragment A | input | µM | fragment screen (case, illustrative) |
| ${\color{#c2410c}{N_{A}}}$ | heavy atoms of fragment A | input | — | fragment screen (case, illustrative) |
| ${\color{#15803d}{K_{d,B}}}$ | Kd of fragment B | input | µM | fragment screen (case, illustrative) |
| ${\color{#7c3aed}{N_{B}}}$ | heavy atoms of fragment B | input | — | fragment screen (case, illustrative) |
| ${\color{#b91c1c}{N_{link}}}$ | heavy atoms in the linker | input | — | design (case) |
| ${\color{#0e7490}{\Delta G_{link}}}$ | linking penalty (strain, poor geometry); positive = worse | input | kcal/mol | modelling estimate (case, illustrative) |
| ${\color{#a16207}{T}}$ | absolute temperature | input | K | 25 °C convention |
| ${\color{#be185d}{\Delta G_{linked}}}$ | binding free energy of the linked compound | result | kcal/mol | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* additivity of fragment binding free energies (illustrative model)

**2. Practice problem (case)**

Fragment A ($\mathbf{{\color{#c2410c}{10}}}$) binds with $\mathbf{{\color{#1d6fb8}{1000\,\mathrm{\mu\ M}}}}$; fragment B ($\mathbf{{\color{#7c3aed}{9}}}$) with $\mathbf{{\color{#15803d}{500\,\mathrm{\mu\ M}}}}$ in the neighbouring pocket. They are joined by a $\mathbf{{\color{#b91c1c}{3}}}$ linker. Estimate the Kd of the linked compound if the energies add perfectly, and with a strain penalty of $\mathbf{{\color{#0e7490}{1.5\,\mathrm{kcal/mol}}}}$ ($\mathbf{{\color{#a16207}{298\,\mathrm{K}}}}$).

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#be185d}{\Delta G_{linked}}} = \Delta G_{A} + \Delta G_{B} + {\color{#0e7490}{\Delta G_{link}}};\quad \Delta G_{X} = 2.303\,{\color{#4b5563}{R}}{\color{#a16207}{T}}\log_{10} K_{d,X}$ |
| 2. Values from the case | ${\color{#1d6fb8}{K_{d,A}}} = 1000\,\mathrm{\mu\ M}$; ${\color{#c2410c}{N_{A}}} = 10$; ${\color{#15803d}{K_{d,B}}} = 500\,\mathrm{\mu\ M}$; ${\color{#7c3aed}{N_{B}}} = 9$; ${\color{#b91c1c}{N_{link}}} = 3$; ${\color{#0e7490}{\Delta G_{link}}} = 1.5\,\mathrm{kcal/mol}$; ${\color{#a16207}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | Prefixes written out: $1000\,\mathrm{\mu\ M} = 1000 \times 10^{-6}\,\mathrm{M} = 0.001\,\mathrm{M}$; $500\,\mathrm{\mu\ M} = 500 \times 10^{-6}\,\mathrm{M} = 5 \times 10^{-4}\,\mathrm{M}$; $1.5\,\mathrm{kcal/mol} = 1.5 \times 10^{3}\,\mathrm{cal/mol} = 1500\,\mathrm{cal/mol}$. Kd in µM is written in mol/L (micro = 10⁻⁶) before taking −log₁₀; free energies are in kcal/mol and add directly. |
| 4. Temperature scales | ${\color{#a16207}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#a16207}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Free energy per log unit | $\ln 10 \times {\color{#4b5563}{R}} {\color{#a16207}{T}} = 2.303 \times 0.5921 = 1.363\,\mathrm{kcal/mol}$ |
| 7. Fragment A | $pK_{A} = -\log_{10}\left(1000 \times 10^{-6}\right) = 3;\quad \Delta G_{A} = -1.363 \times 3 = -4.09\,\mathrm{kcal/mol}$ |
| 8. Fragment B | $pK_{B} = -\log_{10}\left(500 \times 10^{-6}\right) = 3.301;\quad \Delta G_{B} = -1.363 \times 3.301 = -4.501\,\mathrm{kcal/mol}$ |
| 9. Perfect additivity | $\Delta G_{ideal} = -4.09 + \left(-4.501\right) = -8.591\,\mathrm{kcal/mol};\quad pK_d = 6.301;\quad K_d = 500\,\mathrm{nM}$ |
| 10. With the linking penalty | ${\color{#be185d}{\Delta G_{linked}}} = -8.591 + {\color{#0e7490}{\Delta G_{link}}} = -7.091\,\mathrm{kcal/mol};\quad pK_d = 5.201$ |
| 11. Kd of the linked compound | $K_d = 10^{-5.201} = 6.297 \times 10^{-6}\,\mathrm{M} = 6.297\,\mathrm{\mu M}$ |
| 12. Heavy atoms and LE | ${\color{#c2410c}{N_{A}}} + {\color{#7c3aed}{N_{B}}} + {\color{#b91c1c}{N_{link}}} = 22;\quad LE_{ideal} = \frac{8.591}{22} = 0.3905;\quad LE = \frac{7.091}{22} = 0.3223$ |
| 13. Answer | $\boxed{K_{d,linked} = 6.297\,\mathrm{\mu\ M}}$ |
| 14. What it means | Two weak fragments (1000 and 500 µM) give about 500 nM if their energies add perfectly, but 6.297 µM with a 1.5 kcal/mol linking penalty (LE 0.3223): 1.5 kcal/mol costs a factor of 12.59. The linker must hold both fragments in place without strain. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio; pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM; 2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C; 1 mM = 1000 µM = 10⁻³ M
</details>

In [ ]:
#@title 3.10 Solve it yourself: Linking two fragments (additivity model) { display-mode: "form" }
_cell('3.10 Solve it yourself: Linking two fragments (additivity model)')
card('l27_link')

**1. Practice: linking fragments found by NMR**

$${\color{#be185d}{\Delta G_{linked}}} = \Delta G_{A} + \Delta G_{B} + {\color{#0e7490}{\Delta G_{link}}};\quad \Delta G_{X} = 2.303\,{\color{#4b5563}{R}}{\color{#a16207}{T}}\log_{10} K_{d,X}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{K_{d,A}}}$ | Kd of fragment A | input | µM | NMR fragment screen (case, illustrative) |
| ${\color{#c2410c}{N_{A}}}$ | heavy atoms of fragment A | input | — | NMR fragment screen (case, illustrative) |
| ${\color{#15803d}{K_{d,B}}}$ | Kd of fragment B | input | µM | NMR fragment screen (case, illustrative) |
| ${\color{#7c3aed}{N_{B}}}$ | heavy atoms of fragment B | input | — | NMR fragment screen (case, illustrative) |
| ${\color{#b91c1c}{N_{link}}}$ | heavy atoms in the linker | input | — | design (case) |
| ${\color{#0e7490}{\Delta G_{link}}}$ | linking penalty (strain, poor geometry); positive = worse | input | kcal/mol | modelling estimate (case, illustrative) |
| ${\color{#a16207}{T}}$ | absolute temperature | input | K | 25 °C convention |
| ${\color{#be185d}{\Delta G_{linked}}}$ | binding free energy of the linked compound | result | kcal/mol | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* additivity of fragment binding free energies (illustrative model)

**2. Practice problem (case)**

An NMR screen against an anti-apoptotic protein finds fragment A ($\mathbf{{\color{#c2410c}{8}}}$, $\mathbf{{\color{#1d6fb8}{2000\,\mathrm{\mu\ M}}}}$) and fragment B ($\mathbf{{\color{#7c3aed}{11}}}$, $\mathbf{{\color{#15803d}{300\,\mathrm{\mu\ M}}}}$) in adjacent pockets. A $\mathbf{{\color{#b91c1c}{2}}}$ linker is modelled with an estimated penalty of $\mathbf{{\color{#0e7490}{1\,\mathrm{kcal/mol}}}}$ ($\mathbf{{\color{#a16207}{298\,\mathrm{K}}}}$). What Kd and LE should the linked compound have?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#be185d}{\Delta G_{linked}}} = \Delta G_{A} + \Delta G_{B} + {\color{#0e7490}{\Delta G_{link}}};\quad \Delta G_{X} = 2.303\,{\color{#4b5563}{R}}{\color{#a16207}{T}}\log_{10} K_{d,X}$ |
| 2. Values from the case | ${\color{#1d6fb8}{K_{d,A}}} = 2000\,\mathrm{\mu\ M}$; ${\color{#c2410c}{N_{A}}} = 8$; ${\color{#15803d}{K_{d,B}}} = 300\,\mathrm{\mu\ M}$; ${\color{#7c3aed}{N_{B}}} = 11$; ${\color{#b91c1c}{N_{link}}} = 2$; ${\color{#0e7490}{\Delta G_{link}}} = 1\,\mathrm{kcal/mol}$; ${\color{#a16207}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | Prefixes written out: $2000\,\mathrm{\mu\ M} = 2000 \times 10^{-6}\,\mathrm{M} = 0.002\,\mathrm{M}$; $300\,\mathrm{\mu\ M} = 300 \times 10^{-6}\,\mathrm{M} = 3 \times 10^{-4}\,\mathrm{M}$; $1\,\mathrm{kcal/mol} = 1 \times 10^{3}\,\mathrm{cal/mol} = 1000\,\mathrm{cal/mol}$. Kd in µM is written in mol/L (micro = 10⁻⁶) before taking −log₁₀; free energies are in kcal/mol and add directly. |
| 4. Temperature scales | ${\color{#a16207}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#a16207}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Free energy per log unit | $\ln 10 \times {\color{#4b5563}{R}} {\color{#a16207}{T}} = 2.303 \times 0.5921 = 1.363\,\mathrm{kcal/mol}$ |
| 7. Fragment A | $pK_{A} = -\log_{10}\left(2000 \times 10^{-6}\right) = 2.699;\quad \Delta G_{A} = -1.363 \times 2.699 = -3.68\,\mathrm{kcal/mol}$ |
| 8. Fragment B | $pK_{B} = -\log_{10}\left(300 \times 10^{-6}\right) = 3.523;\quad \Delta G_{B} = -1.363 \times 3.523 = -4.803\,\mathrm{kcal/mol}$ |
| 9. Perfect additivity | $\Delta G_{ideal} = -3.68 + \left(-4.803\right) = -8.483\,\mathrm{kcal/mol};\quad pK_d = 6.222;\quad K_d = 600\,\mathrm{nM}$ |
| 10. With the linking penalty | ${\color{#be185d}{\Delta G_{linked}}} = -8.483 + {\color{#0e7490}{\Delta G_{link}}} = -7.483\,\mathrm{kcal/mol};\quad pK_d = 5.488$ |
| 11. Kd of the linked compound | $K_d = 10^{-5.488} = 3.248 \times 10^{-6}\,\mathrm{M} = 3.248\,\mathrm{\mu M}$ |
| 12. Heavy atoms and LE | ${\color{#c2410c}{N_{A}}} + {\color{#7c3aed}{N_{B}}} + {\color{#b91c1c}{N_{link}}} = 21;\quad LE_{ideal} = \frac{8.483}{21} = 0.404;\quad LE = \frac{7.483}{21} = 0.3563$ |
| 13. Answer | $\boxed{K_{d,linked} = 3.248\,\mathrm{\mu\ M}}$ |
| 14. What it means | Two weak fragments (2000 and 300 µM) give about 600 nM if their energies add perfectly, but 3.248 µM with a 1 kcal/mol linking penalty (LE 0.3563): 1 kcal/mol costs a factor of 5.413. The linker must hold both fragments in place without strain. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio; pKd 6 = Kd 1 µM = 10⁻⁶ M; pKd 9 = 1 nM; 2.303RT = 1.364 kcal/mol (5.708 kJ/mol) per pKd unit at 25 °C; 1 mM = 1000 µM = 10⁻³ M
</details>

In [ ]:
#@title 3.11 Solve it yourself: Practice: linking fragments found by NMR { display-mode: "form" }
_cell('3.11 Solve it yourself: Practice: linking fragments found by NMR')
card('l27_link_practice')

In [ ]:
#@title Check L27-X1   (run after choosing) { display-mode: "form" }
#@markdown **L27-X1.** A site has one polar region (a backbone N-H donor) between two hydrophobic pockets. Screening finds a weak aromatic fragment in each pocket, and crystal structures show both poses. Which first design follows a sound fragment-linking strategy?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Join the two fragments with a linker that keeps both poses and carries an acceptor for the N-H", "B. Join the two fragments directly, letting each shift until they meet, and add the acceptor later", "C. Keep one fragment and add atoms until both pockets and the polar region are completely filled", "D. Set the fragments aside and choose a catalogue compound that docks across all three regions"]
gb.answer("L27-X1", response[:1])

## Part 4 - Case study: de novo design against thymidylate synthase

*Objectives addressed: L27.1 · L27.2 · L27.3*


**The target.** Thymidylate synthase (TS) makes **dTMP** from **dUMP** by adding a methyl group at C5 of the uracil ring. The one-carbon unit (and a hydride) come from the cofactor **N⁵,N¹⁰-methylene-tetrahydrofolate**, which leaves as dihydrofolate. dTMP is the only *de novo* source of thymidine nucleotides for DNA, so blocking TS stops DNA synthesis in dividing cells, which makes it an anticancer target. Inhibitors can be modelled on either partner:

| Site | Inhibitor idea | Example | Comment |
|---|---|---|---|
| **Substrate (dUMP) site** | substrate analogue that traps the enzyme | 5-fluorouracil → FdUMP | covalent ternary complex with enzyme and cofactor |
| **Cofactor (folate) site** | folate analogue | CB3717 (a quinazoline antifolate), then **raltitrexed** | CB3717's quinazoline ring makes three key H-bonds; a bridging water is part of the network. Selectivity is harder because other enzymes also bind folates |
| **Cofactor site, de novo** | new scaffold designed into the empty pocket | naphthostyril (benz[cd]indolone) series | the case study below |

In [ ]:
#@title 4.1 Figure — the TS reaction and the 5-fluorouracil / FdUMP substrate-site inhibitor { display-mode: "form" }
_cell('4.1 Figure — the TS reaction and the 5-fluorouracil / FdUMP substrate-site inhibitor')
fig_ts_reaction()

**The de novo route.** Starting from the crystal structure of TS with CB3717 bound, the inhibitor was removed in silico and a grid was laid over the empty cofactor pocket. An aromatic C–H probe at each grid point showed that the pocket for the pteridine-like ring was **hydrophobic**.

| Step | Change | Reason |
|---|---|---|
| 1 | naphthalene scaffold | fits the hydrophobic pocket by van der Waals contacts but is **smaller than the pocket**, leaving room for groups |
| 2 | add a lactam → benz[cd]indol-2(1H)-one (naphthostyril) | the lactam N–H and C=O can make hydrogen bonds with the site |
| 3 | amino substituent at the 5-position | reaches a vacant region; position chosen for **synthetic feasibility**; N-alkyl groups can be varied without creating a stereocentre |
| 4 | benzyl group on that nitrogen | mimics the benzene ring of the cofactor |
| 5 | phenylsulfonyl-piperazine on the benzyl ring | water solubility; points **out to solvent**, so no desolvation penalty |
| 6 | synthesize, test, co-crystallize | active; but the crystal structure showed it sat **deeper** in the pocket than planned: Ala-263 moved to avoid a clash and the bridging water shifted |

In [ ]:
#@title 4.2 Figure — first two steps of the de novo scaffold { display-mode: "form" }
_cell('4.2 Figure — first two steps of the de novo scaffold')
_ = fig_ts_scaffold()

**Structure-based optimization** then used the *actual* binding mode to explore four regions: a small alkyl (or hydroxyethyl) group in a pocket that becomes polar with depth (R1); replacing the lactam C=O by an NH₂ to make a protonated **amidine** that hydrogen-bonds and forms an ionic interaction instead of clashing with Ala-263 (R2, the largest single gain; binding returned to the intended mode with the water in place); a methyl or chloro in a small hydrophobic pocket (R3); and a morpholine for selectivity and pharmacokinetics (R4). Combining **every** best group was *not* best; the amidine + morpholine + methyl combination was about 500 times more active than the starting lead and was taken into clinical trials.

**Lesson:** the first de novo design was active **because** it was not a perfect, rigid fit — it could find a slightly different binding mode, and the space it left was what the optimization filled.

**1. What a fold improvement is worth: ΔΔG = RT ln(fold)**

$${\color{#15803d}{-\Delta\Delta G}} = {\color{#4b5563}{R}}{\color{#c2410c}{T}}\ln {\color{#1d6fb8}{F}};\quad {\color{#1d6fb8}{F}} = \frac{K_{d,lead}}{K_{d,optimized}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{F}}$ | fold improvement in affinity | input | — | TS case study (notes) |
| ${\color{#c2410c}{T}}$ | absolute temperature | input | K | assay temperature (case) |
| ${\color{#15803d}{-\Delta\Delta G}}$ | free-energy gain from optimization | result | kcal/mol | calculated |
| ${\color{#4b5563}{R}}$ | gas constant (calorie units) | constant | cal/mol/K | value = 1.987 cal/mol/K; physical constant (1.987 cal·mol⁻¹·K⁻¹ = 8.314 J·mol⁻¹·K⁻¹) |

*Source of the relationship:* ΔG° = RT ln Kd applied to two compounds

**2. Practice problem (case)**

Structure-based optimization of the naphthostyril thymidylate synthase lead gave a compound about $\mathbf{{\color{#1d6fb8}{500}}}$ more active. Treating activity as affinity at $\mathbf{{\color{#c2410c}{298\,\mathrm{K}}}}$, how much binding free energy did the optimization add?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#15803d}{-\Delta\Delta G}} = {\color{#4b5563}{R}}{\color{#c2410c}{T}}\ln {\color{#1d6fb8}{F}};\quad {\color{#1d6fb8}{F}} = \frac{K_{d,lead}}{K_{d,optimized}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{F}} = 500$; ${\color{#c2410c}{T}} = 298\,\mathrm{K}$; ${\color{#4b5563}{R}} = 1.987\,\mathrm{cal/mol/K}$ (constant) |
| 3. Unit check | No unit prefixes to expand. The fold change is a ratio of two Kd values, so their units (and any nM/µM prefix) cancel and ln F is a pure number. R is in cal/(mol·K), so RT comes out in cal/mol and is divided by 10³ to give kcal/mol; K cancels in R × T. |
| 4. Temperature scales | ${\color{#c2410c}{T}} = 298\,\mathrm{K} = 24.85\,{}^{\circ}\mathrm{C} = 76.73\,{}^{\circ}\mathrm{F}$ |
| 5. Thermal energy RT | ${\color{#4b5563}{R}} {\color{#c2410c}{T}} = 1.987\,\mathrm{cal/(mol\cdot K)} \times 298\,\mathrm{K} = 592.1\,\mathrm{cal/mol} = 0.5921\,\mathrm{kcal/mol}$ |
| 6. Natural log of the fold change | $\ln {\color{#1d6fb8}{F}} = \ln 500 = 6.215$ |
| 7. Free-energy gain | ${\color{#15803d}{-\Delta\Delta G}} = 0.5921 \times 6.215 = 3.68\,\mathrm{kcal/mol}$ |
| 8. In kJ/mol and log units | $3.68 \times 4.184 = 15.4\,\mathrm{kJ/mol};\quad \log_{10} 500 = 2.699$ |
| 9. Answer | $\boxed{-\Delta\Delta G = 3.68\,\mathrm{kcal/mol}}$ |
| 10. What it means | A 500-fold gain in affinity is worth 3.68 kcal/mol (15.4 kJ/mol), about the energy of one or two good hydrogen bonds or an ionic contact made in the right place, here the amidine replacing the clashing lactam C=O. |

**Common conversions for this calculation:** T(K) = T(°C) + 273.15; T(°F) = 1.8 × T(°C) + 32; 1 kcal = 4.184 kJ; R = 8.314 J/(mol·K) = 1.987 cal/(mol·K) = N_A × k_B, with k_B = 1.381 × 10⁻²³ J/K; RT = 2.479 kJ/mol (0.5925 kcal/mol) at 298.15 K and 2.579 kJ/mol (0.6163 kcal/mol) at 310.15 K; RT ln 10 = 5.708 kJ/mol (1.364 kcal/mol) at 298.15 K: each 5.7 kJ/mol is a 10-fold ratio
</details>

In [ ]:
#@title 4.3 Solve it yourself: What a fold improvement is worth: ΔΔG = RT ln(fold) { display-mode: "form" }
_cell('4.3 Solve it yourself: What a fold improvement is worth: ΔΔG = RT ln(fold)')
card('l27_fold_ddg')

In [ ]:
#@title 4.4 Figure — raltitrexed, a marketed cofactor-site TS inhibitor { display-mode: "form" }
_cell('4.4 Figure — raltitrexed, a marketed cofactor-site TS inhibitor')
_ = fig_raltitrexed()

#### Worked example: auditing a de novo design
A student proposes a rigid tetracyclic design that fills every cavity of the TS cofactor pocket, buries a carboxylic acid against a hydrophobic wall and scores better than any compound in the series. Name two binding risks and two synthetic or property risks.

<details>
<summary><b>▸ Worked solution</b></summary>

**Binding risks:** (1) filling every cavity leaves no tolerance for errors in the crystal coordinates or for the protein movement actually seen in this case (Ala-263 shifted); a rigid molecule cannot adopt an alternative binding mode. (2) A carboxylate buried against a hydrophobic wall pays a large **desolvation** penalty with no partner, which the score may not capture.

**Synthetic / property risks:** (1) a fused tetracycle may need a long, low-yield synthesis (check synthetic accessibility, as CAESA or SYNOPSIS would). (2) No group points out to solvent, so solubility will be poor, and a fully filled design leaves no position to fix that later.

A better first design is smaller, a little flexible, and has one solvent-exposed position reserved for solubilizing groups.
</details>

**Audit a design yourself.** The tool sorts the features of a proposed structure into binding risks and synthetic-feasibility or property risks. The defaults describe the tetracyclic design in the worked example; change one feature at a time and see which list it affects.

In [ ]:
#@title 4.5 Tool — audit a proposed design for binding and synthesis risks { display-mode: "form" }
_cell('4.5 Tool — audit a proposed design for binding and synthesis risks')
fills_site_completely = True #@param {type:"boolean"}
rotatable_bonds = 1 #@param {type:"slider", min:0, max:15, step:1}
buried_polar_without_partner = True #@param {type:"boolean"}
solvent_exposed_position = False #@param {type:"boolean"}
stereocentres = 4 #@param {type:"slider", min:0, max:8, step:1}
fused_rings = 4 #@param {type:"slider", min:0, max:6, step:1}
heavy_atoms = 38 #@param {type:"integer"}
pKd = 8.0 #@param {type:"number"}
clogP = 5.5 #@param {type:"number"}
_ = design_risk_audit(fills_site_completely, rotatable_bonds, buried_polar_without_partner, solvent_exposed_position, stereocentres, fused_rings, heavy_atoms, pKd, clogP)

#### Worked example: rule-of-three check
Two fragment hits: (a) MW 245, clog P 2.1, 2 donors, 3 acceptors, 2 rotatable bonds; (b) MW 340, clog P 3.6, 1 donor, 4 acceptors, 5 rotatable bonds. Which meets the rule of three?

<details>
<summary><b>▸ Worked solution</b></summary>

(a) passes every test (245 ≤ 300; 2.1 ≤ 3; 2 ≤ 3; 3 ≤ 3; 2 ≤ 3). (b) fails MW, clog P, acceptors and rotatable bonds: it is already lead-sized, leaves little room to grow, and would be a poor fragment-library member even if it binds.
</details>

## Part 5 - Where is there actually room to grow

*Objectives addressed: L27.2 · L27.4*

De novo design means building a ligand into empty space. So find the
empty space. The cell lays a grid over the gorge around the bound tacrine and
marks every point that a probe of your chosen radius could occupy without
touching the protein or the existing ligand.

> **About this structure.** 1ACJ is *Torpedo californica*
> acetylcholinesterase with tacrine bound, from the Protein Data Bank
> (Harel *et al.*, *Proc. Natl. Acad. Sci. USA* 1993, **90**, 9031).
> The coordinates are embedded in this notebook, so the cell below
> downloads nothing and sends nothing anywhere. No personal data of any
> kind is involved, and nothing you type is transmitted or stored.

That free volume is what a de novo program would try to fill. Change the probe
radius and watch how much the answer depends on a choice you made.

In [ ]:
#@title Find the unfilled volume { display-mode: "form" }
#@markdown Probe radius; water is about 1.4 angstrom. Finer grids are slower.
probe_radius = 1.4 #@param {type:"slider", min:1.0, max:2.0, step:0.1}
grid_spacing = 0.7 #@param [1.0, 0.7, 0.5] {type:"raw"}
r = crystal.empty_space(probe=probe_radius, grid=grid_spacing)

In [ ]:
#@title C1   (run after choosing) { display-mode: "form" }
#@markdown **C1.**  At a 1.6 angstrom probe your output reports about 31 cubic angstrom of free space; at 1.4 angstrom about 67; at 1.2 angstrom about 118. Refining the grid from 1.0 to 0.5 angstrom changes each value by only a few per cent. What do the two patterns together establish?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Converged on the grid but probe-dependent, so the probe must be reported", "B. Unreliable, since two separate choices change it and neither is justified", "C. Grid spacing dominates the error and ought to be made finer still", "D. The probe has no physical meaning, so the smallest value should be used"]
gb.answer("C1", response[:1])

In [ ]:
#@title C2   (run after choosing) { display-mode: "form" }
#@markdown **C2.**  Measured with a 1.4 angstrom probe, the free space beside tacrine comes to roughly 67 cubic angstrom, which is on the order of a few heavy atoms. Historically, potent acetylcholinesterase inhibitors were obtained by linking two binding units with a long chain. What does the measured volume suggest about why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Too little room beside the ligand, so designers reached to a second site", "B. The volume allows a large extension, so the linked design was arbitrary", "C. Linking two units lowers the volume needed, which is why it succeeds", "D. The measurement is irrelevant, since affinity ignores molecular shape"]
gb.answer("C2", response[:1])

In [ ]:
#@title C3   (run after choosing) { display-mode: "form" }
#@markdown **C3.**  Your output warns that a first design should not fill all the free volume. Why would filling it completely be a poor starting point?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. A fully filled design leaves no tolerance for error in pose or model", "B. A filled design is too heavy for oral availability whatever its shape", "C. Filling the volume is impossible, so the attempt wastes computing time", "D. Filled designs bind too tightly, and excess affinity is undesirable"]
gb.answer("C3", response[:1])

## Part 6 - Fragment-based and de novo design behind marketed drugs (reference)


| Drug | Target / use (mechanism level) | Design origin |
|---|---|---|
| **Vemurafenib** | BRAF V600E kinase inhibitor; melanoma with that mutation | fragment-based: a small 7-azaindole fragment bound to the kinase hinge was grown using repeated crystal structures |
| **Venetoclax** | BCL-2 inhibitor (BH3 mimetic) that frees pro-apoptotic proteins so cancer cells undergo apoptosis; CLL and AML | fragment **linking**: two weak fragments found by NMR (SAR by NMR) were linked, then optimized through ABT-737 and navitoclax to a BCL-2-selective drug |
| **Erdafitinib** | FGFR kinase inhibitor; urothelial carcinoma with FGFR alterations | fragment-based screening and structure-guided growth |
| **Asciminib** | allosteric BCR-ABL1 inhibitor binding the myristoyl pocket; CML | fragment-based NMR screening, then structure-based optimization |
| **Raltitrexed** | thymidylate synthase, folate (cofactor) site; colorectal cancer (not approved in the US) | structure-guided analogue design from the quinazoline antifolate CB3717 to improve solubility and remove its kidney toxicity |
| **5-Fluorouracil** | thymidylate synthase via FdUMP (substrate site); solid tumours | substrate-analogue design: F for H at C5 blocks the reaction |

**Looking ahead (L29–L30 QSAR).** De novo design and docking reason from the target's structure. QSAR reasons from a series of measured activities and the properties of the compounds, and is used when the target structure is unknown or to fine-tune a series.

## Part 7 - End-of-activity assessment

Five questions in the style of the course quizzes and exams, mapped to the
Student Intended Learning Outcomes for this unit (**SILO 6**).
This time you get full feedback, including why each wrong option is wrong.
Answer each one, then run the summary cell at the bottom.

In [ ]:
#@title Q1   (run after choosing) { display-mode: "form" }
#@markdown **Q1.**  What distinguishes de novo design from virtual screening?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Candidate structures are restricted to compounds already in catalogues", "B. Candidate structures are ranked by measured affinity rather than score", "C. Candidate structures are generated from the site rather than retrieved", "D. Candidate structures are built without reference to the target protein"]
gb.answer("Q1", response[:1])

In [ ]:
#@title Q2   (run after choosing) { display-mode: "form" }
#@markdown **Q2.**  Why is a space-filling first-generation de novo ligand usually a poor choice?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Site coordinates carry error and no room is left to optimize", "B. Larger ligands are always metabolized faster than smaller ones", "C. Programs cannot build structures above about thirty heavy atoms", "D. Full occupancy prevents the ligand from forming hydrogen bonds"]
gb.answer("Q2", response[:1])

In [ ]:
#@title Q3   (run after choosing) { display-mode: "form" }
#@markdown **Q3.**  How does LUDI assemble a candidate structure?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It adds single atoms along a grid, adjusting each for local fit", "B. It joins amino acid residues until the pocket is filled in", "C. It mutates and recombines torsion strings over many generations", "D. It fits library fragments to interaction sites, then bridges them"]
gb.answer("Q3", response[:1])

In [ ]:
#@title Q4   (run after choosing) { display-mode: "form" }
#@markdown **Q4.**  In SPROUT, a fragment template is specified by
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. three interaction points forming a pharmacophoric triangle", "B. a ranked list of commercially purchasable starting materials", "C. vertices for hybridization states and edges for bond orders", "D. partial atomic charges assigned to every atom in the fragment"]
gb.answer("Q4", response[:1])

In [ ]:
#@title Q5   (run after choosing) { display-mode: "form" }
#@markdown **Q5.**  What does the CAESA program contribute to a de novo workflow?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It estimates how accessible a designed structure is synthetically", "B. It calculates the desolvation penalty of the designed ligand", "C. It generates conformations of each designed structure for docking", "D. It aligns designed structures onto a reference pharmacophore"]
gb.answer("Q5", response[:1])

## Lab report (automatic)
Every figure you displayed, every calculation you solved with **Solve step by step** and your first answer to each check were collected as you worked. Add your name and a short discussion, then build the report: a **Word (.docx)** file to edit and submit and an **HTML** copy with the equations exactly as shown here. A file is written only when you press the button.

In [ ]:
#@title Build and download my lab report { display-mode: "form" }
_cell('Build and download my lab report')
report_panel()

## Your results

In [ ]:
#@title Show my performance and my report code { display-mode: "form" }
gb.summary()
print()
_ = gb.report_code()

### What to submit

1. Copy the single `L27|...` line printed above.
2. Paste it into the Canvas assignment **L27 Colab activity**.
3. With your partner, be ready to say one thing the simulation showed you that
   the reading did not.

Your report code records how many questions you answered correctly. It is a
participation and self-check record, not a proctored exam score.